<a href="https://colab.research.google.com/github/muntherlafi/cti/blob/master/MARL_Trust_Colab_(8).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤝 Reputation-Weighted Communication in Cooperative MARL
### *Resilient cooperation under defecting agents*

**Project:** Trust & Reputation Management in Multi-Agent Reinforcement Learning  
**Environment:** MPE `simple_spread_v3` (PettingZoo)  
**Algorithm:** MADDPG + Reputation-Weighted Communication (RWC) module  
**Duration:** ~4 weeks implementation, ~2–4 hrs training on Colab GPU

---

## 📋 Notebook Structure
| Cell | Content |
|------|---------|
| 1 | Install dependencies |
| 2 | Imports & GPU check |
| 3 | `env_wrapper.py` — Defector injection |
| 4 | `reputation.py` — ReputationTracker |
| 5 | `networks.py` — Modified MADDPG Actor & Critic |
| 6 | `buffer.py` — Replay buffer |
| 7 | `train.py` — Training loop |
| 8 | Unit tests |
| 9 | Run baseline (Condition A) |
| 10 | Run binary trust (Condition B) |
| 11 | Run soft reputation (Condition C — RWC) |
| 12 | Full experiment matrix (all 27 runs) |
| 13 | Results & figures |

> **Runtime:** Set to `GPU` in Runtime → Change runtime type for faster training.


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install Dependencies                          ║
# ╚══════════════════════════════════════════════════════════╝
# Run once. After it finishes, go to:
#   Runtime -> Restart session
# Then run all remaining cells from Cell 2 onwards.
# Do NOT run Cell 1 again after restarting.

import subprocess, sys, importlib

def pip_install(package):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", package],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("FAILED: " + package)
        print(result.stderr[-1500:])
        raise RuntimeError("Install failed: " + package)
    print("OK: " + package)

# ── Install in dependency order ───────────────────────────────────────────
# torch is pre-installed on Colab — install it explicitly here so it is
# available immediately without needing a runtime restart for that package.
pip_install("torch")
pip_install("mpe2")
pip_install("pettingzoo>=1.25.0")
pip_install("gymnasium>=1.0.0")
pip_install("seaborn>=0.12")
pip_install("tqdm>=4.65")

# ── Verify every package imports cleanly ─────────────────────────────────
print("")
print("Verifying imports...")
ok = True
for mod, pkg in [
    ("torch",      "torch"),
    ("mpe2",       "mpe2"),
    ("pettingzoo", "pettingzoo"),
    ("gymnasium",  "gymnasium"),
    ("numpy",      "numpy"),
    ("seaborn",    "seaborn"),
    ("tqdm",       "tqdm"),
    ("scipy",      "scipy"),
    ("pandas",     "pandas"),
    ("matplotlib", "matplotlib"),
]:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        print("  " + pkg + " " + ver)
    except ModuleNotFoundError as e:
        print("  MISSING: " + pkg + " -> " + str(e))
        ok = False

print("")
if ok:
    print("All packages installed successfully.")
    print("")
    print(">>> ACTION REQUIRED: Runtime -> Restart session <<<")
    print("Then run from Cell 2 onwards. Do not re-run Cell 1.")
else:
    print("Some packages failed. Check errors above before continuing.")


OK: torch
OK: mpe2
OK: pettingzoo>=1.25.0
OK: gymnasium>=1.0.0
OK: seaborn>=0.12
OK: tqdm>=4.65

Verifying imports...
  torch 2.11.0+cpu
  mpe2 1.1.0
  pettingzoo 1.26.1
  gymnasium 1.3.0
  numpy 2.0.2
  seaborn 0.13.2
  tqdm 4.67.3
  scipy 1.16.3
  pandas 2.2.2
  matplotlib 3.10.0

All packages installed successfully.

>>> ACTION REQUIRED: Runtime -> Restart session <<<
Then run from Cell 2 onwards. Do not re-run Cell 1.


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — Imports & Environment Check                   ║
# ╚══════════════════════════════════════════════════════════╝

import os, sys, time, random, copy
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats
from tqdm.auto import tqdm, trange

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: " + str(DEVICE))
if DEVICE.type == "cuda":
    print("  GPU : " + torch.cuda.get_device_name(0))

# ── mpe2 observation layout for simple_spread_v3, N=4 ────────────────────
# [self_vel(2), self_pos(2), landmark_rel_pos(4*2=8),
#  other_agent_rel_pos(3*2=6), communication(3*2=6)]
# Total = 2+2+8+6+6 = 24
from mpe2 import simple_spread_v3

env = simple_spread_v3.env(N=4, local_ratio=0.5, render_mode=None)
env.reset(seed=0)
for agent in env.agent_iter():
    obs, rew, term, trunc, info = env.last()
    env.step(env.action_space(agent).sample())
    break
env.close()

print("mpe2 simple_spread_v3 (N=4):")
print("  obs shape : " + str(obs.shape))
print("  agents    : " + str(env.agents))

# Confirm layout matches our constants
N = 4
expected_obs_dim = 2 + 2 + N*2 + (N-1)*2 + (N-1)*2   # = 24
assert obs.shape == (expected_obs_dim,), (
    "Obs shape mismatch: got " + str(obs.shape) +
    ", expected (" + str(expected_obs_dim) + ",)"
)
print("  obs_dim   : " + str(expected_obs_dim) + "  [2 vel + 2 pos + 8 landmarks + 6 others + 6 comms]")
print("  msg_start : 18  (slice [18:24] = 3 peers x 2D comm vectors)")
print("  msg_end   : 24")
print("  Shape check passed.")


Device: cpu
mpe2 simple_spread_v3 (N=4):
  obs shape : (24,)
  agents    : ['agent_0', 'agent_1', 'agent_2', 'agent_3']
  obs_dim   : 24  [2 vel + 2 pos + 8 landmarks + 6 others + 6 comms]
  msg_start : 18  (slice [18:24] = 3 peers x 2D comm vectors)
  msg_end   : 24
  Shape check passed.


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — Hyperparameter Config                         ║
# ╚══════════════════════════════════════════════════════════╝

@dataclass
class Config:
    # ── Environment ──────────────────────────────────────────
    n_agents:        int   = 4
    n_landmarks:     int   = 4
    local_ratio:     float = 0.5
    max_steps:       int   = 50
    #
    # mpe2 obs layout for N=4:
    # [self_vel(2), self_pos(2), landmark_rel_pos(8),
    #  other_agent_rel_pos(6), communication(6)]
    # Total obs_dim = 24
    # msg_start = 18  (start of communication slice)
    # msg_end   = 24  (end of communication slice)
    obs_dim:         int   = 24   # 2+2+8+6+6 for N=4
    act_dim:         int   = 5    # Discrete(5)
    msg_dim:         int   = 2    # each peer sends a 2D comm vector
    msg_start:       int   = 18   # index where comm slice begins
    msg_end:         int   = 24   # index where comm slice ends

    # ── Defectors ────────────────────────────────────────────
    n_defectors:     int   = 1
    noise_scale:     float = 1.0

    # ── Trust condition ──────────────────────────────────────
    # "none"   = Condition A: equal weights
    # "binary" = Condition B: hard threshold gate
    # "soft"   = Condition C: EMA reputation weighting (RWC)
    trust_condition:    str   = "soft"
    rep_alpha:          float = 0.05
    rep_baseline_alpha: float = 0.01
    rep_temperature:    float = 5.0
    binary_threshold:   float = 0.5

    # ── MADDPG ───────────────────────────────────────────────
    hidden_dim:      int   = 128
    actor_lr:        float = 1e-3
    critic_lr:       float = 1e-3
    gamma:           float = 0.95
    tau:             float = 0.01
    buffer_size:     int   = 100_000
    batch_size:      int   = 256
    warmup_steps:    int   = 1000

    # ── Training ─────────────────────────────────────────────
    n_episodes:      int   = 500
    eval_interval:   int   = 25
    eval_episodes:   int   = 20
    seed:            int   = 42

    # ── Output ───────────────────────────────────────────────
    save_dir:        str   = "/content/marl_trust"

    def __post_init__(self):
        import os
        os.makedirs(self.save_dir, exist_ok=True)
        self.n_peers = self.n_agents - 1
        all_ids = ["agent_" + str(i) for i in range(self.n_agents)]
        self.defector_ids = set(all_ids[-self.n_defectors:]) if self.n_defectors > 0 else set()
        self.honest_ids   = set(all_ids) - self.defector_ids

cfg = Config()
print("Config created.")
print("  obs_dim   = " + str(cfg.obs_dim))
print("  msg_start = " + str(cfg.msg_start) +
      "  (slice [" + str(cfg.msg_start) + ":" + str(cfg.msg_end) +
      "] = " + str(cfg.n_peers) + " peers x " + str(cfg.msg_dim) + "D comms)")
print("  defectors = " + str(cfg.defector_ids))

# Sanity check: msg slice must exactly fit n_peers * msg_dim
assert cfg.msg_end - cfg.msg_start == cfg.n_peers * cfg.msg_dim, (
    "Msg slice length " + str(cfg.msg_end - cfg.msg_start) +
    " != n_peers*msg_dim " + str(cfg.n_peers * cfg.msg_dim)
)
print("  Slice sanity check passed.")


Config created.
  obs_dim   = 24
  msg_start = 18  (slice [18:24] = 3 peers x 2D comms)
  defectors = {'agent_3'}
  Slice sanity check passed.


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 — DefectorWrapper (env_wrapper.py)              ║
# ╚══════════════════════════════════════════════════════════╝

# MPE environments are now in the mpe2 package (moved from mpe2)
from mpe2 import simple_spread_v3

class DefectorWrapper:
    """
    Wraps simple_spread_v3 and corrupts outgoing communication messages
    from designated defector agents by injecting Gaussian noise.

    Compatible with PettingZoo 1.22.x and 1.24.x.

    Design:
    - Only obs[msg_start:msg_end] is corrupted for honest agents.
    - Defectors' own observations remain clean (they can still sense
      positions), making the problem harder for honest agents.
    - The wrapper never modifies the underlying environment state.
    """

    def __init__(self, cfg, seed: int = None):
        self.cfg  = cfg
        self.rng  = np.random.default_rng(seed)
        self._env = simple_spread_v3.env(
            N           = cfg.n_agents,
            local_ratio = cfg.local_ratio,
            render_mode = None,
            max_cycles  = cfg.max_steps,
        )
        self.agents    = self._env.possible_agents   # stable ordered list
        self.n_agents  = len(self.agents)
        self.agent_idx = {a: i for i, a in enumerate(self.agents)}

    def reset(self, seed=None):
        self._env.reset(seed=seed)
        return self._collect_obs()

    def step_all(self, actions: Dict[str, np.ndarray]):
        """
        Execute one full round: each agent acts once.
        actions: {agent_id: one-hot np.ndarray}
        Returns: obs, rewards, dones (all dicts)
        """
        rewards, dones = {a: 0.0 for a in self.agents}, {a: False for a in self.agents}

        for agent in self._env.agent_iter():
            obs_raw, rew, term, trunc, info = self._env.last()
            done = term or trunc
            rewards[agent] = float(rew)
            dones[agent]   = done
            if done:
                self._env.step(None)
            else:
                act_oh = actions.get(agent)
                if act_oh is not None:
                    act_int = int(np.argmax(act_oh))
                else:
                    act_int = self._env.action_space(agent).sample()
                self._env.step(act_int)

        obs = self._collect_obs()
        return obs, rewards, dones

    def _collect_obs(self) -> Dict[str, np.ndarray]:
        obs = {}
        for agent in self.agents:
            try:
                o = self._env.observe(agent)
                if o is None:
                    o = np.zeros(self.cfg.obs_dim, dtype=np.float32)
                else:
                    o = np.array(o, dtype=np.float32).copy()
            except Exception:
                o = np.zeros(self.cfg.obs_dim, dtype=np.float32)
            obs[agent] = self._maybe_corrupt(agent, o)
        return obs

    def _maybe_corrupt(self, observer: str, obs: np.ndarray) -> np.ndarray:
        """Add Gaussian noise to peer-message slice for honest observers."""
        if not self.cfg.defector_ids or observer in self.cfg.defector_ids:
            return obs
        noise = (self.rng.standard_normal(self.cfg.msg_end - self.cfg.msg_start)
                 .astype(np.float32) * self.cfg.noise_scale)
        obs[self.cfg.msg_start:self.cfg.msg_end] += noise
        return obs

    def sample_action(self, agent_id: str) -> int:
        return self._env.action_space(agent_id).sample()

    def close(self):
        self._env.close()


# ── Unit test ─────────────────────────────────────────────────────────────────
def test_defector_wrapper():
    from dataclasses import replace

    # With 1 defector
    cfg_dirty = Config(n_defectors=1)
    env_dirty = DefectorWrapper(cfg_dirty, seed=0)
    env_dirty.reset(seed=0)
    dummy = np.zeros(cfg_dirty.obs_dim, dtype=np.float32)
    corrupted = env_dirty._maybe_corrupt("agent_0", dummy.copy())
    std_dirty = corrupted[cfg_dirty.msg_start:cfg_dirty.msg_end].std()

    # With 0 defectors
    cfg_clean = Config(n_defectors=0)
    env_clean = DefectorWrapper(cfg_clean, seed=0)
    env_clean.reset(seed=0)
    clean = env_clean._maybe_corrupt("agent_0", dummy.copy())
    std_clean = clean[cfg_clean.msg_start:cfg_clean.msg_end].std()

    print(f"✅ DefectorWrapper unit test")
    print(f"   Msg slice std with  defector : {std_dirty:.4f}  (should be > 0.1)")
    print(f"   Msg slice std without defector: {std_clean:.4f}  (should be 0.0)")
    assert std_dirty > 0.1, "Corruption not applied!"
    assert std_clean == 0.0, "Clean env is corrupted — bug!"

    # Smoke: run a few steps
    env2 = DefectorWrapper(Config(n_defectors=1), seed=7)
    obs  = env2.reset(seed=7)
    assert len(obs) == 4, f"Expected 4 agent obs, got {len(obs)}"
    dummy_acts = {a: np.eye(5)[env2.sample_action(a)] for a in env2.agents}
    obs2, rews, dones = env2.step_all(dummy_acts)
    assert len(rews) == 4
    env2.close()
    env_dirty.close()
    env_clean.close()
    print("   Step test passed ✓")
    print("   All assertions passed ✓")

test_defector_wrapper()


✅ DefectorWrapper unit test
   Msg slice std with  defector : 0.6786  (should be > 0.1)
   Msg slice std without defector: 0.0000  (should be 0.0)
   Step test passed ✓
   All assertions passed ✓


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5 — ReputationTracker (reputation.py)             ║
# ╚══════════════════════════════════════════════════════════╝

from scipy.special import softmax as scipy_softmax

class ReputationTracker:
    """
    Maintains a continuous reputation score r[j] in [0,1] for each peer j.

    FIX: baseline is now initialised to None and set from the first real
    reward, preventing the 'always negative delta' problem that caused all
    scores to drift to 0 then snap to 1.0 when rewards improved.

    Update rule (per step):
        delta  = reward - baseline
        signal = 1.0 if delta > 0 else 0.0
        r[j]  <- (1 - alpha) * r[j] + alpha * signal  for all j
        baseline <- (1 - b_alpha) * baseline + b_alpha * reward
    """

    def __init__(self, peer_ids, cfg):
        self.peer_ids    = list(peer_ids)
        self.alpha       = cfg.rep_alpha
        self.b_alpha     = cfg.rep_baseline_alpha
        self.temperature = cfg.rep_temperature
        self.scores      = {j: 0.5 for j in peer_ids}
        self.baseline    = None    # FIX: initialise lazily from first reward

    def update(self, reward):
        """Called once per step with this agent's observed reward."""
        # FIX: initialise baseline to first real reward so delta starts near 0
        if self.baseline is None:
            self.baseline = reward
            return  # skip update on very first step

        delta  = reward - self.baseline
        signal = 1.0 if delta > 0 else 0.0

        self.baseline = (1 - self.b_alpha) * self.baseline + self.b_alpha * reward

        for j in self.peer_ids:
            self.scores[j] = (
                (1 - self.alpha) * self.scores[j] + self.alpha * signal
            )

    def get_weights(self):
        """Softmax-normalised weights, ordered by peer_ids."""
        scores_arr = np.array([self.scores[j] for j in self.peer_ids],
                               dtype=np.float32)
        return scipy_softmax(scores_arr * self.temperature).astype(np.float32)

    def get_weights_binary(self, threshold=0.5):
        """Condition B: hard binary gate, re-normalised."""
        scores_arr = np.array([self.scores[j] for j in self.peer_ids],
                               dtype=np.float32)
        weights = (scores_arr >= threshold).astype(np.float32)
        total = weights.sum()
        if total == 0:
            weights = np.ones(len(self.peer_ids), dtype=np.float32)
            total   = float(len(self.peer_ids))
        return weights / total

    def get_weights_uniform(self):
        """Condition A: no trust — uniform weights."""
        n = len(self.peer_ids)
        return np.ones(n, dtype=np.float32) / n

    def get_scores_dict(self):
        return dict(self.scores)

    def reset(self):
        self.scores   = {j: 0.5 for j in self.peer_ids}
        self.baseline = None   # FIX: reset to None, not 0.0


# ── Unit tests ────────────────────────────────────────────────────────────────
def test_reputation_tracker():
    cfg_t = Config()
    peers = ["agent_1", "agent_2", "agent_3"]
    tracker = ReputationTracker(peers, cfg_t)

    # Test 1: neutral init
    w = tracker.get_weights()
    assert abs(w.sum() - 1.0) < 1e-5, "Weights do not sum to 1"
    assert abs(w[0] - w[1]) < 1e-5, "Initial weights not uniform"
    print("  Test 1 passed: neutral init OK")

    # Test 2: positive drift — rewards consistently above baseline
    tracker.reset()
    for i in range(200):
        tracker.update(reward=-5.0 + i * 0.05)   # rising rewards
    scores = list(tracker.scores.values())
    print("  Test 2: scores after rising rewards: " + str([round(s,3) for s in scores]))
    assert all(s > 0.5 for s in scores), "Scores did not drift up: " + str(scores)
    print("  Test 2 passed: positive drift OK (mean=" + str(round(float(np.mean(scores)),3)) + ")")

    # Test 3: negative drift — rewards consistently below baseline
    tracker.reset()
    for i in range(200):
        tracker.update(reward=-5.0 - i * 0.05)   # falling rewards
    scores = list(tracker.scores.values())
    print("  Test 3: scores after falling rewards: " + str([round(s,3) for s in scores]))
    assert all(s < 0.5 for s in scores), "Scores did not drift down: " + str(scores)
    print("  Test 3 passed: negative drift OK (mean=" + str(round(float(np.mean(scores)),3)) + ")")

    # Test 4: weights always sum to 1
    for _ in range(50):
        tracker.update(reward=np.random.randn() - 5.0)
    for method in ["get_weights", "get_weights_binary", "get_weights_uniform"]:
        w = getattr(tracker, method)()
        assert abs(w.sum() - 1.0) < 1e-5, method + " does not sum to 1"
    print("  Test 4 passed: all weight methods sum to 1.0 OK")

    # Test 5: FIX verification — baseline initialised from first reward
    tracker2 = ReputationTracker(peers, cfg_t)
    tracker2.update(reward=-8.0)   # first step, sets baseline to -8, no score update
    scores_after_first = list(tracker2.scores.values())
    assert all(s == 0.5 for s in scores_after_first), "Scores changed on first step — baseline bug!"
    tracker2.update(reward=-7.5)   # second step: delta = -7.5 - (-8.0) = +0.5, signal=1
    scores_after_second = list(tracker2.scores.values())
    assert all(s > 0.5 for s in scores_after_second), "Positive delta did not increase scores"
    print("  Test 5 passed: baseline initialisation fix verified OK")

    print("")
    print("ReputationTracker — all 5 unit tests passed.")

test_reputation_tracker()


  Test 1 passed: neutral init OK
  Test 2: scores after rising rewards: [1.0, 1.0, 1.0]
  Test 2 passed: positive drift OK (mean=1.0)
  Test 3: scores after falling rewards: [0.0, 0.0, 0.0]
  Test 3 passed: negative drift OK (mean=0.0)
  Test 4 passed: all weight methods sum to 1.0 OK
  Test 5 passed: baseline initialisation fix verified OK

ReputationTracker — all 5 unit tests passed.


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6 — Actor & Critic Networks (networks.py)         ║
# ╚══════════════════════════════════════════════════════════╝

class Actor(nn.Module):
    """
    MADDPG actor with reputation-weighted message aggregation.

    Forward pass:
        1. Receive own_obs [batch, msg_start] and
           peer_msgs [batch, n_peers, msg_dim]
        2. Aggregate peer_msgs weighted by rep_weights [batch, n_peers]
        3. Concatenate own_obs + aggregated_msg → policy MLP → action logits

    The reputation weighting is the ONLY modification from standard MADDPG.
    When rep_weights is uniform, output is identical to an unmodified actor.
    """

    def __init__(self, own_obs_dim: int, n_peers: int,
                 msg_dim: int, act_dim: int, hidden_dim: int):
        super().__init__()
        self.n_peers  = n_peers
        self.msg_dim  = msg_dim
        self.act_dim  = act_dim

        input_dim = own_obs_dim + msg_dim   # own obs + aggregated msg
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, act_dim),
        )

    def forward(self,
                own_obs:     torch.Tensor,   # [batch, own_obs_dim]
                peer_msgs:   torch.Tensor,   # [batch, n_peers, msg_dim]
                rep_weights: torch.Tensor,   # [batch, n_peers]
                ) -> torch.Tensor:           # [batch, act_dim]

        # ── Reputation-weighted message aggregation ───────────────────────────
        w   = rep_weights.unsqueeze(-1)              # [batch, n_peers, 1]
        agg = (peer_msgs * w).sum(dim=1)             # [batch, msg_dim]
        # ─────────────────────────────────────────────────────────────────────

        x = torch.cat([own_obs, agg], dim=-1)        # [batch, own_obs_dim + msg_dim]
        return self.net(x)                           # [batch, act_dim]

    def select_action(self, own_obs: np.ndarray,
                      peer_msgs: np.ndarray,
                      rep_weights: np.ndarray,
                      deterministic: bool = False) -> np.ndarray:
        with torch.no_grad():
            o = torch.FloatTensor(own_obs).unsqueeze(0).to(DEVICE)
            m = torch.FloatTensor(peer_msgs).unsqueeze(0).to(DEVICE)
            w = torch.FloatTensor(rep_weights).unsqueeze(0).to(DEVICE)
            logits = self.forward(o, m, w).squeeze(0)
            if deterministic:
                action = logits.argmax().item()
            else:
                probs  = F.softmax(logits, dim=-1)
                action = torch.multinomial(probs, 1).item()
        return action


class Critic(nn.Module):
    """
    Centralised MADDPG critic: Q(all_obs, all_actions).
    Has full visibility during training (CTDE paradigm).
    """

    def __init__(self, n_agents: int, obs_dim: int,
                 act_dim: int, hidden_dim: int):
        super().__init__()
        input_dim = n_agents * (obs_dim + act_dim)
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, all_obs: torch.Tensor,
                all_acts: torch.Tensor) -> torch.Tensor:
        """
        all_obs:  [batch, n_agents, obs_dim]
        all_acts: [batch, n_agents, act_dim]
        Returns:  [batch, 1]
        """
        batch = all_obs.shape[0]
        x = torch.cat([
            all_obs.reshape(batch, -1),
            all_acts.reshape(batch, -1)
        ], dim=-1)
        return self.net(x)


# ── Verify parity: uniform weights == standard MADDPG ─────────────────────────
def test_network_parity():
    cfg_t = Config()
    own_obs_dim = cfg_t.msg_start        # 28 dims (everything before msg slice)
    actor = Actor(own_obs_dim, cfg_t.n_peers, cfg_t.msg_dim,
                  cfg_t.act_dim, cfg_t.hidden_dim)

    # Uniform weights → same as concatenating mean messages
    batch = 4
    obs   = torch.randn(batch, own_obs_dim)
    msgs  = torch.randn(batch, cfg_t.n_peers, cfg_t.msg_dim)
    w_uni = torch.ones(batch, cfg_t.n_peers) / cfg_t.n_peers

    out_uni  = actor(obs, msgs, w_uni)

    # Manually compute what unmodified MADDPG would do: simple mean
    agg_mean = msgs.mean(dim=1)
    x_manual = torch.cat([obs, agg_mean], dim=-1)
    out_man  = actor.net(x_manual)

    diff = (out_uni - out_man).abs().max().item()
    assert diff < 1e-5, f"Parity test failed, max diff: {diff}"
    print(f"✅ Network parity test: uniform weights == mean aggregation (max diff = {diff:.2e}) ✓")

    # Shape checks
    critic = Critic(cfg_t.n_agents, cfg_t.obs_dim, cfg_t.act_dim, cfg_t.hidden_dim)
    all_obs  = torch.randn(batch, cfg_t.n_agents, cfg_t.obs_dim)
    all_acts = torch.randn(batch, cfg_t.n_agents, cfg_t.act_dim)
    q = critic(all_obs, all_acts)
    assert q.shape == (batch, 1), f"Critic output shape wrong: {q.shape}"
    print(f"   Critic output shape: {q.shape} ✓")
    print(f"   Actor  output shape: {out_uni.shape} ✓")

test_network_parity()


✅ Network parity test: uniform weights == mean aggregation (max diff = 7.45e-08) ✓
   Critic output shape: torch.Size([4, 1]) ✓
   Actor  output shape: torch.Size([4, 5]) ✓


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 7 — Replay Buffer (buffer.py)                     ║
# ╚══════════════════════════════════════════════════════════╝

class ReplayBuffer:
    """
    Circular replay buffer for MADDPG.
    Stores transitions for all agents simultaneously.
    Also stores reputation scores for logging/analysis.
    """

    def __init__(self, cfg: Config):
        self.capacity   = cfg.buffer_size
        self.n_agents   = cfg.n_agents
        self.obs_dim    = cfg.obs_dim
        self.act_dim    = cfg.act_dim
        self.n_peers    = cfg.n_peers
        self.ptr        = 0
        self.size       = 0

        # Pre-allocate numpy arrays for speed
        self.obs       = np.zeros((self.capacity, self.n_agents, self.obs_dim),  dtype=np.float32)
        self.acts      = np.zeros((self.capacity, self.n_agents, self.act_dim),  dtype=np.float32)
        self.rews      = np.zeros((self.capacity, self.n_agents),                dtype=np.float32)
        self.next_obs  = np.zeros((self.capacity, self.n_agents, self.obs_dim),  dtype=np.float32)
        self.dones     = np.zeros((self.capacity, self.n_agents),                dtype=np.float32)
        self.rep_scores= np.zeros((self.capacity, self.n_agents, self.n_peers),  dtype=np.float32)

    def push(self,
             obs:        np.ndarray,   # [n_agents, obs_dim]
             acts:       np.ndarray,   # [n_agents, act_dim]
             rews:       np.ndarray,   # [n_agents]
             next_obs:   np.ndarray,   # [n_agents, obs_dim]
             dones:      np.ndarray,   # [n_agents]
             rep_scores: np.ndarray,   # [n_agents, n_peers]
             ):
        idx = self.ptr % self.capacity
        self.obs[idx]        = obs
        self.acts[idx]       = acts
        self.rews[idx]       = rews
        self.next_obs[idx]   = next_obs
        self.dones[idx]      = dones
        self.rep_scores[idx] = rep_scores
        self.ptr  += 1
        self.size  = min(self.size + 1, self.capacity)

    def sample(self, batch_size: int):
        idx = np.random.randint(0, self.size, size=batch_size)
        to_t = lambda x: torch.FloatTensor(x).to(DEVICE)
        return (
            to_t(self.obs[idx]),        # [B, n_agents, obs_dim]
            to_t(self.acts[idx]),       # [B, n_agents, act_dim]
            to_t(self.rews[idx]),       # [B, n_agents]
            to_t(self.next_obs[idx]),   # [B, n_agents, obs_dim]
            to_t(self.dones[idx]),      # [B, n_agents]
        )

    def __len__(self):
        return self.size


def test_replay_buffer():
    cfg_t  = Config()
    buf    = ReplayBuffer(cfg_t)
    n, na, od, ad, np_ = 1000, cfg_t.n_agents, cfg_t.obs_dim, cfg_t.act_dim, cfg_t.n_peers

    for _ in range(n):
        buf.push(
            np.random.randn(na, od).astype(np.float32),
            np.random.randn(na, ad).astype(np.float32),
            np.random.randn(na).astype(np.float32),
            np.random.randn(na, od).astype(np.float32),
            np.zeros(na, dtype=np.float32),
            np.random.rand(na, np_).astype(np.float32),
        )

    assert len(buf) == n
    obs, acts, rews, nobs, dones = buf.sample(256)
    assert obs.shape   == (256, na, od), f"obs shape wrong: {obs.shape}"
    assert acts.shape  == (256, na, ad)
    assert rews.shape  == (256, na)
    print(f"✅ ReplayBuffer: pushed {n} transitions, sampled batch of 256")
    print(f"   obs shape: {obs.shape}, acts shape: {acts.shape} ✓")

test_replay_buffer()


✅ ReplayBuffer: pushed 1000 transitions, sampled batch of 256
   obs shape: torch.Size([256, 4, 24]), acts shape: torch.Size([256, 4, 5]) ✓


In [ ]:
import subprocess
import sys
import importlib

# FIX for AttributeError: module 'sympy' has no attribute 'printing'
# This error typically occurs due to an incompatibility between the installed
# versions of PyTorch and SymPy. Downgrading SymPy often resolves it.
# Note: A full runtime restart (Runtime -> Restart session) is highly recommended
# after changing core library versions for full effect, especially for PyTorch.

try:
    import sympy
    current_sympy_version = getattr(sympy, '__version__', 'unknown')
    if current_sympy_version != '1.11.0': # Or another known compatible version like '1.12'
        print(f"Detected sympy version: {current_sympy_version}. Attempting to downgrade to 1.11.0 for PyTorch compatibility.")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "sympy==1.11.0"], check=True)

        # Attempt to reload relevant torch modules. This is not guaranteed to work
        # perfectly for all cases, especially if parts of torch are deeply initialized.
        if 'torch' in sys.modules:
            try:
                # These modules were in the traceback stack, so reloading them is crucial.
                if 'torch._dynamo' in sys.modules: importlib.reload(sys.modules['torch._dynamo'])
                if 'torch.optim' in sys.modules: importlib.reload(sys.modules['torch.optim'])
                if 'torch.utils._sympy.functions' in sys.modules: importlib.reload(sys.modules['torch.utils._sympy.functions'])
                print("Attempted to reload torch modules. If the error persists, please restart the runtime.")
            except Exception as reload_err:
                print(f"Warning: Failed to reload torch modules after sympy downgrade: {reload_err}")
                print("If the error persists, please restart the Colab runtime (Runtime -> Restart session) and re-run cells from Cell 2 onwards.")

except ImportError:
    print("sympy not found, installing sympy==1.11.0.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sympy==1.11.0"], check=True)
    if 'torch' in sys.modules:
        try:
            if 'torch._dynamo' in sys.modules: importlib.reload(sys.modules['torch._dynamo'])
            if 'torch.optim' in sys.modules: importlib.reload(sys.modules['torch.optim'])
            if 'torch.utils._sympy.functions' in sys.modules: importlib.reload(sys.modules['torch.utils._sympy.functions'])
            print("Attempted to reload torch modules after sympy installation.")
        except Exception as reload_err:
            print(f"Warning: Failed to reload torch modules after sympy installation: {reload_err}")
            print("If the error persists, please restart the Colab runtime (Runtime -> Restart session) and re-run cells from Cell 2 onwards.")
except Exception as e:
    print(f"An unexpected error occurred during sympy version check/fix: {e}")


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 8 — MADDPGAgent & Helper Functions                ║
# ╚══════════════════════════════════════════════════════════╝

class MADDPGAgent:
    """
    Single agent in the MADDPG framework.
    Holds actor + target_actor, critic + target_critic,
    their optimizers, and a ReputationTracker.
    """

    def __init__(self, agent_id: str, peer_ids: List[str], cfg):
        self.agent_id = agent_id
        self.peer_ids = peer_ids
        self.cfg      = cfg
        own_obs_dim   = cfg.msg_start   # dims before the comm slice

        # ── Networks ──────────────────────────────────────────────────────
        self.actor   = Actor(own_obs_dim, cfg.n_peers, cfg.msg_dim,
                             cfg.act_dim, cfg.hidden_dim).to(DEVICE)
        self.t_actor = copy.deepcopy(self.actor)

        self.critic  = Critic(cfg.n_agents, cfg.obs_dim,
                              cfg.act_dim, cfg.hidden_dim).to(DEVICE)
        self.t_critic = copy.deepcopy(self.critic)

        # ── Optimisers ────────────────────────────────────────────────────
        self.actor_opt  = optim.Adam(self.actor.parameters(),  lr=cfg.actor_lr)
        self.critic_opt = optim.Adam(self.critic.parameters(), lr=cfg.critic_lr)

        # ── Reputation tracker ────────────────────────────────────────────
        self.tracker = ReputationTracker(peer_ids, cfg)

    def get_rep_weights(self) -> np.ndarray:
        c = self.cfg.trust_condition
        if   c == "none":   return self.tracker.get_weights_uniform()
        elif c == "binary": return self.tracker.get_weights_binary(self.cfg.binary_threshold)
        else:               return self.tracker.get_weights()   # soft

    @torch.no_grad()
    def soft_update(self, tau: float):
        for p, tp in zip(self.actor.parameters(), self.t_actor.parameters()):
            tp.data.copy_(tau * p.data + (1 - tau) * tp.data)
        for p, tp in zip(self.critic.parameters(), self.t_critic.parameters()):
            tp.data.copy_(tau * p.data + (1 - tau) * tp.data)


# ── Helper functions ──────────────────────────────────────────────────────

def extract_obs_parts(obs: np.ndarray, cfg):
    """Split raw obs into own_obs and peer_msgs."""
    own_obs   = obs[:cfg.msg_start]                          # [msg_start]
    msg_block = obs[cfg.msg_start:cfg.msg_end]               # [n_peers * msg_dim]
    peer_msgs = msg_block.reshape(cfg.n_peers, cfg.msg_dim)  # [n_peers, msg_dim]
    return own_obs, peer_msgs


def onehot(action: int, dim: int) -> np.ndarray:
    v = np.zeros(dim, dtype=np.float32)
    v[action] = 1.0
    return v


def stack_agent_data(d: Dict[str, np.ndarray],
                     agent_list: List[str]) -> np.ndarray:
    """Stack per-agent arrays into [n_agents, *] numpy array."""
    return np.stack([d[a] for a in agent_list], axis=0)


# ── Quick sanity check ────────────────────────────────────────────────────
def test_maddpg_agent():
    cfg_t    = Config()
    peer_ids = ["agent_1", "agent_2", "agent_3"]
    agent    = MADDPGAgent("agent_0", peer_ids, cfg_t)

    # Check networks are on the right device
    assert next(agent.actor.parameters()).device.type == DEVICE.type
    assert next(agent.critic.parameters()).device.type == DEVICE.type

    # Check weight shapes
    w = agent.get_rep_weights()
    assert len(w) == cfg_t.n_peers, "Wrong number of weights"
    assert abs(w.sum() - 1.0) < 1e-5, "Weights do not sum to 1"

    # Check extract_obs_parts
    dummy_obs = np.random.randn(cfg_t.obs_dim).astype(np.float32)
    own_obs, peer_msgs = extract_obs_parts(dummy_obs, cfg_t)
    assert own_obs.shape   == (cfg_t.msg_start,),                "own_obs shape wrong"
    assert peer_msgs.shape == (cfg_t.n_peers, cfg_t.msg_dim),   "peer_msgs shape wrong"

    n_actor  = sum(p.numel() for p in agent.actor.parameters())
    n_critic = sum(p.numel() for p in agent.critic.parameters())
    print("MADDPGAgent sanity check passed.")
    print("  own_obs_dim : " + str(cfg_t.msg_start))
    print("  actor params: " + str(n_actor))
    print("  critic params: " + str(n_critic))
    print("  rep weights : " + str(w) + "  (sum=" + str(round(w.sum(),4)) + ")")

test_maddpg_agent()


Detected sympy version: 1.14.0. Attempting to downgrade to 1.11.0 for PyTorch compatibility.
If the error persists, please restart the Colab runtime (Runtime -> Restart session) and re-run cells from Cell 2 onwards.


AttributeError: module 'sympy' has no attribute 'printing'

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 9 — Training & Evaluation Loop (train.py)         ║
# ╚══════════════════════════════════════════════════════════╝

def update_agents(agents, buffer, cfg):
    """One MADDPG gradient update step for all agents."""
    if len(buffer) < cfg.batch_size:
        return 0.0, 0.0

    obs_b, acts_b, rews_b, nobs_b, dones_b = buffer.sample(cfg.batch_size)
    total_closs, total_aloss = 0.0, 0.0

    for i, agent in enumerate(agents):
        # ── Critic update ─────────────────────────────────────────────────────
        with torch.no_grad():
            t_acts_list = []
            for j, ag in enumerate(agents):
                o_j = nobs_b[:, j, :cfg.msg_start]
                m_j = nobs_b[:, j, cfg.msg_start:cfg.msg_end]
                m_j = m_j.view(-1, cfg.n_peers, cfg.msg_dim)
                # Target actor always uses uniform weights (stable training target)
                w_j = torch.ones(cfg.batch_size, cfg.n_peers, device=DEVICE) / cfg.n_peers
                t_logits = ag.t_actor(o_j, m_j, w_j)
                t_act    = F.softmax(t_logits, dim=-1)
                t_acts_list.append(t_act)
            t_acts = torch.stack(t_acts_list, dim=1)
            next_q = agent.t_critic(nobs_b, t_acts)
            r_i    = rews_b[:, i:i+1]
            d_i    = dones_b[:, i:i+1]
            target = r_i + cfg.gamma * (1 - d_i) * next_q

        current_q = agent.critic(obs_b, acts_b)
        closs     = F.mse_loss(current_q, target)
        agent.critic_opt.zero_grad()
        closs.backward()
        torch.nn.utils.clip_grad_norm_(agent.critic.parameters(), 1.0)
        agent.critic_opt.step()

        # ── Actor update — FIX: use CURRENT reputation weights ────────────────
        # The actor must be trained WITH reputation weights so it learns to
        # exploit the trust signal. Using uniform weights here (the original bug)
        # meant the network was never exposed to reputation-modulated inputs
        # during training, so it could not benefit from them at inference time.
        o_i = obs_b[:, i, :cfg.msg_start]
        m_i = obs_b[:, i, cfg.msg_start:cfg.msg_end].view(-1, cfg.n_peers, cfg.msg_dim)

        # Get current rep weights for this agent (shape: [n_peers])
        rep_w = torch.FloatTensor(agent.get_rep_weights()).to(DEVICE)
        # Expand to batch: [1, n_peers] -> [batch, n_peers]
        rep_w_batch = rep_w.unsqueeze(0).expand(cfg.batch_size, -1)

        logits  = agent.actor(o_i, m_i, rep_w_batch)
        new_act = F.softmax(logits, dim=-1)
        acts_q  = acts_b.clone()
        acts_q[:, i, :] = new_act
        aloss = -agent.critic(obs_b, acts_q).mean()
        agent.actor_opt.zero_grad()
        aloss.backward()
        torch.nn.utils.clip_grad_norm_(agent.actor.parameters(), 1.0)
        agent.actor_opt.step()

        agent.soft_update(cfg.tau)
        total_closs += closs.item()
        total_aloss += aloss.item()

    return total_closs / cfg.n_agents, total_aloss / cfg.n_agents


def run_eval_episode(agents, cfg, seed=0):
    """One evaluation episode — deterministic actions, no buffer push.
    FIX: tracker is updated during eval so rep scores reflect eval behaviour."""
    env      = DefectorWrapper(cfg, seed=seed)
    obs_dict = env.reset(seed=seed)
    ep_reward = 0.0

    for step in range(cfg.max_steps):
        actions = {}
        for ag in agents:
            aid     = ag.agent_id
            obs_raw = obs_dict.get(aid, np.zeros(cfg.obs_dim, dtype=np.float32))
            own_obs, peer_msgs = extract_obs_parts(obs_raw, cfg)
            w      = ag.get_rep_weights()
            act_oh = np.eye(cfg.act_dim)[
                ag.actor.select_action(own_obs, peer_msgs, w, deterministic=True)
            ]
            actions[aid] = act_oh

        obs_dict, rews, dones = env.step_all(actions)
        ep_reward += sum(rews.values())

        # FIX: update tracker during eval so scores reflect actual eval dynamics
        for ag in agents:
            ag.tracker.update(rews.get(ag.agent_id, 0.0))

    rep_snap = {ag.agent_id: dict(ag.tracker.scores) for ag in agents}
    env.close()
    return ep_reward, rep_snap


def train(cfg, verbose=True):
    """Full training run. Returns results dict with curves and metrics."""
    set_seed(cfg.seed)
    agent_ids = ["agent_" + str(i) for i in range(cfg.n_agents)]

    agents = []
    for aid in agent_ids:
        peer_ids = [a for a in agent_ids if a != aid]
        agents.append(MADDPGAgent(aid, peer_ids, cfg))

    buffer  = ReplayBuffer(cfg)
    results = {
        "train_rewards":  [],
        "eval_rewards":   [],
        "eval_episodes":  [],
        "rep_scores_log": [],
        "critic_losses":  [],
        "actor_losses":   [],
    }

    total_steps = 0
    pbar = trange(cfg.n_episodes,
                  desc="[" + cfg.trust_condition.upper() + "|" + str(cfg.n_defectors) + "def]",
                  disable=not verbose)

    for episode in pbar:
        env      = DefectorWrapper(cfg, seed=cfg.seed + episode)
        obs_dict = env.reset(seed=cfg.seed + episode)
        ep_reward = 0.0

        for step in range(cfg.max_steps):
            actions = {}

            for ag in agents:
                aid     = ag.agent_id
                obs_raw = obs_dict.get(aid, np.zeros(cfg.obs_dim, dtype=np.float32))
                own_obs, peer_msgs = extract_obs_parts(obs_raw, cfg)
                w = ag.get_rep_weights()

                if total_steps < cfg.warmup_steps:
                    act_int = env.sample_action(aid)
                else:
                    act_int = ag.actor.select_action(own_obs, peer_msgs, w)

                actions[aid] = onehot(act_int, cfg.act_dim)

            next_obs_dict, rews, dones = env.step_all(actions)
            ep_reward += sum(rews.values())

            # Reputation update BEFORE buffer push so scores reflect current step
            for ag in agents:
                ag.tracker.update(rews.get(ag.agent_id, 0.0))

            rep_scores_arr = np.array(
                [list(ag.tracker.scores.values()) for ag in agents],
                dtype=np.float32
            )
            buffer.push(
                obs        = stack_agent_data(obs_dict,      agent_ids),
                acts       = stack_agent_data(actions,       agent_ids),
                rews       = np.array([rews.get(a, 0.) for a in agent_ids], dtype=np.float32),
                next_obs   = stack_agent_data(next_obs_dict, agent_ids),
                dones      = np.array([float(dones.get(a, False)) for a in agent_ids], dtype=np.float32),
                rep_scores = rep_scores_arr,
            )

            obs_dict    = next_obs_dict
            total_steps += 1

            if total_steps >= cfg.warmup_steps and total_steps % 2 == 0:
                cl, al = update_agents(agents, buffer, cfg)
                results["critic_losses"].append(cl)
                results["actor_losses"].append(al)

        results["train_rewards"].append(ep_reward)
        env.close()

        if (episode + 1) % cfg.eval_interval == 0:
            eval_rews = []
            rep_snap_last = {}
            for e in range(cfg.eval_episodes):
                er, rep_snap = run_eval_episode(agents, cfg, seed=9000 + e)
                eval_rews.append(er)
                if e == cfg.eval_episodes - 1:
                    rep_snap_last = rep_snap

            mean_eval = float(np.mean(eval_rews))
            results["eval_rewards"].append(mean_eval)
            results["eval_episodes"].append(episode + 1)
            results["rep_scores_log"].append({
                "episode":          episode + 1,
                "mean_eval_reward": mean_eval,
                "rep_scores":       rep_snap_last,
            })

            if verbose:
                pbar.set_postfix({"eval": str(round(mean_eval, 1)), "buf": len(buffer)})

    return results

print("Training loop defined — ready to run experiments.")
print("FIXES APPLIED:")
print("  1. Actor update now uses current reputation weights (not uniform)")
print("  2. Reputation baseline initialised from first real reward (not 0.0)")
print("  3. Tracker updated during eval episodes for accurate score logging")


In [ ]:
# ── Dependency guard ─────────────────────────────────────────────────────
import sys as _sys
if "Config" not in dir():
    print("Re-running setup cells...")
    _cells = [
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 2 — Imports & Environment Check                   ║\n# ╚══════════════════════════════════════════════════════════╝\n\nimport os, sys, time, random, copy\nfrom dataclasses import dataclass\nfrom typing import Dict, List, Tuple, Optional\n\nimport copy\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.optim as optim\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport pandas as pd\nfrom scipy import stats\nfrom tqdm.auto import tqdm, trange\n\ndef set_seed(seed: int):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\nset_seed(42)\n\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nprint("Device: " + str(DEVICE))\nif DEVICE.type == "cuda":\n    print("  GPU : " + torch.cuda.get_device_name(0))\n\n# ── mpe2 observation layout for simple_spread_v3, N=4 ────────────────────\n# [self_vel(2), self_pos(2), landmark_rel_pos(4*2=8),\n#  other_agent_rel_pos(3*2=6), communication(3*2=6)]\n# Total = 2+2+8+6+6 = 24\nfrom mpe2 import simple_spread_v3\n\nenv = simple_spread_v3.env(N=4, local_ratio=0.5, render_mode=None)\nenv.reset(seed=0)\nfor agent in env.agent_iter():\n    obs, rew, term, trunc, info = env.last()\n    env.step(env.action_space(agent).sample())\n    break\nenv.close()\n\nprint("mpe2 simple_spread_v3 (N=4):")\nprint("  obs shape : " + str(obs.shape))\nprint("  agents    : " + str(env.agents))\n\n# Confirm layout matches our constants\nN = 4\nexpected_obs_dim = 2 + 2 + N*2 + (N-1)*2 + (N-1)*2   # = 24\nassert obs.shape == (expected_obs_dim,), (\n    "Obs shape mismatch: got " + str(obs.shape) +\n    ", expected (" + str(expected_obs_dim) + ",)"\n)\nprint("  obs_dim   : " + str(expected_obs_dim) + "  [2 vel + 2 pos + 8 landmarks + 6 others + 6 comms]")\nprint("  msg_start : 18  (slice [18:24] = 3 peers x 2D comm vectors)")\nprint("  msg_end   : 24")\nprint("  Shape check passed.")\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 3 — Hyperparameter Config                         ║\n# ╚══════════════════════════════════════════════════════════╝\n\n@dataclass\nclass Config:\n    # ── Environment ──────────────────────────────────────────\n    n_agents:        int   = 4\n    n_landmarks:     int   = 4\n    local_ratio:     float = 0.5\n    max_steps:       int   = 50\n    #\n    # mpe2 obs layout for N=4:\n    # [self_vel(2), self_pos(2), landmark_rel_pos(8),\n    #  other_agent_rel_pos(6), communication(6)]\n    # Total obs_dim = 24\n    # msg_start = 18  (start of communication slice)\n    # msg_end   = 24  (end of communication slice)\n    obs_dim:         int   = 24   # 2+2+8+6+6 for N=4\n    act_dim:         int   = 5    # Discrete(5)\n    msg_dim:         int   = 2    # each peer sends a 2D comm vector\n    msg_start:       int   = 18   # index where comm slice begins\n    msg_end:         int   = 24   # index where comm slice ends\n\n    # ── Defectors ────────────────────────────────────────────\n    n_defectors:     int   = 1\n    noise_scale:     float = 1.0\n\n    # ── Trust condition ──────────────────────────────────────\n    # "none"   = Condition A: equal weights\n    # "binary" = Condition B: hard threshold gate\n    # "soft"   = Condition C: EMA reputation weighting (RWC)\n    trust_condition:    str   = "soft"\n    rep_alpha:          float = 0.05\n    rep_baseline_alpha: float = 0.01\n    rep_temperature:    float = 5.0\n    binary_threshold:   float = 0.5\n\n    # ── MADDPG ───────────────────────────────────────────────\n    hidden_dim:      int   = 128\n    actor_lr:        float = 1e-3\n    critic_lr:       float = 1e-3\n    gamma:           float = 0.95\n    tau:             float = 0.01\n    buffer_size:     int   = 100_000\n    batch_size:      int   = 256\n    warmup_steps:    int   = 1000\n\n    # ── Training ─────────────────────────────────────────────\n    n_episodes:      int   = 500\n    eval_interval:   int   = 25\n    eval_episodes:   int   = 20\n    seed:            int   = 42\n\n    # ── Output ───────────────────────────────────────────────\n    save_dir:        str   = "/content/marl_trust"\n\n    def __post_init__(self):\n        import os\n        os.makedirs(self.save_dir, exist_ok=True)\n        self.n_peers = self.n_agents - 1\n        all_ids = ["agent_" + str(i) for i in range(self.n_agents)]\n        self.defector_ids = set(all_ids[-self.n_defectors:]) if self.n_defectors > 0 else set()\n        self.honest_ids   = set(all_ids) - self.defector_ids\n\ncfg = Config()\nprint("Config created.")\nprint("  obs_dim   = " + str(cfg.obs_dim))\nprint("  msg_start = " + str(cfg.msg_start) + \n      "  (slice [" + str(cfg.msg_start) + ":" + str(cfg.msg_end) + \n      "] = " + str(cfg.n_peers) + " peers x " + str(cfg.msg_dim) + "D comms)")\nprint("  defectors = " + str(cfg.defector_ids))\n\n# Sanity check: msg slice must exactly fit n_peers * msg_dim\nassert cfg.msg_end - cfg.msg_start == cfg.n_peers * cfg.msg_dim, (\n    "Msg slice length " + str(cfg.msg_end - cfg.msg_start) +\n    " != n_peers*msg_dim " + str(cfg.n_peers * cfg.msg_dim)\n)\nprint("  Slice sanity check passed.")\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 4 — DefectorWrapper (env_wrapper.py)              ║\n# ╚══════════════════════════════════════════════════════════╝\n\n# MPE environments are now in the mpe2 package (moved from mpe2)\nfrom mpe2 import simple_spread_v3\n\nclass DefectorWrapper:\n    """\n    Wraps simple_spread_v3 and corrupts outgoing communication messages\n    from designated defector agents by injecting Gaussian noise.\n\n    Compatible with PettingZoo 1.22.x and 1.24.x.\n\n    Design:\n    - Only obs[msg_start:msg_end] is corrupted for honest agents.\n    - Defectors\' own observations remain clean (they can still sense\n      positions), making the problem harder for honest agents.\n    - The wrapper never modifies the underlying environment state.\n    """\n\n    def __init__(self, cfg, seed: int = None):\n        self.cfg  = cfg\n        self.rng  = np.random.default_rng(seed)\n        self._env = simple_spread_v3.env(\n            N           = cfg.n_agents,\n            local_ratio = cfg.local_ratio,\n            render_mode = None,\n            max_cycles  = cfg.max_steps,\n        )\n        self.agents    = self._env.possible_agents   # stable ordered list\n        self.n_agents  = len(self.agents)\n        self.agent_idx = {a: i for i, a in enumerate(self.agents)}\n\n    def reset(self, seed=None):\n        self._env.reset(seed=seed)\n        return self._collect_obs()\n\n    def step_all(self, actions: Dict[str, np.ndarray]):\n        """\n        Execute one full round: each agent acts once.\n        actions: {agent_id: one-hot np.ndarray}\n        Returns: obs, rewards, dones (all dicts)\n        """\n        rewards, dones = {a: 0.0 for a in self.agents}, {a: False for a in self.agents}\n\n        for agent in self._env.agent_iter():\n            obs_raw, rew, term, trunc, info = self._env.last()\n            done = term or trunc\n            rewards[agent] = float(rew)\n            dones[agent]   = done\n            if done:\n                self._env.step(None)\n            else:\n                act_oh = actions.get(agent)\n                if act_oh is not None:\n                    act_int = int(np.argmax(act_oh))\n                else:\n                    act_int = self._env.action_space(agent).sample()\n                self._env.step(act_int)\n\n        obs = self._collect_obs()\n        return obs, rewards, dones\n\n    def _collect_obs(self) -> Dict[str, np.ndarray]:\n        obs = {}\n        for agent in self.agents:\n            try:\n                o = self._env.observe(agent)\n                if o is None:\n                    o = np.zeros(self.cfg.obs_dim, dtype=np.float32)\n                else:\n                    o = np.array(o, dtype=np.float32).copy()\n            except Exception:\n                o = np.zeros(self.cfg.obs_dim, dtype=np.float32)\n            obs[agent] = self._maybe_corrupt(agent, o)\n        return obs\n\n    def _maybe_corrupt(self, observer: str, obs: np.ndarray) -> np.ndarray:\n        """Add Gaussian noise to peer-message slice for honest observers."""\n        if not self.cfg.defector_ids or observer in self.cfg.defector_ids:\n            return obs\n        noise = (self.rng.standard_normal(self.cfg.msg_end - self.cfg.msg_start)\n                 .astype(np.float32) * self.cfg.noise_scale)\n        obs[self.cfg.msg_start:self.cfg.msg_end] += noise\n        return obs\n\n    def sample_action(self, agent_id: str) -> int:\n        return self._env.action_space(agent_id).sample()\n\n    def close(self):\n        self._env.close()\n\n\n# ── Unit test ─────────────────────────────────────────────────────────────────\ndef test_defector_wrapper():\n    from dataclasses import replace\n\n    # With 1 defector\n    cfg_dirty = Config(n_defectors=1)\n    env_dirty = DefectorWrapper(cfg_dirty, seed=0)\n    env_dirty.reset(seed=0)\n    dummy = np.zeros(cfg_dirty.obs_dim, dtype=np.float32)\n    corrupted = env_dirty._maybe_corrupt("agent_0", dummy.copy())\n    std_dirty = corrupted[cfg_dirty.msg_start:cfg_dirty.msg_end].std()\n\n    # With 0 defectors\n    cfg_clean = Config(n_defectors=0)\n    env_clean = DefectorWrapper(cfg_clean, seed=0)\n    env_clean.reset(seed=0)\n    clean = env_clean._maybe_corrupt("agent_0", dummy.copy())\n    std_clean = clean[cfg_clean.msg_start:cfg_clean.msg_end].std()\n\n    print(f"✅ DefectorWrapper unit test")\n    print(f"   Msg slice std with  defector : {std_dirty:.4f}  (should be > 0.1)")\n    print(f"   Msg slice std without defector: {std_clean:.4f}  (should be 0.0)")\n    assert std_dirty > 0.1, "Corruption not applied!"\n    assert std_clean == 0.0, "Clean env is corrupted — bug!"\n\n    # Smoke: run a few steps\n    env2 = DefectorWrapper(Config(n_defectors=1), seed=7)\n    obs  = env2.reset(seed=7)\n    assert len(obs) == 4, f"Expected 4 agent obs, got {len(obs)}"\n    dummy_acts = {a: np.eye(5)[env2.sample_action(a)] for a in env2.agents}\n    obs2, rews, dones = env2.step_all(dummy_acts)\n    assert len(rews) == 4\n    env2.close()\n    env_dirty.close()\n    env_clean.close()\n    print("   Step test passed ✓")\n    print("   All assertions passed ✓")\n\ntest_defector_wrapper()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 5 — ReputationTracker (reputation.py)             ║\n# ╚══════════════════════════════════════════════════════════╝\n\nfrom scipy.special import softmax as scipy_softmax\n\nclass ReputationTracker:\n    """\n    Maintains a continuous reputation score r[j] in [0,1] for each peer j.\n\n    FIX: baseline is now initialised to None and set from the first real\n    reward, preventing the \'always negative delta\' problem that caused all\n    scores to drift to 0 then snap to 1.0 when rewards improved.\n\n    Update rule (per step):\n        delta  = reward - baseline\n        signal = 1.0 if delta > 0 else 0.0\n        r[j]  <- (1 - alpha) * r[j] + alpha * signal  for all j\n        baseline <- (1 - b_alpha) * baseline + b_alpha * reward\n    """\n\n    def __init__(self, peer_ids, cfg):\n        self.peer_ids    = list(peer_ids)\n        self.alpha       = cfg.rep_alpha\n        self.b_alpha     = cfg.rep_baseline_alpha\n        self.temperature = cfg.rep_temperature\n        self.scores      = {j: 0.5 for j in peer_ids}\n        self.baseline    = None    # FIX: initialise lazily from first reward\n\n    def update(self, reward):\n        """Called once per step with this agent\'s observed reward."""\n        # FIX: initialise baseline to first real reward so delta starts near 0\n        if self.baseline is None:\n            self.baseline = reward\n            return  # skip update on very first step\n\n        delta  = reward - self.baseline\n        signal = 1.0 if delta > 0 else 0.0\n\n        self.baseline = (1 - self.b_alpha) * self.baseline + self.b_alpha * reward\n\n        for j in self.peer_ids:\n            self.scores[j] = (\n                (1 - self.alpha) * self.scores[j] + self.alpha * signal\n            )\n\n    def get_weights(self):\n        """Softmax-normalised weights, ordered by peer_ids."""\n        scores_arr = np.array([self.scores[j] for j in self.peer_ids],\n                               dtype=np.float32)\n        return scipy_softmax(scores_arr * self.temperature).astype(np.float32)\n\n    def get_weights_binary(self, threshold=0.5):\n        """Condition B: hard binary gate, re-normalised."""\n        scores_arr = np.array([self.scores[j] for j in self.peer_ids],\n                               dtype=np.float32)\n        weights = (scores_arr >= threshold).astype(np.float32)\n        total = weights.sum()\n        if total == 0:\n            weights = np.ones(len(self.peer_ids), dtype=np.float32)\n            total   = float(len(self.peer_ids))\n        return weights / total\n\n    def get_weights_uniform(self):\n        """Condition A: no trust — uniform weights."""\n        n = len(self.peer_ids)\n        return np.ones(n, dtype=np.float32) / n\n\n    def get_scores_dict(self):\n        return dict(self.scores)\n\n    def reset(self):\n        self.scores   = {j: 0.5 for j in self.peer_ids}\n        self.baseline = None   # FIX: reset to None, not 0.0\n\n\n# ── Unit tests ────────────────────────────────────────────────────────────────\ndef test_reputation_tracker():\n    cfg_t = Config()\n    peers = ["agent_1", "agent_2", "agent_3"]\n    tracker = ReputationTracker(peers, cfg_t)\n\n    # Test 1: neutral init\n    w = tracker.get_weights()\n    assert abs(w.sum() - 1.0) < 1e-5, "Weights do not sum to 1"\n    assert abs(w[0] - w[1]) < 1e-5, "Initial weights not uniform"\n    print("  Test 1 passed: neutral init OK")\n\n    # Test 2: positive drift — rewards consistently above baseline\n    tracker.reset()\n    for i in range(200):\n        tracker.update(reward=-5.0 + i * 0.05)   # rising rewards\n    scores = list(tracker.scores.values())\n    print("  Test 2: scores after rising rewards: " + str([round(s,3) for s in scores]))\n    assert all(s > 0.5 for s in scores), "Scores did not drift up: " + str(scores)\n    print("  Test 2 passed: positive drift OK (mean=" + str(round(float(np.mean(scores)),3)) + ")")\n\n    # Test 3: negative drift — rewards consistently below baseline\n    tracker.reset()\n    for i in range(200):\n        tracker.update(reward=-5.0 - i * 0.05)   # falling rewards\n    scores = list(tracker.scores.values())\n    print("  Test 3: scores after falling rewards: " + str([round(s,3) for s in scores]))\n    assert all(s < 0.5 for s in scores), "Scores did not drift down: " + str(scores)\n    print("  Test 3 passed: negative drift OK (mean=" + str(round(float(np.mean(scores)),3)) + ")")\n\n    # Test 4: weights always sum to 1\n    for _ in range(50):\n        tracker.update(reward=np.random.randn() - 5.0)\n    for method in ["get_weights", "get_weights_binary", "get_weights_uniform"]:\n        w = getattr(tracker, method)()\n        assert abs(w.sum() - 1.0) < 1e-5, method + " does not sum to 1"\n    print("  Test 4 passed: all weight methods sum to 1.0 OK")\n\n    # Test 5: FIX verification — baseline initialised from first reward\n    tracker2 = ReputationTracker(peers, cfg_t)\n    tracker2.update(reward=-8.0)   # first step, sets baseline to -8, no score update\n    scores_after_first = list(tracker2.scores.values())\n    assert all(s == 0.5 for s in scores_after_first), "Scores changed on first step — baseline bug!"\n    tracker2.update(reward=-7.5)   # second step: delta = -7.5 - (-8.0) = +0.5, signal=1\n    scores_after_second = list(tracker2.scores.values())\n    assert all(s > 0.5 for s in scores_after_second), "Positive delta did not increase scores"\n    print("  Test 5 passed: baseline initialisation fix verified OK")\n\n    print("")\n    print("ReputationTracker — all 5 unit tests passed.")\n\ntest_reputation_tracker()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 6 — Actor & Critic Networks (networks.py)         ║\n# ╚══════════════════════════════════════════════════════════╝\n\nclass Actor(nn.Module):\n    """\n    MADDPG actor with reputation-weighted message aggregation.\n\n    Forward pass:\n        1. Receive own_obs [batch, msg_start] and\n           peer_msgs [batch, n_peers, msg_dim]\n        2. Aggregate peer_msgs weighted by rep_weights [batch, n_peers]\n        3. Concatenate own_obs + aggregated_msg → policy MLP → action logits\n\n    The reputation weighting is the ONLY modification from standard MADDPG.\n    When rep_weights is uniform, output is identical to an unmodified actor.\n    """\n\n    def __init__(self, own_obs_dim: int, n_peers: int,\n                 msg_dim: int, act_dim: int, hidden_dim: int):\n        super().__init__()\n        self.n_peers  = n_peers\n        self.msg_dim  = msg_dim\n        self.act_dim  = act_dim\n\n        input_dim = own_obs_dim + msg_dim   # own obs + aggregated msg\n        self.net = nn.Sequential(\n            nn.Linear(input_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, act_dim),\n        )\n\n    def forward(self,\n                own_obs:     torch.Tensor,   # [batch, own_obs_dim]\n                peer_msgs:   torch.Tensor,   # [batch, n_peers, msg_dim]\n                rep_weights: torch.Tensor,   # [batch, n_peers]\n                ) -> torch.Tensor:           # [batch, act_dim]\n\n        # ── Reputation-weighted message aggregation ───────────────────────────\n        w   = rep_weights.unsqueeze(-1)              # [batch, n_peers, 1]\n        agg = (peer_msgs * w).sum(dim=1)             # [batch, msg_dim]\n        # ─────────────────────────────────────────────────────────────────────\n\n        x = torch.cat([own_obs, agg], dim=-1)        # [batch, own_obs_dim + msg_dim]\n        return self.net(x)                           # [batch, act_dim]\n\n    def select_action(self, own_obs: np.ndarray,\n                      peer_msgs: np.ndarray,\n                      rep_weights: np.ndarray,\n                      deterministic: bool = False) -> np.ndarray:\n        with torch.no_grad():\n            o = torch.FloatTensor(own_obs).unsqueeze(0).to(DEVICE)\n            m = torch.FloatTensor(peer_msgs).unsqueeze(0).to(DEVICE)\n            w = torch.FloatTensor(rep_weights).unsqueeze(0).to(DEVICE)\n            logits = self.forward(o, m, w).squeeze(0)\n            if deterministic:\n                action = logits.argmax().item()\n            else:\n                probs  = F.softmax(logits, dim=-1)\n                action = torch.multinomial(probs, 1).item()\n        return action\n\n\nclass Critic(nn.Module):\n    """\n    Centralised MADDPG critic: Q(all_obs, all_actions).\n    Has full visibility during training (CTDE paradigm).\n    """\n\n    def __init__(self, n_agents: int, obs_dim: int,\n                 act_dim: int, hidden_dim: int):\n        super().__init__()\n        input_dim = n_agents * (obs_dim + act_dim)\n        self.net = nn.Sequential(\n            nn.Linear(input_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, 1),\n        )\n\n    def forward(self, all_obs: torch.Tensor,\n                all_acts: torch.Tensor) -> torch.Tensor:\n        """\n        all_obs:  [batch, n_agents, obs_dim]\n        all_acts: [batch, n_agents, act_dim]\n        Returns:  [batch, 1]\n        """\n        batch = all_obs.shape[0]\n        x = torch.cat([\n            all_obs.reshape(batch, -1),\n            all_acts.reshape(batch, -1)\n        ], dim=-1)\n        return self.net(x)\n\n\n# ── Verify parity: uniform weights == standard MADDPG ─────────────────────────\ndef test_network_parity():\n    cfg_t = Config()\n    own_obs_dim = cfg_t.msg_start        # 28 dims (everything before msg slice)\n    actor = Actor(own_obs_dim, cfg_t.n_peers, cfg_t.msg_dim,\n                  cfg_t.act_dim, cfg_t.hidden_dim)\n\n    # Uniform weights → same as concatenating mean messages\n    batch = 4\n    obs   = torch.randn(batch, own_obs_dim)\n    msgs  = torch.randn(batch, cfg_t.n_peers, cfg_t.msg_dim)\n    w_uni = torch.ones(batch, cfg_t.n_peers) / cfg_t.n_peers\n\n    out_uni  = actor(obs, msgs, w_uni)\n\n    # Manually compute what unmodified MADDPG would do: simple mean\n    agg_mean = msgs.mean(dim=1)\n    x_manual = torch.cat([obs, agg_mean], dim=-1)\n    out_man  = actor.net(x_manual)\n\n    diff = (out_uni - out_man).abs().max().item()\n    assert diff < 1e-5, f"Parity test failed, max diff: {diff}"\n    print(f"✅ Network parity test: uniform weights == mean aggregation (max diff = {diff:.2e}) ✓")\n\n    # Shape checks\n    critic = Critic(cfg_t.n_agents, cfg_t.obs_dim, cfg_t.act_dim, cfg_t.hidden_dim)\n    all_obs  = torch.randn(batch, cfg_t.n_agents, cfg_t.obs_dim)\n    all_acts = torch.randn(batch, cfg_t.n_agents, cfg_t.act_dim)\n    q = critic(all_obs, all_acts)\n    assert q.shape == (batch, 1), f"Critic output shape wrong: {q.shape}"\n    print(f"   Critic output shape: {q.shape} ✓")\n    print(f"   Actor  output shape: {out_uni.shape} ✓")\n\ntest_network_parity()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 7 — Replay Buffer (buffer.py)                     ║\n# ╚══════════════════════════════════════════════════════════╝\n\nclass ReplayBuffer:\n    """\n    Circular replay buffer for MADDPG.\n    Stores transitions for all agents simultaneously.\n    Also stores reputation scores for logging/analysis.\n    """\n\n    def __init__(self, cfg: Config):\n        self.capacity   = cfg.buffer_size\n        self.n_agents   = cfg.n_agents\n        self.obs_dim    = cfg.obs_dim\n        self.act_dim    = cfg.act_dim\n        self.n_peers    = cfg.n_peers\n        self.ptr        = 0\n        self.size       = 0\n\n        # Pre-allocate numpy arrays for speed\n        self.obs       = np.zeros((self.capacity, self.n_agents, self.obs_dim),  dtype=np.float32)\n        self.acts      = np.zeros((self.capacity, self.n_agents, self.act_dim),  dtype=np.float32)\n        self.rews      = np.zeros((self.capacity, self.n_agents),                dtype=np.float32)\n        self.next_obs  = np.zeros((self.capacity, self.n_agents, self.obs_dim),  dtype=np.float32)\n        self.dones     = np.zeros((self.capacity, self.n_agents),                dtype=np.float32)\n        self.rep_scores= np.zeros((self.capacity, self.n_agents, self.n_peers),  dtype=np.float32)\n\n    def push(self,\n             obs:        np.ndarray,   # [n_agents, obs_dim]\n             acts:       np.ndarray,   # [n_agents, act_dim]\n             rews:       np.ndarray,   # [n_agents]\n             next_obs:   np.ndarray,   # [n_agents, obs_dim]\n             dones:      np.ndarray,   # [n_agents]\n             rep_scores: np.ndarray,   # [n_agents, n_peers]\n             ):\n        idx = self.ptr % self.capacity\n        self.obs[idx]        = obs\n        self.acts[idx]       = acts\n        self.rews[idx]       = rews\n        self.next_obs[idx]   = next_obs\n        self.dones[idx]      = dones\n        self.rep_scores[idx] = rep_scores\n        self.ptr  += 1\n        self.size  = min(self.size + 1, self.capacity)\n\n    def sample(self, batch_size: int):\n        idx = np.random.randint(0, self.size, size=batch_size)\n        to_t = lambda x: torch.FloatTensor(x).to(DEVICE)\n        return (\n            to_t(self.obs[idx]),        # [B, n_agents, obs_dim]\n            to_t(self.acts[idx]),       # [B, n_agents, act_dim]\n            to_t(self.rews[idx]),       # [B, n_agents]\n            to_t(self.next_obs[idx]),   # [B, n_agents, obs_dim]\n            to_t(self.dones[idx]),      # [B, n_agents]\n        )\n\n    def __len__(self):\n        return self.size\n\n\ndef test_replay_buffer():\n    cfg_t  = Config()\n    buf    = ReplayBuffer(cfg_t)\n    n, na, od, ad, np_ = 1000, cfg_t.n_agents, cfg_t.obs_dim, cfg_t.act_dim, cfg_t.n_peers\n\n    for _ in range(n):\n        buf.push(\n            np.random.randn(na, od).astype(np.float32),\n            np.random.randn(na, ad).astype(np.float32),\n            np.random.randn(na).astype(np.float32),\n            np.random.randn(na, od).astype(np.float32),\n            np.zeros(na, dtype=np.float32),\n            np.random.rand(na, np_).astype(np.float32),\n        )\n\n    assert len(buf) == n\n    obs, acts, rews, nobs, dones = buf.sample(256)\n    assert obs.shape   == (256, na, od), f"obs shape wrong: {obs.shape}"\n    assert acts.shape  == (256, na, ad)\n    assert rews.shape  == (256, na)\n    print(f"✅ ReplayBuffer: pushed {n} transitions, sampled batch of 256")\n    print(f"   obs shape: {obs.shape}, acts shape: {acts.shape} ✓")\n\ntest_replay_buffer()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 8 — MADDPGAgent & Helper Functions                ║\n# ╚══════════════════════════════════════════════════════════╝\n\nclass MADDPGAgent:\n    """\n    Single agent in the MADDPG framework.\n    Holds actor + target_actor, critic + target_critic,\n    their optimizers, and a ReputationTracker.\n    """\n\n    def __init__(self, agent_id: str, peer_ids: List[str], cfg):\n        self.agent_id = agent_id\n        self.peer_ids = peer_ids\n        self.cfg      = cfg\n        own_obs_dim   = cfg.msg_start   # dims before the comm slice\n\n        # ── Networks ──────────────────────────────────────────────────────\n        self.actor   = Actor(own_obs_dim, cfg.n_peers, cfg.msg_dim,\n                             cfg.act_dim, cfg.hidden_dim).to(DEVICE)\n        self.t_actor = copy.deepcopy(self.actor)\n\n        self.critic  = Critic(cfg.n_agents, cfg.obs_dim,\n                              cfg.act_dim, cfg.hidden_dim).to(DEVICE)\n        self.t_critic = copy.deepcopy(self.critic)\n\n        # ── Optimisers ────────────────────────────────────────────────────\n        self.actor_opt  = optim.Adam(self.actor.parameters(),  lr=cfg.actor_lr)\n        self.critic_opt = optim.Adam(self.critic.parameters(), lr=cfg.critic_lr)\n\n        # ── Reputation tracker ────────────────────────────────────────────\n        self.tracker = ReputationTracker(peer_ids, cfg)\n\n    def get_rep_weights(self) -> np.ndarray:\n        c = self.cfg.trust_condition\n        if   c == "none":   return self.tracker.get_weights_uniform()\n        elif c == "binary": return self.tracker.get_weights_binary(self.cfg.binary_threshold)\n        else:               return self.tracker.get_weights()   # soft\n\n    @torch.no_grad()\n    def soft_update(self, tau: float):\n        for p, tp in zip(self.actor.parameters(), self.t_actor.parameters()):\n            tp.data.copy_(tau * p.data + (1 - tau) * tp.data)\n        for p, tp in zip(self.critic.parameters(), self.t_critic.parameters()):\n            tp.data.copy_(tau * p.data + (1 - tau) * tp.data)\n\n\n# ── Helper functions ──────────────────────────────────────────────────────\n\ndef extract_obs_parts(obs: np.ndarray, cfg):\n    """Split raw obs into own_obs and peer_msgs."""\n    own_obs   = obs[:cfg.msg_start]                          # [msg_start]\n    msg_block = obs[cfg.msg_start:cfg.msg_end]               # [n_peers * msg_dim]\n    peer_msgs = msg_block.reshape(cfg.n_peers, cfg.msg_dim)  # [n_peers, msg_dim]\n    return own_obs, peer_msgs\n\n\ndef onehot(action: int, dim: int) -> np.ndarray:\n    v = np.zeros(dim, dtype=np.float32)\n    v[action] = 1.0\n    return v\n\n\ndef stack_agent_data(d: Dict[str, np.ndarray],\n                     agent_list: List[str]) -> np.ndarray:\n    """Stack per-agent arrays into [n_agents, *] numpy array."""\n    return np.stack([d[a] for a in agent_list], axis=0)\n\n\n# ── Quick sanity check ────────────────────────────────────────────────────\ndef test_maddpg_agent():\n    cfg_t    = Config()\n    peer_ids = ["agent_1", "agent_2", "agent_3"]\n    agent    = MADDPGAgent("agent_0", peer_ids, cfg_t)\n\n    # Check networks are on the right device\n    assert next(agent.actor.parameters()).device.type == DEVICE.type\n    assert next(agent.critic.parameters()).device.type == DEVICE.type\n\n    # Check weight shapes\n    w = agent.get_rep_weights()\n    assert len(w) == cfg_t.n_peers, "Wrong number of weights"\n    assert abs(w.sum() - 1.0) < 1e-5, "Weights do not sum to 1"\n\n    # Check extract_obs_parts\n    dummy_obs = np.random.randn(cfg_t.obs_dim).astype(np.float32)\n    own_obs, peer_msgs = extract_obs_parts(dummy_obs, cfg_t)\n    assert own_obs.shape   == (cfg_t.msg_start,),                "own_obs shape wrong"\n    assert peer_msgs.shape == (cfg_t.n_peers, cfg_t.msg_dim),   "peer_msgs shape wrong"\n\n    n_actor  = sum(p.numel() for p in agent.actor.parameters())\n    n_critic = sum(p.numel() for p in agent.critic.parameters())\n    print("MADDPGAgent sanity check passed.")\n    print("  own_obs_dim : " + str(cfg_t.msg_start))\n    print("  actor params: " + str(n_actor))\n    print("  critic params: " + str(n_critic))\n    print("  rep weights : " + str(w) + "  (sum=" + str(round(w.sum(),4)) + ")")\n\ntest_maddpg_agent()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 9 — Training & Evaluation Loop (train.py)         ║\n# ╚══════════════════════════════════════════════════════════╝\n\ndef update_agents(agents, buffer, cfg):\n    """One MADDPG gradient update step for all agents."""\n    if len(buffer) < cfg.batch_size:\n        return 0.0, 0.0\n\n    obs_b, acts_b, rews_b, nobs_b, dones_b = buffer.sample(cfg.batch_size)\n    total_closs, total_aloss = 0.0, 0.0\n\n    for i, agent in enumerate(agents):\n        # ── Critic update ─────────────────────────────────────────────────────\n        with torch.no_grad():\n            t_acts_list = []\n            for j, ag in enumerate(agents):\n                o_j = nobs_b[:, j, :cfg.msg_start]\n                m_j = nobs_b[:, j, cfg.msg_start:cfg.msg_end]\n                m_j = m_j.view(-1, cfg.n_peers, cfg.msg_dim)\n                # Target actor always uses uniform weights (stable training target)\n                w_j = torch.ones(cfg.batch_size, cfg.n_peers, device=DEVICE) / cfg.n_peers\n                t_logits = ag.t_actor(o_j, m_j, w_j)\n                t_act    = F.softmax(t_logits, dim=-1)\n                t_acts_list.append(t_act)\n            t_acts = torch.stack(t_acts_list, dim=1)\n            next_q = agent.t_critic(nobs_b, t_acts)\n            r_i    = rews_b[:, i:i+1]\n            d_i    = dones_b[:, i:i+1]\n            target = r_i + cfg.gamma * (1 - d_i) * next_q\n\n        current_q = agent.critic(obs_b, acts_b)\n        closs     = F.mse_loss(current_q, target)\n        agent.critic_opt.zero_grad()\n        closs.backward()\n        torch.nn.utils.clip_grad_norm_(agent.critic.parameters(), 1.0)\n        agent.critic_opt.step()\n\n        # ── Actor update — FIX: use CURRENT reputation weights ────────────────\n        # The actor must be trained WITH reputation weights so it learns to\n        # exploit the trust signal. Using uniform weights here (the original bug)\n        # meant the network was never exposed to reputation-modulated inputs\n        # during training, so it could not benefit from them at inference time.\n        o_i = obs_b[:, i, :cfg.msg_start]\n        m_i = obs_b[:, i, cfg.msg_start:cfg.msg_end].view(-1, cfg.n_peers, cfg.msg_dim)\n\n        # Get current rep weights for this agent (shape: [n_peers])\n        rep_w = torch.FloatTensor(agent.get_rep_weights()).to(DEVICE)\n        # Expand to batch: [1, n_peers] -> [batch, n_peers]\n        rep_w_batch = rep_w.unsqueeze(0).expand(cfg.batch_size, -1)\n\n        logits  = agent.actor(o_i, m_i, rep_w_batch)\n        new_act = F.softmax(logits, dim=-1)\n        acts_q  = acts_b.clone()\n        acts_q[:, i, :] = new_act\n        aloss = -agent.critic(obs_b, acts_q).mean()\n        agent.actor_opt.zero_grad()\n        aloss.backward()\n        torch.nn.utils.clip_grad_norm_(agent.actor.parameters(), 1.0)\n        agent.actor_opt.step()\n\n        agent.soft_update(cfg.tau)\n        total_closs += closs.item()\n        total_aloss += aloss.item()\n\n    return total_closs / cfg.n_agents, total_aloss / cfg.n_agents\n\n\ndef run_eval_episode(agents, cfg, seed=0):\n    """One evaluation episode — deterministic actions, no buffer push.\n    FIX: tracker is updated during eval so rep scores reflect eval behaviour."""\n    env      = DefectorWrapper(cfg, seed=seed)\n    obs_dict = env.reset(seed=seed)\n    ep_reward = 0.0\n\n    for step in range(cfg.max_steps):\n        actions = {}\n        for ag in agents:\n            aid     = ag.agent_id\n            obs_raw = obs_dict.get(aid, np.zeros(cfg.obs_dim, dtype=np.float32))\n            own_obs, peer_msgs = extract_obs_parts(obs_raw, cfg)\n            w      = ag.get_rep_weights()\n            act_oh = np.eye(cfg.act_dim)[\n                ag.actor.select_action(own_obs, peer_msgs, w, deterministic=True)\n            ]\n            actions[aid] = act_oh\n\n        obs_dict, rews, dones = env.step_all(actions)\n        ep_reward += sum(rews.values())\n\n        # FIX: update tracker during eval so scores reflect actual eval dynamics\n        for ag in agents:\n            ag.tracker.update(rews.get(ag.agent_id, 0.0))\n\n    rep_snap = {ag.agent_id: dict(ag.tracker.scores) for ag in agents}\n    env.close()\n    return ep_reward, rep_snap\n\n\ndef train(cfg, verbose=True):\n    """Full training run. Returns results dict with curves and metrics."""\n    set_seed(cfg.seed)\n    agent_ids = ["agent_" + str(i) for i in range(cfg.n_agents)]\n\n    agents = []\n    for aid in agent_ids:\n        peer_ids = [a for a in agent_ids if a != aid]\n        agents.append(MADDPGAgent(aid, peer_ids, cfg))\n\n    buffer  = ReplayBuffer(cfg)\n    results = {\n        "train_rewards":  [],\n        "eval_rewards":   [],\n        "eval_episodes":  [],\n        "rep_scores_log": [],\n        "critic_losses":  [],\n        "actor_losses":   [],\n    }\n\n    total_steps = 0\n    pbar = trange(cfg.n_episodes,\n                  desc="[" + cfg.trust_condition.upper() + "|" + str(cfg.n_defectors) + "def]",\n                  disable=not verbose)\n\n    for episode in pbar:\n        env      = DefectorWrapper(cfg, seed=cfg.seed + episode)\n        obs_dict = env.reset(seed=cfg.seed + episode)\n        ep_reward = 0.0\n\n        for step in range(cfg.max_steps):\n            actions = {}\n\n            for ag in agents:\n                aid     = ag.agent_id\n                obs_raw = obs_dict.get(aid, np.zeros(cfg.obs_dim, dtype=np.float32))\n                own_obs, peer_msgs = extract_obs_parts(obs_raw, cfg)\n                w = ag.get_rep_weights()\n\n                if total_steps < cfg.warmup_steps:\n                    act_int = env.sample_action(aid)\n                else:\n                    act_int = ag.actor.select_action(own_obs, peer_msgs, w)\n\n                actions[aid] = onehot(act_int, cfg.act_dim)\n\n            next_obs_dict, rews, dones = env.step_all(actions)\n            ep_reward += sum(rews.values())\n\n            # Reputation update BEFORE buffer push so scores reflect current step\n            for ag in agents:\n                ag.tracker.update(rews.get(ag.agent_id, 0.0))\n\n            rep_scores_arr = np.array(\n                [list(ag.tracker.scores.values()) for ag in agents],\n                dtype=np.float32\n            )\n            buffer.push(\n                obs        = stack_agent_data(obs_dict,      agent_ids),\n                acts       = stack_agent_data(actions,       agent_ids),\n                rews       = np.array([rews.get(a, 0.) for a in agent_ids], dtype=np.float32),\n                next_obs   = stack_agent_data(next_obs_dict, agent_ids),\n                dones      = np.array([float(dones.get(a, False)) for a in agent_ids], dtype=np.float32),\n                rep_scores = rep_scores_arr,\n            )\n\n            obs_dict    = next_obs_dict\n            total_steps += 1\n\n            if total_steps >= cfg.warmup_steps and total_steps % 2 == 0:\n                cl, al = update_agents(agents, buffer, cfg)\n                results["critic_losses"].append(cl)\n                results["actor_losses"].append(al)\n\n        results["train_rewards"].append(ep_reward)\n        env.close()\n\n        if (episode + 1) % cfg.eval_interval == 0:\n            eval_rews = []\n            rep_snap_last = {}\n            for e in range(cfg.eval_episodes):\n                er, rep_snap = run_eval_episode(agents, cfg, seed=9000 + e)\n                eval_rews.append(er)\n                if e == cfg.eval_episodes - 1:\n                    rep_snap_last = rep_snap\n\n            mean_eval = float(np.mean(eval_rews))\n            results["eval_rewards"].append(mean_eval)\n            results["eval_episodes"].append(episode + 1)\n            results["rep_scores_log"].append({\n                "episode":          episode + 1,\n                "mean_eval_reward": mean_eval,\n                "rep_scores":       rep_snap_last,\n            })\n\n            if verbose:\n                pbar.set_postfix({"eval": str(round(mean_eval, 1)), "buf": len(buffer)})\n\n    return results\n\nprint("Training loop defined — ready to run experiments.")\nprint("FIXES APPLIED:")\nprint("  1. Actor update now uses current reputation weights (not uniform)")\nprint("  2. Reputation baseline initialised from first real reward (not 0.0)")\nprint("  3. Tracker updated during eval episodes for accurate score logging")\n',
    ]
    for _src in _cells:
        exec(_src, globals())
    print("Setup complete.")
# ─────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────

# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 10 — Smoke Test (run this before full matrix)     ║
# ╚══════════════════════════════════════════════════════════╝
# Runs Condition C (soft reputation) with 1 defector for 100 episodes.
# GPU: ~3–5 min  |  CPU: ~10–15 min
# Verify the learning curves look reasonable before starting 27-run matrix.

print("🚀 Smoke test: Condition C | 1 defector | 100 episodes")
print("="*60)

smoke_cfg = Config(
    trust_condition = "soft",
    n_defectors     = 1,
    n_episodes      = 100,
    eval_interval   = 20,
    eval_episodes   = 5,
    seed            = 42,
)

t0           = time.time()
smoke_result = train(smoke_cfg, verbose=True)
elapsed      = time.time() - t0

print("\nSmoke test complete in " + str(round(elapsed, 1)) + "s")
if smoke_result["eval_rewards"]:
    print(f"   Final eval reward : {smoke_result['eval_rewards'][-1]:.2f}")
print(f"   Eval checkpoints  : {len(smoke_result['eval_rewards'])}")

# Quick diagnostic plot
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

ax = axes[0]
rews = smoke_result["train_rewards"]
ax.plot(rews, alpha=0.3, color="#2E75B6", linewidth=0.7)
if len(rews) >= 10:
    smoothed = pd.Series(rews).rolling(10, min_periods=1).mean()
    ax.plot(smoothed, color="#2E75B6", linewidth=2, label="Smoothed (window=10)")
ax.set_xlabel("Episode"); ax.set_ylabel("Team Reward")
ax.set_title("Training Rewards"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
ax.spines[["top","right"]].set_visible(False)

ax = axes[1]
if smoke_result["eval_rewards"]:
    ax.plot(smoke_result["eval_episodes"], smoke_result["eval_rewards"],
            "o-", color="#C55A11", linewidth=2, markersize=7)
ax.set_xlabel("Episode"); ax.set_ylabel("Eval Team Reward")
ax.set_title("Evaluation Rewards"); ax.grid(alpha=0.3)
ax.spines[["top","right"]].set_visible(False)

plt.suptitle("Smoke Test — Condition C (Soft RWC) | 1 Defector", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{smoke_cfg.save_dir}/smoke_test.png", dpi=120, bbox_inches="tight")
plt.show()
print("\n🟢 Smoke test passed — proceed to Cell 11 for the full experiment matrix")


In [ ]:
# ── Dependency guard ─────────────────────────────────────────────────────
import sys as _sys
if "Config" not in dir():
    print("Re-running setup cells...")
    _cells = [
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 2 — Imports & Environment Check                   ║\n# ╚══════════════════════════════════════════════════════════╝\n\nimport os, sys, time, random, copy\nfrom dataclasses import dataclass\nfrom typing import Dict, List, Tuple, Optional\n\nimport copy\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.optim as optim\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport pandas as pd\nfrom scipy import stats\nfrom tqdm.auto import tqdm, trange\n\ndef set_seed(seed: int):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\nset_seed(42)\n\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nprint("Device: " + str(DEVICE))\nif DEVICE.type == "cuda":\n    print("  GPU : " + torch.cuda.get_device_name(0))\n\n# ── mpe2 observation layout for simple_spread_v3, N=4 ────────────────────\n# [self_vel(2), self_pos(2), landmark_rel_pos(4*2=8),\n#  other_agent_rel_pos(3*2=6), communication(3*2=6)]\n# Total = 2+2+8+6+6 = 24\nfrom mpe2 import simple_spread_v3\n\nenv = simple_spread_v3.env(N=4, local_ratio=0.5, render_mode=None)\nenv.reset(seed=0)\nfor agent in env.agent_iter():\n    obs, rew, term, trunc, info = env.last()\n    env.step(env.action_space(agent).sample())\n    break\nenv.close()\n\nprint("mpe2 simple_spread_v3 (N=4):")\nprint("  obs shape : " + str(obs.shape))\nprint("  agents    : " + str(env.agents))\n\n# Confirm layout matches our constants\nN = 4\nexpected_obs_dim = 2 + 2 + N*2 + (N-1)*2 + (N-1)*2   # = 24\nassert obs.shape == (expected_obs_dim,), (\n    "Obs shape mismatch: got " + str(obs.shape) +\n    ", expected (" + str(expected_obs_dim) + ",)"\n)\nprint("  obs_dim   : " + str(expected_obs_dim) + "  [2 vel + 2 pos + 8 landmarks + 6 others + 6 comms]")\nprint("  msg_start : 18  (slice [18:24] = 3 peers x 2D comm vectors)")\nprint("  msg_end   : 24")\nprint("  Shape check passed.")\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 3 — Hyperparameter Config                         ║\n# ╚══════════════════════════════════════════════════════════╝\n\n@dataclass\nclass Config:\n    # ── Environment ──────────────────────────────────────────\n    n_agents:        int   = 4\n    n_landmarks:     int   = 4\n    local_ratio:     float = 0.5\n    max_steps:       int   = 50\n    #\n    # mpe2 obs layout for N=4:\n    # [self_vel(2), self_pos(2), landmark_rel_pos(8),\n    #  other_agent_rel_pos(6), communication(6)]\n    # Total obs_dim = 24\n    # msg_start = 18  (start of communication slice)\n    # msg_end   = 24  (end of communication slice)\n    obs_dim:         int   = 24   # 2+2+8+6+6 for N=4\n    act_dim:         int   = 5    # Discrete(5)\n    msg_dim:         int   = 2    # each peer sends a 2D comm vector\n    msg_start:       int   = 18   # index where comm slice begins\n    msg_end:         int   = 24   # index where comm slice ends\n\n    # ── Defectors ────────────────────────────────────────────\n    n_defectors:     int   = 1\n    noise_scale:     float = 1.0\n\n    # ── Trust condition ──────────────────────────────────────\n    # "none"   = Condition A: equal weights\n    # "binary" = Condition B: hard threshold gate\n    # "soft"   = Condition C: EMA reputation weighting (RWC)\n    trust_condition:    str   = "soft"\n    rep_alpha:          float = 0.05\n    rep_baseline_alpha: float = 0.01\n    rep_temperature:    float = 5.0\n    binary_threshold:   float = 0.5\n\n    # ── MADDPG ───────────────────────────────────────────────\n    hidden_dim:      int   = 128\n    actor_lr:        float = 1e-3\n    critic_lr:       float = 1e-3\n    gamma:           float = 0.95\n    tau:             float = 0.01\n    buffer_size:     int   = 100_000\n    batch_size:      int   = 256\n    warmup_steps:    int   = 1000\n\n    # ── Training ─────────────────────────────────────────────\n    n_episodes:      int   = 500\n    eval_interval:   int   = 25\n    eval_episodes:   int   = 20\n    seed:            int   = 42\n\n    # ── Output ───────────────────────────────────────────────\n    save_dir:        str   = "/content/marl_trust"\n\n    def __post_init__(self):\n        import os\n        os.makedirs(self.save_dir, exist_ok=True)\n        self.n_peers = self.n_agents - 1\n        all_ids = ["agent_" + str(i) for i in range(self.n_agents)]\n        self.defector_ids = set(all_ids[-self.n_defectors:]) if self.n_defectors > 0 else set()\n        self.honest_ids   = set(all_ids) - self.defector_ids\n\ncfg = Config()\nprint("Config created.")\nprint("  obs_dim   = " + str(cfg.obs_dim))\nprint("  msg_start = " + str(cfg.msg_start) + \n      "  (slice [" + str(cfg.msg_start) + ":" + str(cfg.msg_end) + \n      "] = " + str(cfg.n_peers) + " peers x " + str(cfg.msg_dim) + "D comms)")\nprint("  defectors = " + str(cfg.defector_ids))\n\n# Sanity check: msg slice must exactly fit n_peers * msg_dim\nassert cfg.msg_end - cfg.msg_start == cfg.n_peers * cfg.msg_dim, (\n    "Msg slice length " + str(cfg.msg_end - cfg.msg_start) +\n    " != n_peers*msg_dim " + str(cfg.n_peers * cfg.msg_dim)\n)\nprint("  Slice sanity check passed.")\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 4 — DefectorWrapper (env_wrapper.py)              ║\n# ╚══════════════════════════════════════════════════════════╝\n\n# MPE environments are now in the mpe2 package (moved from mpe2)\nfrom mpe2 import simple_spread_v3\n\nclass DefectorWrapper:\n    """\n    Wraps simple_spread_v3 and corrupts outgoing communication messages\n    from designated defector agents by injecting Gaussian noise.\n\n    Compatible with PettingZoo 1.22.x and 1.24.x.\n\n    Design:\n    - Only obs[msg_start:msg_end] is corrupted for honest agents.\n    - Defectors\' own observations remain clean (they can still sense\n      positions), making the problem harder for honest agents.\n    - The wrapper never modifies the underlying environment state.\n    """\n\n    def __init__(self, cfg, seed: int = None):\n        self.cfg  = cfg\n        self.rng  = np.random.default_rng(seed)\n        self._env = simple_spread_v3.env(\n            N           = cfg.n_agents,\n            local_ratio = cfg.local_ratio,\n            render_mode = None,\n            max_cycles  = cfg.max_steps,\n        )\n        self.agents    = self._env.possible_agents   # stable ordered list\n        self.n_agents  = len(self.agents)\n        self.agent_idx = {a: i for i, a in enumerate(self.agents)}\n\n    def reset(self, seed=None):\n        self._env.reset(seed=seed)\n        return self._collect_obs()\n\n    def step_all(self, actions: Dict[str, np.ndarray]):\n        """\n        Execute one full round: each agent acts once.\n        actions: {agent_id: one-hot np.ndarray}\n        Returns: obs, rewards, dones (all dicts)\n        """\n        rewards, dones = {a: 0.0 for a in self.agents}, {a: False for a in self.agents}\n\n        for agent in self._env.agent_iter():\n            obs_raw, rew, term, trunc, info = self._env.last()\n            done = term or trunc\n            rewards[agent] = float(rew)\n            dones[agent]   = done\n            if done:\n                self._env.step(None)\n            else:\n                act_oh = actions.get(agent)\n                if act_oh is not None:\n                    act_int = int(np.argmax(act_oh))\n                else:\n                    act_int = self._env.action_space(agent).sample()\n                self._env.step(act_int)\n\n        obs = self._collect_obs()\n        return obs, rewards, dones\n\n    def _collect_obs(self) -> Dict[str, np.ndarray]:\n        obs = {}\n        for agent in self.agents:\n            try:\n                o = self._env.observe(agent)\n                if o is None:\n                    o = np.zeros(self.cfg.obs_dim, dtype=np.float32)\n                else:\n                    o = np.array(o, dtype=np.float32).copy()\n            except Exception:\n                o = np.zeros(self.cfg.obs_dim, dtype=np.float32)\n            obs[agent] = self._maybe_corrupt(agent, o)\n        return obs\n\n    def _maybe_corrupt(self, observer: str, obs: np.ndarray) -> np.ndarray:\n        """Add Gaussian noise to peer-message slice for honest observers."""\n        if not self.cfg.defector_ids or observer in self.cfg.defector_ids:\n            return obs\n        noise = (self.rng.standard_normal(self.cfg.msg_end - self.cfg.msg_start)\n                 .astype(np.float32) * self.cfg.noise_scale)\n        obs[self.cfg.msg_start:self.cfg.msg_end] += noise\n        return obs\n\n    def sample_action(self, agent_id: str) -> int:\n        return self._env.action_space(agent_id).sample()\n\n    def close(self):\n        self._env.close()\n\n\n# ── Unit test ─────────────────────────────────────────────────────────────────\ndef test_defector_wrapper():\n    from dataclasses import replace\n\n    # With 1 defector\n    cfg_dirty = Config(n_defectors=1)\n    env_dirty = DefectorWrapper(cfg_dirty, seed=0)\n    env_dirty.reset(seed=0)\n    dummy = np.zeros(cfg_dirty.obs_dim, dtype=np.float32)\n    corrupted = env_dirty._maybe_corrupt("agent_0", dummy.copy())\n    std_dirty = corrupted[cfg_dirty.msg_start:cfg_dirty.msg_end].std()\n\n    # With 0 defectors\n    cfg_clean = Config(n_defectors=0)\n    env_clean = DefectorWrapper(cfg_clean, seed=0)\n    env_clean.reset(seed=0)\n    clean = env_clean._maybe_corrupt("agent_0", dummy.copy())\n    std_clean = clean[cfg_clean.msg_start:cfg_clean.msg_end].std()\n\n    print(f"✅ DefectorWrapper unit test")\n    print(f"   Msg slice std with  defector : {std_dirty:.4f}  (should be > 0.1)")\n    print(f"   Msg slice std without defector: {std_clean:.4f}  (should be 0.0)")\n    assert std_dirty > 0.1, "Corruption not applied!"\n    assert std_clean == 0.0, "Clean env is corrupted — bug!"\n\n    # Smoke: run a few steps\n    env2 = DefectorWrapper(Config(n_defectors=1), seed=7)\n    obs  = env2.reset(seed=7)\n    assert len(obs) == 4, f"Expected 4 agent obs, got {len(obs)}"\n    dummy_acts = {a: np.eye(5)[env2.sample_action(a)] for a in env2.agents}\n    obs2, rews, dones = env2.step_all(dummy_acts)\n    assert len(rews) == 4\n    env2.close()\n    env_dirty.close()\n    env_clean.close()\n    print("   Step test passed ✓")\n    print("   All assertions passed ✓")\n\ntest_defector_wrapper()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 5 — ReputationTracker (reputation.py)             ║\n# ╚══════════════════════════════════════════════════════════╝\n\nfrom scipy.special import softmax as scipy_softmax\n\nclass ReputationTracker:\n    """\n    Maintains a continuous reputation score r[j] in [0,1] for each peer j.\n\n    FIX: baseline is now initialised to None and set from the first real\n    reward, preventing the \'always negative delta\' problem that caused all\n    scores to drift to 0 then snap to 1.0 when rewards improved.\n\n    Update rule (per step):\n        delta  = reward - baseline\n        signal = 1.0 if delta > 0 else 0.0\n        r[j]  <- (1 - alpha) * r[j] + alpha * signal  for all j\n        baseline <- (1 - b_alpha) * baseline + b_alpha * reward\n    """\n\n    def __init__(self, peer_ids, cfg):\n        self.peer_ids    = list(peer_ids)\n        self.alpha       = cfg.rep_alpha\n        self.b_alpha     = cfg.rep_baseline_alpha\n        self.temperature = cfg.rep_temperature\n        self.scores      = {j: 0.5 for j in peer_ids}\n        self.baseline    = None    # FIX: initialise lazily from first reward\n\n    def update(self, reward):\n        """Called once per step with this agent\'s observed reward."""\n        # FIX: initialise baseline to first real reward so delta starts near 0\n        if self.baseline is None:\n            self.baseline = reward\n            return  # skip update on very first step\n\n        delta  = reward - self.baseline\n        signal = 1.0 if delta > 0 else 0.0\n\n        self.baseline = (1 - self.b_alpha) * self.baseline + self.b_alpha * reward\n\n        for j in self.peer_ids:\n            self.scores[j] = (\n                (1 - self.alpha) * self.scores[j] + self.alpha * signal\n            )\n\n    def get_weights(self):\n        """Softmax-normalised weights, ordered by peer_ids."""\n        scores_arr = np.array([self.scores[j] for j in self.peer_ids],\n                               dtype=np.float32)\n        return scipy_softmax(scores_arr * self.temperature).astype(np.float32)\n\n    def get_weights_binary(self, threshold=0.5):\n        """Condition B: hard binary gate, re-normalised."""\n        scores_arr = np.array([self.scores[j] for j in self.peer_ids],\n                               dtype=np.float32)\n        weights = (scores_arr >= threshold).astype(np.float32)\n        total = weights.sum()\n        if total == 0:\n            weights = np.ones(len(self.peer_ids), dtype=np.float32)\n            total   = float(len(self.peer_ids))\n        return weights / total\n\n    def get_weights_uniform(self):\n        """Condition A: no trust — uniform weights."""\n        n = len(self.peer_ids)\n        return np.ones(n, dtype=np.float32) / n\n\n    def get_scores_dict(self):\n        return dict(self.scores)\n\n    def reset(self):\n        self.scores   = {j: 0.5 for j in self.peer_ids}\n        self.baseline = None   # FIX: reset to None, not 0.0\n\n\n# ── Unit tests ────────────────────────────────────────────────────────────────\ndef test_reputation_tracker():\n    cfg_t = Config()\n    peers = ["agent_1", "agent_2", "agent_3"]\n    tracker = ReputationTracker(peers, cfg_t)\n\n    # Test 1: neutral init\n    w = tracker.get_weights()\n    assert abs(w.sum() - 1.0) < 1e-5, "Weights do not sum to 1"\n    assert abs(w[0] - w[1]) < 1e-5, "Initial weights not uniform"\n    print("  Test 1 passed: neutral init OK")\n\n    # Test 2: positive drift — rewards consistently above baseline\n    tracker.reset()\n    for i in range(200):\n        tracker.update(reward=-5.0 + i * 0.05)   # rising rewards\n    scores = list(tracker.scores.values())\n    print("  Test 2: scores after rising rewards: " + str([round(s,3) for s in scores]))\n    assert all(s > 0.5 for s in scores), "Scores did not drift up: " + str(scores)\n    print("  Test 2 passed: positive drift OK (mean=" + str(round(float(np.mean(scores)),3)) + ")")\n\n    # Test 3: negative drift — rewards consistently below baseline\n    tracker.reset()\n    for i in range(200):\n        tracker.update(reward=-5.0 - i * 0.05)   # falling rewards\n    scores = list(tracker.scores.values())\n    print("  Test 3: scores after falling rewards: " + str([round(s,3) for s in scores]))\n    assert all(s < 0.5 for s in scores), "Scores did not drift down: " + str(scores)\n    print("  Test 3 passed: negative drift OK (mean=" + str(round(float(np.mean(scores)),3)) + ")")\n\n    # Test 4: weights always sum to 1\n    for _ in range(50):\n        tracker.update(reward=np.random.randn() - 5.0)\n    for method in ["get_weights", "get_weights_binary", "get_weights_uniform"]:\n        w = getattr(tracker, method)()\n        assert abs(w.sum() - 1.0) < 1e-5, method + " does not sum to 1"\n    print("  Test 4 passed: all weight methods sum to 1.0 OK")\n\n    # Test 5: FIX verification — baseline initialised from first reward\n    tracker2 = ReputationTracker(peers, cfg_t)\n    tracker2.update(reward=-8.0)   # first step, sets baseline to -8, no score update\n    scores_after_first = list(tracker2.scores.values())\n    assert all(s == 0.5 for s in scores_after_first), "Scores changed on first step — baseline bug!"\n    tracker2.update(reward=-7.5)   # second step: delta = -7.5 - (-8.0) = +0.5, signal=1\n    scores_after_second = list(tracker2.scores.values())\n    assert all(s > 0.5 for s in scores_after_second), "Positive delta did not increase scores"\n    print("  Test 5 passed: baseline initialisation fix verified OK")\n\n    print("")\n    print("ReputationTracker — all 5 unit tests passed.")\n\ntest_reputation_tracker()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 6 — Actor & Critic Networks (networks.py)         ║\n# ╚══════════════════════════════════════════════════════════╝\n\nclass Actor(nn.Module):\n    """\n    MADDPG actor with reputation-weighted message aggregation.\n\n    Forward pass:\n        1. Receive own_obs [batch, msg_start] and\n           peer_msgs [batch, n_peers, msg_dim]\n        2. Aggregate peer_msgs weighted by rep_weights [batch, n_peers]\n        3. Concatenate own_obs + aggregated_msg → policy MLP → action logits\n\n    The reputation weighting is the ONLY modification from standard MADDPG.\n    When rep_weights is uniform, output is identical to an unmodified actor.\n    """\n\n    def __init__(self, own_obs_dim: int, n_peers: int,\n                 msg_dim: int, act_dim: int, hidden_dim: int):\n        super().__init__()\n        self.n_peers  = n_peers\n        self.msg_dim  = msg_dim\n        self.act_dim  = act_dim\n\n        input_dim = own_obs_dim + msg_dim   # own obs + aggregated msg\n        self.net = nn.Sequential(\n            nn.Linear(input_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, act_dim),\n        )\n\n    def forward(self,\n                own_obs:     torch.Tensor,   # [batch, own_obs_dim]\n                peer_msgs:   torch.Tensor,   # [batch, n_peers, msg_dim]\n                rep_weights: torch.Tensor,   # [batch, n_peers]\n                ) -> torch.Tensor:           # [batch, act_dim]\n\n        # ── Reputation-weighted message aggregation ───────────────────────────\n        w   = rep_weights.unsqueeze(-1)              # [batch, n_peers, 1]\n        agg = (peer_msgs * w).sum(dim=1)             # [batch, msg_dim]\n        # ─────────────────────────────────────────────────────────────────────\n\n        x = torch.cat([own_obs, agg], dim=-1)        # [batch, own_obs_dim + msg_dim]\n        return self.net(x)                           # [batch, act_dim]\n\n    def select_action(self, own_obs: np.ndarray,\n                      peer_msgs: np.ndarray,\n                      rep_weights: np.ndarray,\n                      deterministic: bool = False) -> np.ndarray:\n        with torch.no_grad():\n            o = torch.FloatTensor(own_obs).unsqueeze(0).to(DEVICE)\n            m = torch.FloatTensor(peer_msgs).unsqueeze(0).to(DEVICE)\n            w = torch.FloatTensor(rep_weights).unsqueeze(0).to(DEVICE)\n            logits = self.forward(o, m, w).squeeze(0)\n            if deterministic:\n                action = logits.argmax().item()\n            else:\n                probs  = F.softmax(logits, dim=-1)\n                action = torch.multinomial(probs, 1).item()\n        return action\n\n\nclass Critic(nn.Module):\n    """\n    Centralised MADDPG critic: Q(all_obs, all_actions).\n    Has full visibility during training (CTDE paradigm).\n    """\n\n    def __init__(self, n_agents: int, obs_dim: int,\n                 act_dim: int, hidden_dim: int):\n        super().__init__()\n        input_dim = n_agents * (obs_dim + act_dim)\n        self.net = nn.Sequential(\n            nn.Linear(input_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, 1),\n        )\n\n    def forward(self, all_obs: torch.Tensor,\n                all_acts: torch.Tensor) -> torch.Tensor:\n        """\n        all_obs:  [batch, n_agents, obs_dim]\n        all_acts: [batch, n_agents, act_dim]\n        Returns:  [batch, 1]\n        """\n        batch = all_obs.shape[0]\n        x = torch.cat([\n            all_obs.reshape(batch, -1),\n            all_acts.reshape(batch, -1)\n        ], dim=-1)\n        return self.net(x)\n\n\n# ── Verify parity: uniform weights == standard MADDPG ─────────────────────────\ndef test_network_parity():\n    cfg_t = Config()\n    own_obs_dim = cfg_t.msg_start        # 28 dims (everything before msg slice)\n    actor = Actor(own_obs_dim, cfg_t.n_peers, cfg_t.msg_dim,\n                  cfg_t.act_dim, cfg_t.hidden_dim)\n\n    # Uniform weights → same as concatenating mean messages\n    batch = 4\n    obs   = torch.randn(batch, own_obs_dim)\n    msgs  = torch.randn(batch, cfg_t.n_peers, cfg_t.msg_dim)\n    w_uni = torch.ones(batch, cfg_t.n_peers) / cfg_t.n_peers\n\n    out_uni  = actor(obs, msgs, w_uni)\n\n    # Manually compute what unmodified MADDPG would do: simple mean\n    agg_mean = msgs.mean(dim=1)\n    x_manual = torch.cat([obs, agg_mean], dim=-1)\n    out_man  = actor.net(x_manual)\n\n    diff = (out_uni - out_man).abs().max().item()\n    assert diff < 1e-5, f"Parity test failed, max diff: {diff}"\n    print(f"✅ Network parity test: uniform weights == mean aggregation (max diff = {diff:.2e}) ✓")\n\n    # Shape checks\n    critic = Critic(cfg_t.n_agents, cfg_t.obs_dim, cfg_t.act_dim, cfg_t.hidden_dim)\n    all_obs  = torch.randn(batch, cfg_t.n_agents, cfg_t.obs_dim)\n    all_acts = torch.randn(batch, cfg_t.n_agents, cfg_t.act_dim)\n    q = critic(all_obs, all_acts)\n    assert q.shape == (batch, 1), f"Critic output shape wrong: {q.shape}"\n    print(f"   Critic output shape: {q.shape} ✓")\n    print(f"   Actor  output shape: {out_uni.shape} ✓")\n\ntest_network_parity()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 7 — Replay Buffer (buffer.py)                     ║\n# ╚══════════════════════════════════════════════════════════╝\n\nclass ReplayBuffer:\n    """\n    Circular replay buffer for MADDPG.\n    Stores transitions for all agents simultaneously.\n    Also stores reputation scores for logging/analysis.\n    """\n\n    def __init__(self, cfg: Config):\n        self.capacity   = cfg.buffer_size\n        self.n_agents   = cfg.n_agents\n        self.obs_dim    = cfg.obs_dim\n        self.act_dim    = cfg.act_dim\n        self.n_peers    = cfg.n_peers\n        self.ptr        = 0\n        self.size       = 0\n\n        # Pre-allocate numpy arrays for speed\n        self.obs       = np.zeros((self.capacity, self.n_agents, self.obs_dim),  dtype=np.float32)\n        self.acts      = np.zeros((self.capacity, self.n_agents, self.act_dim),  dtype=np.float32)\n        self.rews      = np.zeros((self.capacity, self.n_agents),                dtype=np.float32)\n        self.next_obs  = np.zeros((self.capacity, self.n_agents, self.obs_dim),  dtype=np.float32)\n        self.dones     = np.zeros((self.capacity, self.n_agents),                dtype=np.float32)\n        self.rep_scores= np.zeros((self.capacity, self.n_agents, self.n_peers),  dtype=np.float32)\n\n    def push(self,\n             obs:        np.ndarray,   # [n_agents, obs_dim]\n             acts:       np.ndarray,   # [n_agents, act_dim]\n             rews:       np.ndarray,   # [n_agents]\n             next_obs:   np.ndarray,   # [n_agents, obs_dim]\n             dones:      np.ndarray,   # [n_agents]\n             rep_scores: np.ndarray,   # [n_agents, n_peers]\n             ):\n        idx = self.ptr % self.capacity\n        self.obs[idx]        = obs\n        self.acts[idx]       = acts\n        self.rews[idx]       = rews\n        self.next_obs[idx]   = next_obs\n        self.dones[idx]      = dones\n        self.rep_scores[idx] = rep_scores\n        self.ptr  += 1\n        self.size  = min(self.size + 1, self.capacity)\n\n    def sample(self, batch_size: int):\n        idx = np.random.randint(0, self.size, size=batch_size)\n        to_t = lambda x: torch.FloatTensor(x).to(DEVICE)\n        return (\n            to_t(self.obs[idx]),        # [B, n_agents, obs_dim]\n            to_t(self.acts[idx]),       # [B, n_agents, act_dim]\n            to_t(self.rews[idx]),       # [B, n_agents]\n            to_t(self.next_obs[idx]),   # [B, n_agents, obs_dim]\n            to_t(self.dones[idx]),      # [B, n_agents]\n        )\n\n    def __len__(self):\n        return self.size\n\n\ndef test_replay_buffer():\n    cfg_t  = Config()\n    buf    = ReplayBuffer(cfg_t)\n    n, na, od, ad, np_ = 1000, cfg_t.n_agents, cfg_t.obs_dim, cfg_t.act_dim, cfg_t.n_peers\n\n    for _ in range(n):\n        buf.push(\n            np.random.randn(na, od).astype(np.float32),\n            np.random.randn(na, ad).astype(np.float32),\n            np.random.randn(na).astype(np.float32),\n            np.random.randn(na, od).astype(np.float32),\n            np.zeros(na, dtype=np.float32),\n            np.random.rand(na, np_).astype(np.float32),\n        )\n\n    assert len(buf) == n\n    obs, acts, rews, nobs, dones = buf.sample(256)\n    assert obs.shape   == (256, na, od), f"obs shape wrong: {obs.shape}"\n    assert acts.shape  == (256, na, ad)\n    assert rews.shape  == (256, na)\n    print(f"✅ ReplayBuffer: pushed {n} transitions, sampled batch of 256")\n    print(f"   obs shape: {obs.shape}, acts shape: {acts.shape} ✓")\n\ntest_replay_buffer()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 8 — MADDPGAgent & Helper Functions                ║\n# ╚══════════════════════════════════════════════════════════╝\n\nclass MADDPGAgent:\n    """\n    Single agent in the MADDPG framework.\n    Holds actor + target_actor, critic + target_critic,\n    their optimizers, and a ReputationTracker.\n    """\n\n    def __init__(self, agent_id: str, peer_ids: List[str], cfg):\n        self.agent_id = agent_id\n        self.peer_ids = peer_ids\n        self.cfg      = cfg\n        own_obs_dim   = cfg.msg_start   # dims before the comm slice\n\n        # ── Networks ──────────────────────────────────────────────────────\n        self.actor   = Actor(own_obs_dim, cfg.n_peers, cfg.msg_dim,\n                             cfg.act_dim, cfg.hidden_dim).to(DEVICE)\n        self.t_actor = copy.deepcopy(self.actor)\n\n        self.critic  = Critic(cfg.n_agents, cfg.obs_dim,\n                              cfg.act_dim, cfg.hidden_dim).to(DEVICE)\n        self.t_critic = copy.deepcopy(self.critic)\n\n        # ── Optimisers ────────────────────────────────────────────────────\n        self.actor_opt  = optim.Adam(self.actor.parameters(),  lr=cfg.actor_lr)\n        self.critic_opt = optim.Adam(self.critic.parameters(), lr=cfg.critic_lr)\n\n        # ── Reputation tracker ────────────────────────────────────────────\n        self.tracker = ReputationTracker(peer_ids, cfg)\n\n    def get_rep_weights(self) -> np.ndarray:\n        c = self.cfg.trust_condition\n        if   c == "none":   return self.tracker.get_weights_uniform()\n        elif c == "binary": return self.tracker.get_weights_binary(self.cfg.binary_threshold)\n        else:               return self.tracker.get_weights()   # soft\n\n    @torch.no_grad()\n    def soft_update(self, tau: float):\n        for p, tp in zip(self.actor.parameters(), self.t_actor.parameters()):\n            tp.data.copy_(tau * p.data + (1 - tau) * tp.data)\n        for p, tp in zip(self.critic.parameters(), self.t_critic.parameters()):\n            tp.data.copy_(tau * p.data + (1 - tau) * tp.data)\n\n\n# ── Helper functions ──────────────────────────────────────────────────────\n\ndef extract_obs_parts(obs: np.ndarray, cfg):\n    """Split raw obs into own_obs and peer_msgs."""\n    own_obs   = obs[:cfg.msg_start]                          # [msg_start]\n    msg_block = obs[cfg.msg_start:cfg.msg_end]               # [n_peers * msg_dim]\n    peer_msgs = msg_block.reshape(cfg.n_peers, cfg.msg_dim)  # [n_peers, msg_dim]\n    return own_obs, peer_msgs\n\n\ndef onehot(action: int, dim: int) -> np.ndarray:\n    v = np.zeros(dim, dtype=np.float32)\n    v[action] = 1.0\n    return v\n\n\ndef stack_agent_data(d: Dict[str, np.ndarray],\n                     agent_list: List[str]) -> np.ndarray:\n    """Stack per-agent arrays into [n_agents, *] numpy array."""\n    return np.stack([d[a] for a in agent_list], axis=0)\n\n\n# ── Quick sanity check ────────────────────────────────────────────────────\ndef test_maddpg_agent():\n    cfg_t    = Config()\n    peer_ids = ["agent_1", "agent_2", "agent_3"]\n    agent    = MADDPGAgent("agent_0", peer_ids, cfg_t)\n\n    # Check networks are on the right device\n    assert next(agent.actor.parameters()).device.type == DEVICE.type\n    assert next(agent.critic.parameters()).device.type == DEVICE.type\n\n    # Check weight shapes\n    w = agent.get_rep_weights()\n    assert len(w) == cfg_t.n_peers, "Wrong number of weights"\n    assert abs(w.sum() - 1.0) < 1e-5, "Weights do not sum to 1"\n\n    # Check extract_obs_parts\n    dummy_obs = np.random.randn(cfg_t.obs_dim).astype(np.float32)\n    own_obs, peer_msgs = extract_obs_parts(dummy_obs, cfg_t)\n    assert own_obs.shape   == (cfg_t.msg_start,),                "own_obs shape wrong"\n    assert peer_msgs.shape == (cfg_t.n_peers, cfg_t.msg_dim),   "peer_msgs shape wrong"\n\n    n_actor  = sum(p.numel() for p in agent.actor.parameters())\n    n_critic = sum(p.numel() for p in agent.critic.parameters())\n    print("MADDPGAgent sanity check passed.")\n    print("  own_obs_dim : " + str(cfg_t.msg_start))\n    print("  actor params: " + str(n_actor))\n    print("  critic params: " + str(n_critic))\n    print("  rep weights : " + str(w) + "  (sum=" + str(round(w.sum(),4)) + ")")\n\ntest_maddpg_agent()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 9 — Training & Evaluation Loop (train.py)         ║\n# ╚══════════════════════════════════════════════════════════╝\n\ndef update_agents(agents, buffer, cfg):\n    """One MADDPG gradient update step for all agents."""\n    if len(buffer) < cfg.batch_size:\n        return 0.0, 0.0\n\n    obs_b, acts_b, rews_b, nobs_b, dones_b = buffer.sample(cfg.batch_size)\n    total_closs, total_aloss = 0.0, 0.0\n\n    for i, agent in enumerate(agents):\n        # ── Critic update ─────────────────────────────────────────────────────\n        with torch.no_grad():\n            t_acts_list = []\n            for j, ag in enumerate(agents):\n                o_j = nobs_b[:, j, :cfg.msg_start]\n                m_j = nobs_b[:, j, cfg.msg_start:cfg.msg_end]\n                m_j = m_j.view(-1, cfg.n_peers, cfg.msg_dim)\n                # Target actor always uses uniform weights (stable training target)\n                w_j = torch.ones(cfg.batch_size, cfg.n_peers, device=DEVICE) / cfg.n_peers\n                t_logits = ag.t_actor(o_j, m_j, w_j)\n                t_act    = F.softmax(t_logits, dim=-1)\n                t_acts_list.append(t_act)\n            t_acts = torch.stack(t_acts_list, dim=1)\n            next_q = agent.t_critic(nobs_b, t_acts)\n            r_i    = rews_b[:, i:i+1]\n            d_i    = dones_b[:, i:i+1]\n            target = r_i + cfg.gamma * (1 - d_i) * next_q\n\n        current_q = agent.critic(obs_b, acts_b)\n        closs     = F.mse_loss(current_q, target)\n        agent.critic_opt.zero_grad()\n        closs.backward()\n        torch.nn.utils.clip_grad_norm_(agent.critic.parameters(), 1.0)\n        agent.critic_opt.step()\n\n        # ── Actor update — FIX: use CURRENT reputation weights ────────────────\n        # The actor must be trained WITH reputation weights so it learns to\n        # exploit the trust signal. Using uniform weights here (the original bug)\n        # meant the network was never exposed to reputation-modulated inputs\n        # during training, so it could not benefit from them at inference time.\n        o_i = obs_b[:, i, :cfg.msg_start]\n        m_i = obs_b[:, i, cfg.msg_start:cfg.msg_end].view(-1, cfg.n_peers, cfg.msg_dim)\n\n        # Get current rep weights for this agent (shape: [n_peers])\n        rep_w = torch.FloatTensor(agent.get_rep_weights()).to(DEVICE)\n        # Expand to batch: [1, n_peers] -> [batch, n_peers]\n        rep_w_batch = rep_w.unsqueeze(0).expand(cfg.batch_size, -1)\n\n        logits  = agent.actor(o_i, m_i, rep_w_batch)\n        new_act = F.softmax(logits, dim=-1)\n        acts_q  = acts_b.clone()\n        acts_q[:, i, :] = new_act\n        aloss = -agent.critic(obs_b, acts_q).mean()\n        agent.actor_opt.zero_grad()\n        aloss.backward()\n        torch.nn.utils.clip_grad_norm_(agent.actor.parameters(), 1.0)\n        agent.actor_opt.step()\n\n        agent.soft_update(cfg.tau)\n        total_closs += closs.item()\n        total_aloss += aloss.item()\n\n    return total_closs / cfg.n_agents, total_aloss / cfg.n_agents\n\n\ndef run_eval_episode(agents, cfg, seed=0):\n    """One evaluation episode — deterministic actions, no buffer push.\n    FIX: tracker is updated during eval so rep scores reflect eval behaviour."""\n    env      = DefectorWrapper(cfg, seed=seed)\n    obs_dict = env.reset(seed=seed)\n    ep_reward = 0.0\n\n    for step in range(cfg.max_steps):\n        actions = {}\n        for ag in agents:\n            aid     = ag.agent_id\n            obs_raw = obs_dict.get(aid, np.zeros(cfg.obs_dim, dtype=np.float32))\n            own_obs, peer_msgs = extract_obs_parts(obs_raw, cfg)\n            w      = ag.get_rep_weights()\n            act_oh = np.eye(cfg.act_dim)[\n                ag.actor.select_action(own_obs, peer_msgs, w, deterministic=True)\n            ]\n            actions[aid] = act_oh\n\n        obs_dict, rews, dones = env.step_all(actions)\n        ep_reward += sum(rews.values())\n\n        # FIX: update tracker during eval so scores reflect actual eval dynamics\n        for ag in agents:\n            ag.tracker.update(rews.get(ag.agent_id, 0.0))\n\n    rep_snap = {ag.agent_id: dict(ag.tracker.scores) for ag in agents}\n    env.close()\n    return ep_reward, rep_snap\n\n\ndef train(cfg, verbose=True):\n    """Full training run. Returns results dict with curves and metrics."""\n    set_seed(cfg.seed)\n    agent_ids = ["agent_" + str(i) for i in range(cfg.n_agents)]\n\n    agents = []\n    for aid in agent_ids:\n        peer_ids = [a for a in agent_ids if a != aid]\n        agents.append(MADDPGAgent(aid, peer_ids, cfg))\n\n    buffer  = ReplayBuffer(cfg)\n    results = {\n        "train_rewards":  [],\n        "eval_rewards":   [],\n        "eval_episodes":  [],\n        "rep_scores_log": [],\n        "critic_losses":  [],\n        "actor_losses":   [],\n    }\n\n    total_steps = 0\n    pbar = trange(cfg.n_episodes,\n                  desc="[" + cfg.trust_condition.upper() + "|" + str(cfg.n_defectors) + "def]",\n                  disable=not verbose)\n\n    for episode in pbar:\n        env      = DefectorWrapper(cfg, seed=cfg.seed + episode)\n        obs_dict = env.reset(seed=cfg.seed + episode)\n        ep_reward = 0.0\n\n        for step in range(cfg.max_steps):\n            actions = {}\n\n            for ag in agents:\n                aid     = ag.agent_id\n                obs_raw = obs_dict.get(aid, np.zeros(cfg.obs_dim, dtype=np.float32))\n                own_obs, peer_msgs = extract_obs_parts(obs_raw, cfg)\n                w = ag.get_rep_weights()\n\n                if total_steps < cfg.warmup_steps:\n                    act_int = env.sample_action(aid)\n                else:\n                    act_int = ag.actor.select_action(own_obs, peer_msgs, w)\n\n                actions[aid] = onehot(act_int, cfg.act_dim)\n\n            next_obs_dict, rews, dones = env.step_all(actions)\n            ep_reward += sum(rews.values())\n\n            # Reputation update BEFORE buffer push so scores reflect current step\n            for ag in agents:\n                ag.tracker.update(rews.get(ag.agent_id, 0.0))\n\n            rep_scores_arr = np.array(\n                [list(ag.tracker.scores.values()) for ag in agents],\n                dtype=np.float32\n            )\n            buffer.push(\n                obs        = stack_agent_data(obs_dict,      agent_ids),\n                acts       = stack_agent_data(actions,       agent_ids),\n                rews       = np.array([rews.get(a, 0.) for a in agent_ids], dtype=np.float32),\n                next_obs   = stack_agent_data(next_obs_dict, agent_ids),\n                dones      = np.array([float(dones.get(a, False)) for a in agent_ids], dtype=np.float32),\n                rep_scores = rep_scores_arr,\n            )\n\n            obs_dict    = next_obs_dict\n            total_steps += 1\n\n            if total_steps >= cfg.warmup_steps and total_steps % 2 == 0:\n                cl, al = update_agents(agents, buffer, cfg)\n                results["critic_losses"].append(cl)\n                results["actor_losses"].append(al)\n\n        results["train_rewards"].append(ep_reward)\n        env.close()\n\n        if (episode + 1) % cfg.eval_interval == 0:\n            eval_rews = []\n            rep_snap_last = {}\n            for e in range(cfg.eval_episodes):\n                er, rep_snap = run_eval_episode(agents, cfg, seed=9000 + e)\n                eval_rews.append(er)\n                if e == cfg.eval_episodes - 1:\n                    rep_snap_last = rep_snap\n\n            mean_eval = float(np.mean(eval_rews))\n            results["eval_rewards"].append(mean_eval)\n            results["eval_episodes"].append(episode + 1)\n            results["rep_scores_log"].append({\n                "episode":          episode + 1,\n                "mean_eval_reward": mean_eval,\n                "rep_scores":       rep_snap_last,\n            })\n\n            if verbose:\n                pbar.set_postfix({"eval": str(round(mean_eval, 1)), "buf": len(buffer)})\n\n    return results\n\nprint("Training loop defined — ready to run experiments.")\nprint("FIXES APPLIED:")\nprint("  1. Actor update now uses current reputation weights (not uniform)")\nprint("  2. Reputation baseline initialised from first real reward (not 0.0)")\nprint("  3. Tracker updated during eval episodes for accurate score logging")\n',
    ]
    for _src in _cells:
        exec(_src, globals())
    print("Setup complete.")
# ─────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────

# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 11 — Full Experiment Matrix (27 runs)             ║
# ║                                                          ║
# ║  3 conditions × 3 defector counts × 3 seeds = 27 runs   ║
# ║  Estimated time:                                         ║
# ║    GPU (T4): ~2–3 hours    CPU: ~8–12 hours              ║
# ║                                                          ║
# ║  Tip: set N_EPISODES = 200 for a faster ~1 hr run.      ║
# ╚══════════════════════════════════════════════════════════╝

import pickle

N_EPISODES   = 300     # reduce to 200 for faster results
EVAL_EVERY   = 25
EVAL_EPS     = 15
SEEDS        = [42, 123, 999]
CONDITIONS   = ["none", "binary", "soft"]
DEFECTORS    = [0, 1, 2]

all_results  = {}   # key: (condition, n_defectors, seed)
total_runs   = len(CONDITIONS) * len(DEFECTORS) * len(SEEDS)
run_num      = 0
SAVE_DIR     = "/content/marl_trust"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"🔬 Full experiment matrix: {total_runs} runs")
print(f"   Episodes/run : {N_EPISODES}")
print(f"   Eval every   : {EVAL_EVERY} episodes ({EVAL_EPS} eval eps each)")
print(f"   Seeds        : {SEEDS}")
print("="*60)

start_all = time.time()

for condition in CONDITIONS:
    for n_def in DEFECTORS:
        for seed in SEEDS:
            run_num += 1
            key = (condition, n_def, seed)
            print("\n[" + str(run_num) + "/" + str(total_runs) + "] condition=" + condition.upper() + " | defectors=" + str(n_def) + " | seed=" + str(seed))

            cfg = Config(
                trust_condition = condition,
                n_defectors     = n_def,
                n_episodes      = N_EPISODES,
                eval_interval   = EVAL_EVERY,
                eval_episodes   = EVAL_EPS,
                seed            = seed,
                save_dir        = SAVE_DIR,
            )

            t0  = time.time()
            res = train(cfg, verbose=True)
            t1  = time.time()

            all_results[key] = res
            final = np.mean(res["eval_rewards"][-3:]) if res["eval_rewards"] else 0.0
            print(f"   ✓ {t1-t0:.0f}s | final eval reward: {final:.2f}")

            # Save incrementally (so Colab disconnect doesn't lose everything)
            with open(f"{SAVE_DIR}/all_results.pkl", "wb") as f:
                pickle.dump(all_results, f)

total_time = time.time() - start_all
print("\n" + "="*60)
print("All " + str(total_runs) + " runs complete in " + str(round(total_time/60, 1)) + " minutes")
print("Results saved to " + SAVE_DIR + "/all_results.pkl")


In [ ]:
# ── Dependency guard ─────────────────────────────────────────────────────
import sys as _sys
if "Config" not in dir():
    print("Re-running setup cells...")
    _cells = [
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 2 — Imports & Environment Check                   ║\n# ╚══════════════════════════════════════════════════════════╝\n\nimport os, sys, time, random, copy\nfrom dataclasses import dataclass\nfrom typing import Dict, List, Tuple, Optional\n\nimport copy\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.optim as optim\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport pandas as pd\nfrom scipy import stats\nfrom tqdm.auto import tqdm, trange\n\ndef set_seed(seed: int):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\nset_seed(42)\n\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nprint("Device: " + str(DEVICE))\nif DEVICE.type == "cuda":\n    print("  GPU : " + torch.cuda.get_device_name(0))\n\n# ── mpe2 observation layout for simple_spread_v3, N=4 ────────────────────\n# [self_vel(2), self_pos(2), landmark_rel_pos(4*2=8),\n#  other_agent_rel_pos(3*2=6), communication(3*2=6)]\n# Total = 2+2+8+6+6 = 24\nfrom mpe2 import simple_spread_v3\n\nenv = simple_spread_v3.env(N=4, local_ratio=0.5, render_mode=None)\nenv.reset(seed=0)\nfor agent in env.agent_iter():\n    obs, rew, term, trunc, info = env.last()\n    env.step(env.action_space(agent).sample())\n    break\nenv.close()\n\nprint("mpe2 simple_spread_v3 (N=4):")\nprint("  obs shape : " + str(obs.shape))\nprint("  agents    : " + str(env.agents))\n\n# Confirm layout matches our constants\nN = 4\nexpected_obs_dim = 2 + 2 + N*2 + (N-1)*2 + (N-1)*2   # = 24\nassert obs.shape == (expected_obs_dim,), (\n    "Obs shape mismatch: got " + str(obs.shape) +\n    ", expected (" + str(expected_obs_dim) + ",)"\n)\nprint("  obs_dim   : " + str(expected_obs_dim) + "  [2 vel + 2 pos + 8 landmarks + 6 others + 6 comms]")\nprint("  msg_start : 18  (slice [18:24] = 3 peers x 2D comm vectors)")\nprint("  msg_end   : 24")\nprint("  Shape check passed.")\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 3 — Hyperparameter Config                         ║\n# ╚══════════════════════════════════════════════════════════╝\n\n@dataclass\nclass Config:\n    # ── Environment ──────────────────────────────────────────\n    n_agents:        int   = 4\n    n_landmarks:     int   = 4\n    local_ratio:     float = 0.5\n    max_steps:       int   = 50\n    #\n    # mpe2 obs layout for N=4:\n    # [self_vel(2), self_pos(2), landmark_rel_pos(8),\n    #  other_agent_rel_pos(6), communication(6)]\n    # Total obs_dim = 24\n    # msg_start = 18  (start of communication slice)\n    # msg_end   = 24  (end of communication slice)\n    obs_dim:         int   = 24   # 2+2+8+6+6 for N=4\n    act_dim:         int   = 5    # Discrete(5)\n    msg_dim:         int   = 2    # each peer sends a 2D comm vector\n    msg_start:       int   = 18   # index where comm slice begins\n    msg_end:         int   = 24   # index where comm slice ends\n\n    # ── Defectors ────────────────────────────────────────────\n    n_defectors:     int   = 1\n    noise_scale:     float = 1.0\n\n    # ── Trust condition ──────────────────────────────────────\n    # "none"   = Condition A: equal weights\n    # "binary" = Condition B: hard threshold gate\n    # "soft"   = Condition C: EMA reputation weighting (RWC)\n    trust_condition:    str   = "soft"\n    rep_alpha:          float = 0.05\n    rep_baseline_alpha: float = 0.01\n    rep_temperature:    float = 5.0\n    binary_threshold:   float = 0.5\n\n    # ── MADDPG ───────────────────────────────────────────────\n    hidden_dim:      int   = 128\n    actor_lr:        float = 1e-3\n    critic_lr:       float = 1e-3\n    gamma:           float = 0.95\n    tau:             float = 0.01\n    buffer_size:     int   = 100_000\n    batch_size:      int   = 256\n    warmup_steps:    int   = 1000\n\n    # ── Training ─────────────────────────────────────────────\n    n_episodes:      int   = 500\n    eval_interval:   int   = 25\n    eval_episodes:   int   = 20\n    seed:            int   = 42\n\n    # ── Output ───────────────────────────────────────────────\n    save_dir:        str   = "/content/marl_trust"\n\n    def __post_init__(self):\n        import os\n        os.makedirs(self.save_dir, exist_ok=True)\n        self.n_peers = self.n_agents - 1\n        all_ids = ["agent_" + str(i) for i in range(self.n_agents)]\n        self.defector_ids = set(all_ids[-self.n_defectors:]) if self.n_defectors > 0 else set()\n        self.honest_ids   = set(all_ids) - self.defector_ids\n\ncfg = Config()\nprint("Config created.")\nprint("  obs_dim   = " + str(cfg.obs_dim))\nprint("  msg_start = " + str(cfg.msg_start) + \n      "  (slice [" + str(cfg.msg_start) + ":" + str(cfg.msg_end) + \n      "] = " + str(cfg.n_peers) + " peers x " + str(cfg.msg_dim) + "D comms)")\nprint("  defectors = " + str(cfg.defector_ids))\n\n# Sanity check: msg slice must exactly fit n_peers * msg_dim\nassert cfg.msg_end - cfg.msg_start == cfg.n_peers * cfg.msg_dim, (\n    "Msg slice length " + str(cfg.msg_end - cfg.msg_start) +\n    " != n_peers*msg_dim " + str(cfg.n_peers * cfg.msg_dim)\n)\nprint("  Slice sanity check passed.")\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 4 — DefectorWrapper (env_wrapper.py)              ║\n# ╚══════════════════════════════════════════════════════════╝\n\n# MPE environments are now in the mpe2 package (moved from mpe2)\nfrom mpe2 import simple_spread_v3\n\nclass DefectorWrapper:\n    """\n    Wraps simple_spread_v3 and corrupts outgoing communication messages\n    from designated defector agents by injecting Gaussian noise.\n\n    Compatible with PettingZoo 1.22.x and 1.24.x.\n\n    Design:\n    - Only obs[msg_start:msg_end] is corrupted for honest agents.\n    - Defectors\' own observations remain clean (they can still sense\n      positions), making the problem harder for honest agents.\n    - The wrapper never modifies the underlying environment state.\n    """\n\n    def __init__(self, cfg, seed: int = None):\n        self.cfg  = cfg\n        self.rng  = np.random.default_rng(seed)\n        self._env = simple_spread_v3.env(\n            N           = cfg.n_agents,\n            local_ratio = cfg.local_ratio,\n            render_mode = None,\n            max_cycles  = cfg.max_steps,\n        )\n        self.agents    = self._env.possible_agents   # stable ordered list\n        self.n_agents  = len(self.agents)\n        self.agent_idx = {a: i for i, a in enumerate(self.agents)}\n\n    def reset(self, seed=None):\n        self._env.reset(seed=seed)\n        return self._collect_obs()\n\n    def step_all(self, actions: Dict[str, np.ndarray]):\n        """\n        Execute one full round: each agent acts once.\n        actions: {agent_id: one-hot np.ndarray}\n        Returns: obs, rewards, dones (all dicts)\n        """\n        rewards, dones = {a: 0.0 for a in self.agents}, {a: False for a in self.agents}\n\n        for agent in self._env.agent_iter():\n            obs_raw, rew, term, trunc, info = self._env.last()\n            done = term or trunc\n            rewards[agent] = float(rew)\n            dones[agent]   = done\n            if done:\n                self._env.step(None)\n            else:\n                act_oh = actions.get(agent)\n                if act_oh is not None:\n                    act_int = int(np.argmax(act_oh))\n                else:\n                    act_int = self._env.action_space(agent).sample()\n                self._env.step(act_int)\n\n        obs = self._collect_obs()\n        return obs, rewards, dones\n\n    def _collect_obs(self) -> Dict[str, np.ndarray]:\n        obs = {}\n        for agent in self.agents:\n            try:\n                o = self._env.observe(agent)\n                if o is None:\n                    o = np.zeros(self.cfg.obs_dim, dtype=np.float32)\n                else:\n                    o = np.array(o, dtype=np.float32).copy()\n            except Exception:\n                o = np.zeros(self.cfg.obs_dim, dtype=np.float32)\n            obs[agent] = self._maybe_corrupt(agent, o)\n        return obs\n\n    def _maybe_corrupt(self, observer: str, obs: np.ndarray) -> np.ndarray:\n        """Add Gaussian noise to peer-message slice for honest observers."""\n        if not self.cfg.defector_ids or observer in self.cfg.defector_ids:\n            return obs\n        noise = (self.rng.standard_normal(self.cfg.msg_end - self.cfg.msg_start)\n                 .astype(np.float32) * self.cfg.noise_scale)\n        obs[self.cfg.msg_start:self.cfg.msg_end] += noise\n        return obs\n\n    def sample_action(self, agent_id: str) -> int:\n        return self._env.action_space(agent_id).sample()\n\n    def close(self):\n        self._env.close()\n\n\n# ── Unit test ─────────────────────────────────────────────────────────────────\ndef test_defector_wrapper():\n    from dataclasses import replace\n\n    # With 1 defector\n    cfg_dirty = Config(n_defectors=1)\n    env_dirty = DefectorWrapper(cfg_dirty, seed=0)\n    env_dirty.reset(seed=0)\n    dummy = np.zeros(cfg_dirty.obs_dim, dtype=np.float32)\n    corrupted = env_dirty._maybe_corrupt("agent_0", dummy.copy())\n    std_dirty = corrupted[cfg_dirty.msg_start:cfg_dirty.msg_end].std()\n\n    # With 0 defectors\n    cfg_clean = Config(n_defectors=0)\n    env_clean = DefectorWrapper(cfg_clean, seed=0)\n    env_clean.reset(seed=0)\n    clean = env_clean._maybe_corrupt("agent_0", dummy.copy())\n    std_clean = clean[cfg_clean.msg_start:cfg_clean.msg_end].std()\n\n    print(f"✅ DefectorWrapper unit test")\n    print(f"   Msg slice std with  defector : {std_dirty:.4f}  (should be > 0.1)")\n    print(f"   Msg slice std without defector: {std_clean:.4f}  (should be 0.0)")\n    assert std_dirty > 0.1, "Corruption not applied!"\n    assert std_clean == 0.0, "Clean env is corrupted — bug!"\n\n    # Smoke: run a few steps\n    env2 = DefectorWrapper(Config(n_defectors=1), seed=7)\n    obs  = env2.reset(seed=7)\n    assert len(obs) == 4, f"Expected 4 agent obs, got {len(obs)}"\n    dummy_acts = {a: np.eye(5)[env2.sample_action(a)] for a in env2.agents}\n    obs2, rews, dones = env2.step_all(dummy_acts)\n    assert len(rews) == 4\n    env2.close()\n    env_dirty.close()\n    env_clean.close()\n    print("   Step test passed ✓")\n    print("   All assertions passed ✓")\n\ntest_defector_wrapper()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 5 — ReputationTracker (reputation.py)             ║\n# ╚══════════════════════════════════════════════════════════╝\n\nfrom scipy.special import softmax as scipy_softmax\n\nclass ReputationTracker:\n    """\n    Maintains a continuous reputation score r[j] in [0,1] for each peer j.\n\n    FIX: baseline is now initialised to None and set from the first real\n    reward, preventing the \'always negative delta\' problem that caused all\n    scores to drift to 0 then snap to 1.0 when rewards improved.\n\n    Update rule (per step):\n        delta  = reward - baseline\n        signal = 1.0 if delta > 0 else 0.0\n        r[j]  <- (1 - alpha) * r[j] + alpha * signal  for all j\n        baseline <- (1 - b_alpha) * baseline + b_alpha * reward\n    """\n\n    def __init__(self, peer_ids, cfg):\n        self.peer_ids    = list(peer_ids)\n        self.alpha       = cfg.rep_alpha\n        self.b_alpha     = cfg.rep_baseline_alpha\n        self.temperature = cfg.rep_temperature\n        self.scores      = {j: 0.5 for j in peer_ids}\n        self.baseline    = None    # FIX: initialise lazily from first reward\n\n    def update(self, reward):\n        """Called once per step with this agent\'s observed reward."""\n        # FIX: initialise baseline to first real reward so delta starts near 0\n        if self.baseline is None:\n            self.baseline = reward\n            return  # skip update on very first step\n\n        delta  = reward - self.baseline\n        signal = 1.0 if delta > 0 else 0.0\n\n        self.baseline = (1 - self.b_alpha) * self.baseline + self.b_alpha * reward\n\n        for j in self.peer_ids:\n            self.scores[j] = (\n                (1 - self.alpha) * self.scores[j] + self.alpha * signal\n            )\n\n    def get_weights(self):\n        """Softmax-normalised weights, ordered by peer_ids."""\n        scores_arr = np.array([self.scores[j] for j in self.peer_ids],\n                               dtype=np.float32)\n        return scipy_softmax(scores_arr * self.temperature).astype(np.float32)\n\n    def get_weights_binary(self, threshold=0.5):\n        """Condition B: hard binary gate, re-normalised."""\n        scores_arr = np.array([self.scores[j] for j in self.peer_ids],\n                               dtype=np.float32)\n        weights = (scores_arr >= threshold).astype(np.float32)\n        total = weights.sum()\n        if total == 0:\n            weights = np.ones(len(self.peer_ids), dtype=np.float32)\n            total   = float(len(self.peer_ids))\n        return weights / total\n\n    def get_weights_uniform(self):\n        """Condition A: no trust — uniform weights."""\n        n = len(self.peer_ids)\n        return np.ones(n, dtype=np.float32) / n\n\n    def get_scores_dict(self):\n        return dict(self.scores)\n\n    def reset(self):\n        self.scores   = {j: 0.5 for j in self.peer_ids}\n        self.baseline = None   # FIX: reset to None, not 0.0\n\n\n# ── Unit tests ────────────────────────────────────────────────────────────────\ndef test_reputation_tracker():\n    cfg_t = Config()\n    peers = ["agent_1", "agent_2", "agent_3"]\n    tracker = ReputationTracker(peers, cfg_t)\n\n    # Test 1: neutral init\n    w = tracker.get_weights()\n    assert abs(w.sum() - 1.0) < 1e-5, "Weights do not sum to 1"\n    assert abs(w[0] - w[1]) < 1e-5, "Initial weights not uniform"\n    print("  Test 1 passed: neutral init OK")\n\n    # Test 2: positive drift — rewards consistently above baseline\n    tracker.reset()\n    for i in range(200):\n        tracker.update(reward=-5.0 + i * 0.05)   # rising rewards\n    scores = list(tracker.scores.values())\n    print("  Test 2: scores after rising rewards: " + str([round(s,3) for s in scores]))\n    assert all(s > 0.5 for s in scores), "Scores did not drift up: " + str(scores)\n    print("  Test 2 passed: positive drift OK (mean=" + str(round(float(np.mean(scores)),3)) + ")")\n\n    # Test 3: negative drift — rewards consistently below baseline\n    tracker.reset()\n    for i in range(200):\n        tracker.update(reward=-5.0 - i * 0.05)   # falling rewards\n    scores = list(tracker.scores.values())\n    print("  Test 3: scores after falling rewards: " + str([round(s,3) for s in scores]))\n    assert all(s < 0.5 for s in scores), "Scores did not drift down: " + str(scores)\n    print("  Test 3 passed: negative drift OK (mean=" + str(round(float(np.mean(scores)),3)) + ")")\n\n    # Test 4: weights always sum to 1\n    for _ in range(50):\n        tracker.update(reward=np.random.randn() - 5.0)\n    for method in ["get_weights", "get_weights_binary", "get_weights_uniform"]:\n        w = getattr(tracker, method)()\n        assert abs(w.sum() - 1.0) < 1e-5, method + " does not sum to 1"\n    print("  Test 4 passed: all weight methods sum to 1.0 OK")\n\n    # Test 5: FIX verification — baseline initialised from first reward\n    tracker2 = ReputationTracker(peers, cfg_t)\n    tracker2.update(reward=-8.0)   # first step, sets baseline to -8, no score update\n    scores_after_first = list(tracker2.scores.values())\n    assert all(s == 0.5 for s in scores_after_first), "Scores changed on first step — baseline bug!"\n    tracker2.update(reward=-7.5)   # second step: delta = -7.5 - (-8.0) = +0.5, signal=1\n    scores_after_second = list(tracker2.scores.values())\n    assert all(s > 0.5 for s in scores_after_second), "Positive delta did not increase scores"\n    print("  Test 5 passed: baseline initialisation fix verified OK")\n\n    print("")\n    print("ReputationTracker — all 5 unit tests passed.")\n\ntest_reputation_tracker()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 6 — Actor & Critic Networks (networks.py)         ║\n# ╚══════════════════════════════════════════════════════════╝\n\nclass Actor(nn.Module):\n    """\n    MADDPG actor with reputation-weighted message aggregation.\n\n    Forward pass:\n        1. Receive own_obs [batch, msg_start] and\n           peer_msgs [batch, n_peers, msg_dim]\n        2. Aggregate peer_msgs weighted by rep_weights [batch, n_peers]\n        3. Concatenate own_obs + aggregated_msg → policy MLP → action logits\n\n    The reputation weighting is the ONLY modification from standard MADDPG.\n    When rep_weights is uniform, output is identical to an unmodified actor.\n    """\n\n    def __init__(self, own_obs_dim: int, n_peers: int,\n                 msg_dim: int, act_dim: int, hidden_dim: int):\n        super().__init__()\n        self.n_peers  = n_peers\n        self.msg_dim  = msg_dim\n        self.act_dim  = act_dim\n\n        input_dim = own_obs_dim + msg_dim   # own obs + aggregated msg\n        self.net = nn.Sequential(\n            nn.Linear(input_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, act_dim),\n        )\n\n    def forward(self,\n                own_obs:     torch.Tensor,   # [batch, own_obs_dim]\n                peer_msgs:   torch.Tensor,   # [batch, n_peers, msg_dim]\n                rep_weights: torch.Tensor,   # [batch, n_peers]\n                ) -> torch.Tensor:           # [batch, act_dim]\n\n        # ── Reputation-weighted message aggregation ───────────────────────────\n        w   = rep_weights.unsqueeze(-1)              # [batch, n_peers, 1]\n        agg = (peer_msgs * w).sum(dim=1)             # [batch, msg_dim]\n        # ─────────────────────────────────────────────────────────────────────\n\n        x = torch.cat([own_obs, agg], dim=-1)        # [batch, own_obs_dim + msg_dim]\n        return self.net(x)                           # [batch, act_dim]\n\n    def select_action(self, own_obs: np.ndarray,\n                      peer_msgs: np.ndarray,\n                      rep_weights: np.ndarray,\n                      deterministic: bool = False) -> np.ndarray:\n        with torch.no_grad():\n            o = torch.FloatTensor(own_obs).unsqueeze(0).to(DEVICE)\n            m = torch.FloatTensor(peer_msgs).unsqueeze(0).to(DEVICE)\n            w = torch.FloatTensor(rep_weights).unsqueeze(0).to(DEVICE)\n            logits = self.forward(o, m, w).squeeze(0)\n            if deterministic:\n                action = logits.argmax().item()\n            else:\n                probs  = F.softmax(logits, dim=-1)\n                action = torch.multinomial(probs, 1).item()\n        return action\n\n\nclass Critic(nn.Module):\n    """\n    Centralised MADDPG critic: Q(all_obs, all_actions).\n    Has full visibility during training (CTDE paradigm).\n    """\n\n    def __init__(self, n_agents: int, obs_dim: int,\n                 act_dim: int, hidden_dim: int):\n        super().__init__()\n        input_dim = n_agents * (obs_dim + act_dim)\n        self.net = nn.Sequential(\n            nn.Linear(input_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, 1),\n        )\n\n    def forward(self, all_obs: torch.Tensor,\n                all_acts: torch.Tensor) -> torch.Tensor:\n        """\n        all_obs:  [batch, n_agents, obs_dim]\n        all_acts: [batch, n_agents, act_dim]\n        Returns:  [batch, 1]\n        """\n        batch = all_obs.shape[0]\n        x = torch.cat([\n            all_obs.reshape(batch, -1),\n            all_acts.reshape(batch, -1)\n        ], dim=-1)\n        return self.net(x)\n\n\n# ── Verify parity: uniform weights == standard MADDPG ─────────────────────────\ndef test_network_parity():\n    cfg_t = Config()\n    own_obs_dim = cfg_t.msg_start        # 28 dims (everything before msg slice)\n    actor = Actor(own_obs_dim, cfg_t.n_peers, cfg_t.msg_dim,\n                  cfg_t.act_dim, cfg_t.hidden_dim)\n\n    # Uniform weights → same as concatenating mean messages\n    batch = 4\n    obs   = torch.randn(batch, own_obs_dim)\n    msgs  = torch.randn(batch, cfg_t.n_peers, cfg_t.msg_dim)\n    w_uni = torch.ones(batch, cfg_t.n_peers) / cfg_t.n_peers\n\n    out_uni  = actor(obs, msgs, w_uni)\n\n    # Manually compute what unmodified MADDPG would do: simple mean\n    agg_mean = msgs.mean(dim=1)\n    x_manual = torch.cat([obs, agg_mean], dim=-1)\n    out_man  = actor.net(x_manual)\n\n    diff = (out_uni - out_man).abs().max().item()\n    assert diff < 1e-5, f"Parity test failed, max diff: {diff}"\n    print(f"✅ Network parity test: uniform weights == mean aggregation (max diff = {diff:.2e}) ✓")\n\n    # Shape checks\n    critic = Critic(cfg_t.n_agents, cfg_t.obs_dim, cfg_t.act_dim, cfg_t.hidden_dim)\n    all_obs  = torch.randn(batch, cfg_t.n_agents, cfg_t.obs_dim)\n    all_acts = torch.randn(batch, cfg_t.n_agents, cfg_t.act_dim)\n    q = critic(all_obs, all_acts)\n    assert q.shape == (batch, 1), f"Critic output shape wrong: {q.shape}"\n    print(f"   Critic output shape: {q.shape} ✓")\n    print(f"   Actor  output shape: {out_uni.shape} ✓")\n\ntest_network_parity()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 7 — Replay Buffer (buffer.py)                     ║\n# ╚══════════════════════════════════════════════════════════╝\n\nclass ReplayBuffer:\n    """\n    Circular replay buffer for MADDPG.\n    Stores transitions for all agents simultaneously.\n    Also stores reputation scores for logging/analysis.\n    """\n\n    def __init__(self, cfg: Config):\n        self.capacity   = cfg.buffer_size\n        self.n_agents   = cfg.n_agents\n        self.obs_dim    = cfg.obs_dim\n        self.act_dim    = cfg.act_dim\n        self.n_peers    = cfg.n_peers\n        self.ptr        = 0\n        self.size       = 0\n\n        # Pre-allocate numpy arrays for speed\n        self.obs       = np.zeros((self.capacity, self.n_agents, self.obs_dim),  dtype=np.float32)\n        self.acts      = np.zeros((self.capacity, self.n_agents, self.act_dim),  dtype=np.float32)\n        self.rews      = np.zeros((self.capacity, self.n_agents),                dtype=np.float32)\n        self.next_obs  = np.zeros((self.capacity, self.n_agents, self.obs_dim),  dtype=np.float32)\n        self.dones     = np.zeros((self.capacity, self.n_agents),                dtype=np.float32)\n        self.rep_scores= np.zeros((self.capacity, self.n_agents, self.n_peers),  dtype=np.float32)\n\n    def push(self,\n             obs:        np.ndarray,   # [n_agents, obs_dim]\n             acts:       np.ndarray,   # [n_agents, act_dim]\n             rews:       np.ndarray,   # [n_agents]\n             next_obs:   np.ndarray,   # [n_agents, obs_dim]\n             dones:      np.ndarray,   # [n_agents]\n             rep_scores: np.ndarray,   # [n_agents, n_peers]\n             ):\n        idx = self.ptr % self.capacity\n        self.obs[idx]        = obs\n        self.acts[idx]       = acts\n        self.rews[idx]       = rews\n        self.next_obs[idx]   = next_obs\n        self.dones[idx]      = dones\n        self.rep_scores[idx] = rep_scores\n        self.ptr  += 1\n        self.size  = min(self.size + 1, self.capacity)\n\n    def sample(self, batch_size: int):\n        idx = np.random.randint(0, self.size, size=batch_size)\n        to_t = lambda x: torch.FloatTensor(x).to(DEVICE)\n        return (\n            to_t(self.obs[idx]),        # [B, n_agents, obs_dim]\n            to_t(self.acts[idx]),       # [B, n_agents, act_dim]\n            to_t(self.rews[idx]),       # [B, n_agents]\n            to_t(self.next_obs[idx]),   # [B, n_agents, obs_dim]\n            to_t(self.dones[idx]),      # [B, n_agents]\n        )\n\n    def __len__(self):\n        return self.size\n\n\ndef test_replay_buffer():\n    cfg_t  = Config()\n    buf    = ReplayBuffer(cfg_t)\n    n, na, od, ad, np_ = 1000, cfg_t.n_agents, cfg_t.obs_dim, cfg_t.act_dim, cfg_t.n_peers\n\n    for _ in range(n):\n        buf.push(\n            np.random.randn(na, od).astype(np.float32),\n            np.random.randn(na, ad).astype(np.float32),\n            np.random.randn(na).astype(np.float32),\n            np.random.randn(na, od).astype(np.float32),\n            np.zeros(na, dtype=np.float32),\n            np.random.rand(na, np_).astype(np.float32),\n        )\n\n    assert len(buf) == n\n    obs, acts, rews, nobs, dones = buf.sample(256)\n    assert obs.shape   == (256, na, od), f"obs shape wrong: {obs.shape}"\n    assert acts.shape  == (256, na, ad)\n    assert rews.shape  == (256, na)\n    print(f"✅ ReplayBuffer: pushed {n} transitions, sampled batch of 256")\n    print(f"   obs shape: {obs.shape}, acts shape: {acts.shape} ✓")\n\ntest_replay_buffer()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 8 — MADDPGAgent & Helper Functions                ║\n# ╚══════════════════════════════════════════════════════════╝\n\nclass MADDPGAgent:\n    """\n    Single agent in the MADDPG framework.\n    Holds actor + target_actor, critic + target_critic,\n    their optimizers, and a ReputationTracker.\n    """\n\n    def __init__(self, agent_id: str, peer_ids: List[str], cfg):\n        self.agent_id = agent_id\n        self.peer_ids = peer_ids\n        self.cfg      = cfg\n        own_obs_dim   = cfg.msg_start   # dims before the comm slice\n\n        # ── Networks ──────────────────────────────────────────────────────\n        self.actor   = Actor(own_obs_dim, cfg.n_peers, cfg.msg_dim,\n                             cfg.act_dim, cfg.hidden_dim).to(DEVICE)\n        self.t_actor = copy.deepcopy(self.actor)\n\n        self.critic  = Critic(cfg.n_agents, cfg.obs_dim,\n                              cfg.act_dim, cfg.hidden_dim).to(DEVICE)\n        self.t_critic = copy.deepcopy(self.critic)\n\n        # ── Optimisers ────────────────────────────────────────────────────\n        self.actor_opt  = optim.Adam(self.actor.parameters(),  lr=cfg.actor_lr)\n        self.critic_opt = optim.Adam(self.critic.parameters(), lr=cfg.critic_lr)\n\n        # ── Reputation tracker ────────────────────────────────────────────\n        self.tracker = ReputationTracker(peer_ids, cfg)\n\n    def get_rep_weights(self) -> np.ndarray:\n        c = self.cfg.trust_condition\n        if   c == "none":   return self.tracker.get_weights_uniform()\n        elif c == "binary": return self.tracker.get_weights_binary(self.cfg.binary_threshold)\n        else:               return self.tracker.get_weights()   # soft\n\n    @torch.no_grad()\n    def soft_update(self, tau: float):\n        for p, tp in zip(self.actor.parameters(), self.t_actor.parameters()):\n            tp.data.copy_(tau * p.data + (1 - tau) * tp.data)\n        for p, tp in zip(self.critic.parameters(), self.t_critic.parameters()):\n            tp.data.copy_(tau * p.data + (1 - tau) * tp.data)\n\n\n# ── Helper functions ──────────────────────────────────────────────────────\n\ndef extract_obs_parts(obs: np.ndarray, cfg):\n    """Split raw obs into own_obs and peer_msgs."""\n    own_obs   = obs[:cfg.msg_start]                          # [msg_start]\n    msg_block = obs[cfg.msg_start:cfg.msg_end]               # [n_peers * msg_dim]\n    peer_msgs = msg_block.reshape(cfg.n_peers, cfg.msg_dim)  # [n_peers, msg_dim]\n    return own_obs, peer_msgs\n\n\ndef onehot(action: int, dim: int) -> np.ndarray:\n    v = np.zeros(dim, dtype=np.float32)\n    v[action] = 1.0\n    return v\n\n\ndef stack_agent_data(d: Dict[str, np.ndarray],\n                     agent_list: List[str]) -> np.ndarray:\n    """Stack per-agent arrays into [n_agents, *] numpy array."""\n    return np.stack([d[a] for a in agent_list], axis=0)\n\n\n# ── Quick sanity check ────────────────────────────────────────────────────\ndef test_maddpg_agent():\n    cfg_t    = Config()\n    peer_ids = ["agent_1", "agent_2", "agent_3"]\n    agent    = MADDPGAgent("agent_0", peer_ids, cfg_t)\n\n    # Check networks are on the right device\n    assert next(agent.actor.parameters()).device.type == DEVICE.type\n    assert next(agent.critic.parameters()).device.type == DEVICE.type\n\n    # Check weight shapes\n    w = agent.get_rep_weights()\n    assert len(w) == cfg_t.n_peers, "Wrong number of weights"\n    assert abs(w.sum() - 1.0) < 1e-5, "Weights do not sum to 1"\n\n    # Check extract_obs_parts\n    dummy_obs = np.random.randn(cfg_t.obs_dim).astype(np.float32)\n    own_obs, peer_msgs = extract_obs_parts(dummy_obs, cfg_t)\n    assert own_obs.shape   == (cfg_t.msg_start,),                "own_obs shape wrong"\n    assert peer_msgs.shape == (cfg_t.n_peers, cfg_t.msg_dim),   "peer_msgs shape wrong"\n\n    n_actor  = sum(p.numel() for p in agent.actor.parameters())\n    n_critic = sum(p.numel() for p in agent.critic.parameters())\n    print("MADDPGAgent sanity check passed.")\n    print("  own_obs_dim : " + str(cfg_t.msg_start))\n    print("  actor params: " + str(n_actor))\n    print("  critic params: " + str(n_critic))\n    print("  rep weights : " + str(w) + "  (sum=" + str(round(w.sum(),4)) + ")")\n\ntest_maddpg_agent()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 9 — Training & Evaluation Loop (train.py)         ║\n# ╚══════════════════════════════════════════════════════════╝\n\ndef update_agents(agents, buffer, cfg):\n    """One MADDPG gradient update step for all agents."""\n    if len(buffer) < cfg.batch_size:\n        return 0.0, 0.0\n\n    obs_b, acts_b, rews_b, nobs_b, dones_b = buffer.sample(cfg.batch_size)\n    total_closs, total_aloss = 0.0, 0.0\n\n    for i, agent in enumerate(agents):\n        # ── Critic update ─────────────────────────────────────────────────────\n        with torch.no_grad():\n            t_acts_list = []\n            for j, ag in enumerate(agents):\n                o_j = nobs_b[:, j, :cfg.msg_start]\n                m_j = nobs_b[:, j, cfg.msg_start:cfg.msg_end]\n                m_j = m_j.view(-1, cfg.n_peers, cfg.msg_dim)\n                # Target actor always uses uniform weights (stable training target)\n                w_j = torch.ones(cfg.batch_size, cfg.n_peers, device=DEVICE) / cfg.n_peers\n                t_logits = ag.t_actor(o_j, m_j, w_j)\n                t_act    = F.softmax(t_logits, dim=-1)\n                t_acts_list.append(t_act)\n            t_acts = torch.stack(t_acts_list, dim=1)\n            next_q = agent.t_critic(nobs_b, t_acts)\n            r_i    = rews_b[:, i:i+1]\n            d_i    = dones_b[:, i:i+1]\n            target = r_i + cfg.gamma * (1 - d_i) * next_q\n\n        current_q = agent.critic(obs_b, acts_b)\n        closs     = F.mse_loss(current_q, target)\n        agent.critic_opt.zero_grad()\n        closs.backward()\n        torch.nn.utils.clip_grad_norm_(agent.critic.parameters(), 1.0)\n        agent.critic_opt.step()\n\n        # ── Actor update — FIX: use CURRENT reputation weights ────────────────\n        # The actor must be trained WITH reputation weights so it learns to\n        # exploit the trust signal. Using uniform weights here (the original bug)\n        # meant the network was never exposed to reputation-modulated inputs\n        # during training, so it could not benefit from them at inference time.\n        o_i = obs_b[:, i, :cfg.msg_start]\n        m_i = obs_b[:, i, cfg.msg_start:cfg.msg_end].view(-1, cfg.n_peers, cfg.msg_dim)\n\n        # Get current rep weights for this agent (shape: [n_peers])\n        rep_w = torch.FloatTensor(agent.get_rep_weights()).to(DEVICE)\n        # Expand to batch: [1, n_peers] -> [batch, n_peers]\n        rep_w_batch = rep_w.unsqueeze(0).expand(cfg.batch_size, -1)\n\n        logits  = agent.actor(o_i, m_i, rep_w_batch)\n        new_act = F.softmax(logits, dim=-1)\n        acts_q  = acts_b.clone()\n        acts_q[:, i, :] = new_act\n        aloss = -agent.critic(obs_b, acts_q).mean()\n        agent.actor_opt.zero_grad()\n        aloss.backward()\n        torch.nn.utils.clip_grad_norm_(agent.actor.parameters(), 1.0)\n        agent.actor_opt.step()\n\n        agent.soft_update(cfg.tau)\n        total_closs += closs.item()\n        total_aloss += aloss.item()\n\n    return total_closs / cfg.n_agents, total_aloss / cfg.n_agents\n\n\ndef run_eval_episode(agents, cfg, seed=0):\n    """One evaluation episode — deterministic actions, no buffer push.\n    FIX: tracker is updated during eval so rep scores reflect eval behaviour."""\n    env      = DefectorWrapper(cfg, seed=seed)\n    obs_dict = env.reset(seed=seed)\n    ep_reward = 0.0\n\n    for step in range(cfg.max_steps):\n        actions = {}\n        for ag in agents:\n            aid     = ag.agent_id\n            obs_raw = obs_dict.get(aid, np.zeros(cfg.obs_dim, dtype=np.float32))\n            own_obs, peer_msgs = extract_obs_parts(obs_raw, cfg)\n            w      = ag.get_rep_weights()\n            act_oh = np.eye(cfg.act_dim)[\n                ag.actor.select_action(own_obs, peer_msgs, w, deterministic=True)\n            ]\n            actions[aid] = act_oh\n\n        obs_dict, rews, dones = env.step_all(actions)\n        ep_reward += sum(rews.values())\n\n        # FIX: update tracker during eval so scores reflect actual eval dynamics\n        for ag in agents:\n            ag.tracker.update(rews.get(ag.agent_id, 0.0))\n\n    rep_snap = {ag.agent_id: dict(ag.tracker.scores) for ag in agents}\n    env.close()\n    return ep_reward, rep_snap\n\n\ndef train(cfg, verbose=True):\n    """Full training run. Returns results dict with curves and metrics."""\n    set_seed(cfg.seed)\n    agent_ids = ["agent_" + str(i) for i in range(cfg.n_agents)]\n\n    agents = []\n    for aid in agent_ids:\n        peer_ids = [a for a in agent_ids if a != aid]\n        agents.append(MADDPGAgent(aid, peer_ids, cfg))\n\n    buffer  = ReplayBuffer(cfg)\n    results = {\n        "train_rewards":  [],\n        "eval_rewards":   [],\n        "eval_episodes":  [],\n        "rep_scores_log": [],\n        "critic_losses":  [],\n        "actor_losses":   [],\n    }\n\n    total_steps = 0\n    pbar = trange(cfg.n_episodes,\n                  desc="[" + cfg.trust_condition.upper() + "|" + str(cfg.n_defectors) + "def]",\n                  disable=not verbose)\n\n    for episode in pbar:\n        env      = DefectorWrapper(cfg, seed=cfg.seed + episode)\n        obs_dict = env.reset(seed=cfg.seed + episode)\n        ep_reward = 0.0\n\n        for step in range(cfg.max_steps):\n            actions = {}\n\n            for ag in agents:\n                aid     = ag.agent_id\n                obs_raw = obs_dict.get(aid, np.zeros(cfg.obs_dim, dtype=np.float32))\n                own_obs, peer_msgs = extract_obs_parts(obs_raw, cfg)\n                w = ag.get_rep_weights()\n\n                if total_steps < cfg.warmup_steps:\n                    act_int = env.sample_action(aid)\n                else:\n                    act_int = ag.actor.select_action(own_obs, peer_msgs, w)\n\n                actions[aid] = onehot(act_int, cfg.act_dim)\n\n            next_obs_dict, rews, dones = env.step_all(actions)\n            ep_reward += sum(rews.values())\n\n            # Reputation update BEFORE buffer push so scores reflect current step\n            for ag in agents:\n                ag.tracker.update(rews.get(ag.agent_id, 0.0))\n\n            rep_scores_arr = np.array(\n                [list(ag.tracker.scores.values()) for ag in agents],\n                dtype=np.float32\n            )\n            buffer.push(\n                obs        = stack_agent_data(obs_dict,      agent_ids),\n                acts       = stack_agent_data(actions,       agent_ids),\n                rews       = np.array([rews.get(a, 0.) for a in agent_ids], dtype=np.float32),\n                next_obs   = stack_agent_data(next_obs_dict, agent_ids),\n                dones      = np.array([float(dones.get(a, False)) for a in agent_ids], dtype=np.float32),\n                rep_scores = rep_scores_arr,\n            )\n\n            obs_dict    = next_obs_dict\n            total_steps += 1\n\n            if total_steps >= cfg.warmup_steps and total_steps % 2 == 0:\n                cl, al = update_agents(agents, buffer, cfg)\n                results["critic_losses"].append(cl)\n                results["actor_losses"].append(al)\n\n        results["train_rewards"].append(ep_reward)\n        env.close()\n\n        if (episode + 1) % cfg.eval_interval == 0:\n            eval_rews = []\n            rep_snap_last = {}\n            for e in range(cfg.eval_episodes):\n                er, rep_snap = run_eval_episode(agents, cfg, seed=9000 + e)\n                eval_rews.append(er)\n                if e == cfg.eval_episodes - 1:\n                    rep_snap_last = rep_snap\n\n            mean_eval = float(np.mean(eval_rews))\n            results["eval_rewards"].append(mean_eval)\n            results["eval_episodes"].append(episode + 1)\n            results["rep_scores_log"].append({\n                "episode":          episode + 1,\n                "mean_eval_reward": mean_eval,\n                "rep_scores":       rep_snap_last,\n            })\n\n            if verbose:\n                pbar.set_postfix({"eval": str(round(mean_eval, 1)), "buf": len(buffer)})\n\n    return results\n\nprint("Training loop defined — ready to run experiments.")\nprint("FIXES APPLIED:")\nprint("  1. Actor update now uses current reputation weights (not uniform)")\nprint("  2. Reputation baseline initialised from first real reward (not 0.0)")\nprint("  3. Tracker updated during eval episodes for accurate score logging")\n',
    ]
    for _src in _cells:
        exec(_src, globals())
    print("Setup complete.")
# ─────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────

# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 12 — Alpha Sensitivity Sweep (Condition C only)   ║
# ║                                                          ║
# ║  3 conditions × 3 defector counts × 3 seeds             ║
# ║  Estimated time:                                         ║
# ║    GPU: ~2.5–4 hours    CPU: ~8–12 hours                ║
# ║                                                          ║
# ║  To save time, set N_EPISODES = 200 for a quick run.    ║
# ╚══════════════════════════════════════════════════════════╝

N_EPISODES   = 300     # reduce to 200 for faster results
EVAL_EVERY   = 25
EVAL_EPS     = 15
SEEDS        = [42, 123, 999]
CONDITIONS   = ["none", "binary", "soft"]
DEFECTORS    = [0, 1, 2]

all_results  = {}   # key: (condition, n_defectors, seed)
total_runs   = len(CONDITIONS) * len(DEFECTORS) * len(SEEDS)
run_num      = 0

print(f"🔬 Starting full experiment matrix: {total_runs} runs")
print(f"   Episodes/run: {N_EPISODES} | Eval every: {EVAL_EVERY} | Seeds: {SEEDS}")
print("="*60)

start_all = time.time()

for condition in CONDITIONS:
    for n_def in DEFECTORS:
        for seed in SEEDS:
            run_num += 1
            key = (condition, n_def, seed)
            print("\n[" + str(run_num) + "/" + str(total_runs) + "] condition=" + condition.upper() + " | defectors=" + str(n_def) + " | seed=" + str(seed))

            cfg = Config(
                trust_condition = condition,
                n_defectors     = n_def,
                n_episodes      = N_EPISODES,
                eval_interval   = EVAL_EVERY,
                eval_episodes   = EVAL_EPS,
                seed            = seed,
            )

            t0  = time.time()
            res = train(cfg, verbose=True)
            t1  = time.time()

            all_results[key] = res
            final_rew = np.mean(res["eval_rewards"][-3:]) if res["eval_rewards"] else 0
            print(f"   ✓ Done in {t1-t0:.0f}s | Final eval reward: {final_rew:.2f}")

total_time = time.time() - start_all
print("\n" + "="*60)
print("All " + str(total_runs) + " runs complete in " + str(round(total_time/60, 1)) + " minutes")

# Save results
import pickle
results_path = f"{Config().save_dir}/all_results.pkl"
with open(results_path, "wb") as f:
    pickle.dump(all_results, f)
print(f"💾 Results saved to {results_path}")


In [ ]:
# ── Dependency guard ─────────────────────────────────────────────────────
import sys as _sys
if "Config" not in dir():
    print("Re-running setup cells...")
    _cells = [
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 2 — Imports & Environment Check                   ║\n# ╚══════════════════════════════════════════════════════════╝\n\nimport os, sys, time, random, copy\nfrom dataclasses import dataclass\nfrom typing import Dict, List, Tuple, Optional\n\nimport copy\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.optim as optim\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport pandas as pd\nfrom scipy import stats\nfrom tqdm.auto import tqdm, trange\n\ndef set_seed(seed: int):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\nset_seed(42)\n\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nprint("Device: " + str(DEVICE))\nif DEVICE.type == "cuda":\n    print("  GPU : " + torch.cuda.get_device_name(0))\n\n# ── mpe2 observation layout for simple_spread_v3, N=4 ────────────────────\n# [self_vel(2), self_pos(2), landmark_rel_pos(4*2=8),\n#  other_agent_rel_pos(3*2=6), communication(3*2=6)]\n# Total = 2+2+8+6+6 = 24\nfrom mpe2 import simple_spread_v3\n\nenv = simple_spread_v3.env(N=4, local_ratio=0.5, render_mode=None)\nenv.reset(seed=0)\nfor agent in env.agent_iter():\n    obs, rew, term, trunc, info = env.last()\n    env.step(env.action_space(agent).sample())\n    break\nenv.close()\n\nprint("mpe2 simple_spread_v3 (N=4):")\nprint("  obs shape : " + str(obs.shape))\nprint("  agents    : " + str(env.agents))\n\n# Confirm layout matches our constants\nN = 4\nexpected_obs_dim = 2 + 2 + N*2 + (N-1)*2 + (N-1)*2   # = 24\nassert obs.shape == (expected_obs_dim,), (\n    "Obs shape mismatch: got " + str(obs.shape) +\n    ", expected (" + str(expected_obs_dim) + ",)"\n)\nprint("  obs_dim   : " + str(expected_obs_dim) + "  [2 vel + 2 pos + 8 landmarks + 6 others + 6 comms]")\nprint("  msg_start : 18  (slice [18:24] = 3 peers x 2D comm vectors)")\nprint("  msg_end   : 24")\nprint("  Shape check passed.")\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 3 — Hyperparameter Config                         ║\n# ╚══════════════════════════════════════════════════════════╝\n\n@dataclass\nclass Config:\n    # ── Environment ──────────────────────────────────────────\n    n_agents:        int   = 4\n    n_landmarks:     int   = 4\n    local_ratio:     float = 0.5\n    max_steps:       int   = 50\n    #\n    # mpe2 obs layout for N=4:\n    # [self_vel(2), self_pos(2), landmark_rel_pos(8),\n    #  other_agent_rel_pos(6), communication(6)]\n    # Total obs_dim = 24\n    # msg_start = 18  (start of communication slice)\n    # msg_end   = 24  (end of communication slice)\n    obs_dim:         int   = 24   # 2+2+8+6+6 for N=4\n    act_dim:         int   = 5    # Discrete(5)\n    msg_dim:         int   = 2    # each peer sends a 2D comm vector\n    msg_start:       int   = 18   # index where comm slice begins\n    msg_end:         int   = 24   # index where comm slice ends\n\n    # ── Defectors ────────────────────────────────────────────\n    n_defectors:     int   = 1\n    noise_scale:     float = 1.0\n\n    # ── Trust condition ──────────────────────────────────────\n    # "none"   = Condition A: equal weights\n    # "binary" = Condition B: hard threshold gate\n    # "soft"   = Condition C: EMA reputation weighting (RWC)\n    trust_condition:    str   = "soft"\n    rep_alpha:          float = 0.05\n    rep_baseline_alpha: float = 0.01\n    rep_temperature:    float = 5.0\n    binary_threshold:   float = 0.5\n\n    # ── MADDPG ───────────────────────────────────────────────\n    hidden_dim:      int   = 128\n    actor_lr:        float = 1e-3\n    critic_lr:       float = 1e-3\n    gamma:           float = 0.95\n    tau:             float = 0.01\n    buffer_size:     int   = 100_000\n    batch_size:      int   = 256\n    warmup_steps:    int   = 1000\n\n    # ── Training ─────────────────────────────────────────────\n    n_episodes:      int   = 500\n    eval_interval:   int   = 25\n    eval_episodes:   int   = 20\n    seed:            int   = 42\n\n    # ── Output ───────────────────────────────────────────────\n    save_dir:        str   = "/content/marl_trust"\n\n    def __post_init__(self):\n        import os\n        os.makedirs(self.save_dir, exist_ok=True)\n        self.n_peers = self.n_agents - 1\n        all_ids = ["agent_" + str(i) for i in range(self.n_agents)]\n        self.defector_ids = set(all_ids[-self.n_defectors:]) if self.n_defectors > 0 else set()\n        self.honest_ids   = set(all_ids) - self.defector_ids\n\ncfg = Config()\nprint("Config created.")\nprint("  obs_dim   = " + str(cfg.obs_dim))\nprint("  msg_start = " + str(cfg.msg_start) + \n      "  (slice [" + str(cfg.msg_start) + ":" + str(cfg.msg_end) + \n      "] = " + str(cfg.n_peers) + " peers x " + str(cfg.msg_dim) + "D comms)")\nprint("  defectors = " + str(cfg.defector_ids))\n\n# Sanity check: msg slice must exactly fit n_peers * msg_dim\nassert cfg.msg_end - cfg.msg_start == cfg.n_peers * cfg.msg_dim, (\n    "Msg slice length " + str(cfg.msg_end - cfg.msg_start) +\n    " != n_peers*msg_dim " + str(cfg.n_peers * cfg.msg_dim)\n)\nprint("  Slice sanity check passed.")\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 4 — DefectorWrapper (env_wrapper.py)              ║\n# ╚══════════════════════════════════════════════════════════╝\n\n# MPE environments are now in the mpe2 package (moved from mpe2)\nfrom mpe2 import simple_spread_v3\n\nclass DefectorWrapper:\n    """\n    Wraps simple_spread_v3 and corrupts outgoing communication messages\n    from designated defector agents by injecting Gaussian noise.\n\n    Compatible with PettingZoo 1.22.x and 1.24.x.\n\n    Design:\n    - Only obs[msg_start:msg_end] is corrupted for honest agents.\n    - Defectors\' own observations remain clean (they can still sense\n      positions), making the problem harder for honest agents.\n    - The wrapper never modifies the underlying environment state.\n    """\n\n    def __init__(self, cfg, seed: int = None):\n        self.cfg  = cfg\n        self.rng  = np.random.default_rng(seed)\n        self._env = simple_spread_v3.env(\n            N           = cfg.n_agents,\n            local_ratio = cfg.local_ratio,\n            render_mode = None,\n            max_cycles  = cfg.max_steps,\n        )\n        self.agents    = self._env.possible_agents   # stable ordered list\n        self.n_agents  = len(self.agents)\n        self.agent_idx = {a: i for i, a in enumerate(self.agents)}\n\n    def reset(self, seed=None):\n        self._env.reset(seed=seed)\n        return self._collect_obs()\n\n    def step_all(self, actions: Dict[str, np.ndarray]):\n        """\n        Execute one full round: each agent acts once.\n        actions: {agent_id: one-hot np.ndarray}\n        Returns: obs, rewards, dones (all dicts)\n        """\n        rewards, dones = {a: 0.0 for a in self.agents}, {a: False for a in self.agents}\n\n        for agent in self._env.agent_iter():\n            obs_raw, rew, term, trunc, info = self._env.last()\n            done = term or trunc\n            rewards[agent] = float(rew)\n            dones[agent]   = done\n            if done:\n                self._env.step(None)\n            else:\n                act_oh = actions.get(agent)\n                if act_oh is not None:\n                    act_int = int(np.argmax(act_oh))\n                else:\n                    act_int = self._env.action_space(agent).sample()\n                self._env.step(act_int)\n\n        obs = self._collect_obs()\n        return obs, rewards, dones\n\n    def _collect_obs(self) -> Dict[str, np.ndarray]:\n        obs = {}\n        for agent in self.agents:\n            try:\n                o = self._env.observe(agent)\n                if o is None:\n                    o = np.zeros(self.cfg.obs_dim, dtype=np.float32)\n                else:\n                    o = np.array(o, dtype=np.float32).copy()\n            except Exception:\n                o = np.zeros(self.cfg.obs_dim, dtype=np.float32)\n            obs[agent] = self._maybe_corrupt(agent, o)\n        return obs\n\n    def _maybe_corrupt(self, observer: str, obs: np.ndarray) -> np.ndarray:\n        """Add Gaussian noise to peer-message slice for honest observers."""\n        if not self.cfg.defector_ids or observer in self.cfg.defector_ids:\n            return obs\n        noise = (self.rng.standard_normal(self.cfg.msg_end - self.cfg.msg_start)\n                 .astype(np.float32) * self.cfg.noise_scale)\n        obs[self.cfg.msg_start:self.cfg.msg_end] += noise\n        return obs\n\n    def sample_action(self, agent_id: str) -> int:\n        return self._env.action_space(agent_id).sample()\n\n    def close(self):\n        self._env.close()\n\n\n# ── Unit test ─────────────────────────────────────────────────────────────────\ndef test_defector_wrapper():\n    from dataclasses import replace\n\n    # With 1 defector\n    cfg_dirty = Config(n_defectors=1)\n    env_dirty = DefectorWrapper(cfg_dirty, seed=0)\n    env_dirty.reset(seed=0)\n    dummy = np.zeros(cfg_dirty.obs_dim, dtype=np.float32)\n    corrupted = env_dirty._maybe_corrupt("agent_0", dummy.copy())\n    std_dirty = corrupted[cfg_dirty.msg_start:cfg_dirty.msg_end].std()\n\n    # With 0 defectors\n    cfg_clean = Config(n_defectors=0)\n    env_clean = DefectorWrapper(cfg_clean, seed=0)\n    env_clean.reset(seed=0)\n    clean = env_clean._maybe_corrupt("agent_0", dummy.copy())\n    std_clean = clean[cfg_clean.msg_start:cfg_clean.msg_end].std()\n\n    print(f"✅ DefectorWrapper unit test")\n    print(f"   Msg slice std with  defector : {std_dirty:.4f}  (should be > 0.1)")\n    print(f"   Msg slice std without defector: {std_clean:.4f}  (should be 0.0)")\n    assert std_dirty > 0.1, "Corruption not applied!"\n    assert std_clean == 0.0, "Clean env is corrupted — bug!"\n\n    # Smoke: run a few steps\n    env2 = DefectorWrapper(Config(n_defectors=1), seed=7)\n    obs  = env2.reset(seed=7)\n    assert len(obs) == 4, f"Expected 4 agent obs, got {len(obs)}"\n    dummy_acts = {a: np.eye(5)[env2.sample_action(a)] for a in env2.agents}\n    obs2, rews, dones = env2.step_all(dummy_acts)\n    assert len(rews) == 4\n    env2.close()\n    env_dirty.close()\n    env_clean.close()\n    print("   Step test passed ✓")\n    print("   All assertions passed ✓")\n\ntest_defector_wrapper()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 5 — ReputationTracker (reputation.py)             ║\n# ╚══════════════════════════════════════════════════════════╝\n\nfrom scipy.special import softmax as scipy_softmax\n\nclass ReputationTracker:\n    """\n    Maintains a continuous reputation score r[j] in [0,1] for each peer j.\n\n    FIX: baseline is now initialised to None and set from the first real\n    reward, preventing the \'always negative delta\' problem that caused all\n    scores to drift to 0 then snap to 1.0 when rewards improved.\n\n    Update rule (per step):\n        delta  = reward - baseline\n        signal = 1.0 if delta > 0 else 0.0\n        r[j]  <- (1 - alpha) * r[j] + alpha * signal  for all j\n        baseline <- (1 - b_alpha) * baseline + b_alpha * reward\n    """\n\n    def __init__(self, peer_ids, cfg):\n        self.peer_ids    = list(peer_ids)\n        self.alpha       = cfg.rep_alpha\n        self.b_alpha     = cfg.rep_baseline_alpha\n        self.temperature = cfg.rep_temperature\n        self.scores      = {j: 0.5 for j in peer_ids}\n        self.baseline    = None    # FIX: initialise lazily from first reward\n\n    def update(self, reward):\n        """Called once per step with this agent\'s observed reward."""\n        # FIX: initialise baseline to first real reward so delta starts near 0\n        if self.baseline is None:\n            self.baseline = reward\n            return  # skip update on very first step\n\n        delta  = reward - self.baseline\n        signal = 1.0 if delta > 0 else 0.0\n\n        self.baseline = (1 - self.b_alpha) * self.baseline + self.b_alpha * reward\n\n        for j in self.peer_ids:\n            self.scores[j] = (\n                (1 - self.alpha) * self.scores[j] + self.alpha * signal\n            )\n\n    def get_weights(self):\n        """Softmax-normalised weights, ordered by peer_ids."""\n        scores_arr = np.array([self.scores[j] for j in self.peer_ids],\n                               dtype=np.float32)\n        return scipy_softmax(scores_arr * self.temperature).astype(np.float32)\n\n    def get_weights_binary(self, threshold=0.5):\n        """Condition B: hard binary gate, re-normalised."""\n        scores_arr = np.array([self.scores[j] for j in self.peer_ids],\n                               dtype=np.float32)\n        weights = (scores_arr >= threshold).astype(np.float32)\n        total = weights.sum()\n        if total == 0:\n            weights = np.ones(len(self.peer_ids), dtype=np.float32)\n            total   = float(len(self.peer_ids))\n        return weights / total\n\n    def get_weights_uniform(self):\n        """Condition A: no trust — uniform weights."""\n        n = len(self.peer_ids)\n        return np.ones(n, dtype=np.float32) / n\n\n    def get_scores_dict(self):\n        return dict(self.scores)\n\n    def reset(self):\n        self.scores   = {j: 0.5 for j in self.peer_ids}\n        self.baseline = None   # FIX: reset to None, not 0.0\n\n\n# ── Unit tests ────────────────────────────────────────────────────────────────\ndef test_reputation_tracker():\n    cfg_t = Config()\n    peers = ["agent_1", "agent_2", "agent_3"]\n    tracker = ReputationTracker(peers, cfg_t)\n\n    # Test 1: neutral init\n    w = tracker.get_weights()\n    assert abs(w.sum() - 1.0) < 1e-5, "Weights do not sum to 1"\n    assert abs(w[0] - w[1]) < 1e-5, "Initial weights not uniform"\n    print("  Test 1 passed: neutral init OK")\n\n    # Test 2: positive drift — rewards consistently above baseline\n    tracker.reset()\n    for i in range(200):\n        tracker.update(reward=-5.0 + i * 0.05)   # rising rewards\n    scores = list(tracker.scores.values())\n    print("  Test 2: scores after rising rewards: " + str([round(s,3) for s in scores]))\n    assert all(s > 0.5 for s in scores), "Scores did not drift up: " + str(scores)\n    print("  Test 2 passed: positive drift OK (mean=" + str(round(float(np.mean(scores)),3)) + ")")\n\n    # Test 3: negative drift — rewards consistently below baseline\n    tracker.reset()\n    for i in range(200):\n        tracker.update(reward=-5.0 - i * 0.05)   # falling rewards\n    scores = list(tracker.scores.values())\n    print("  Test 3: scores after falling rewards: " + str([round(s,3) for s in scores]))\n    assert all(s < 0.5 for s in scores), "Scores did not drift down: " + str(scores)\n    print("  Test 3 passed: negative drift OK (mean=" + str(round(float(np.mean(scores)),3)) + ")")\n\n    # Test 4: weights always sum to 1\n    for _ in range(50):\n        tracker.update(reward=np.random.randn() - 5.0)\n    for method in ["get_weights", "get_weights_binary", "get_weights_uniform"]:\n        w = getattr(tracker, method)()\n        assert abs(w.sum() - 1.0) < 1e-5, method + " does not sum to 1"\n    print("  Test 4 passed: all weight methods sum to 1.0 OK")\n\n    # Test 5: FIX verification — baseline initialised from first reward\n    tracker2 = ReputationTracker(peers, cfg_t)\n    tracker2.update(reward=-8.0)   # first step, sets baseline to -8, no score update\n    scores_after_first = list(tracker2.scores.values())\n    assert all(s == 0.5 for s in scores_after_first), "Scores changed on first step — baseline bug!"\n    tracker2.update(reward=-7.5)   # second step: delta = -7.5 - (-8.0) = +0.5, signal=1\n    scores_after_second = list(tracker2.scores.values())\n    assert all(s > 0.5 for s in scores_after_second), "Positive delta did not increase scores"\n    print("  Test 5 passed: baseline initialisation fix verified OK")\n\n    print("")\n    print("ReputationTracker — all 5 unit tests passed.")\n\ntest_reputation_tracker()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 6 — Actor & Critic Networks (networks.py)         ║\n# ╚══════════════════════════════════════════════════════════╝\n\nclass Actor(nn.Module):\n    """\n    MADDPG actor with reputation-weighted message aggregation.\n\n    Forward pass:\n        1. Receive own_obs [batch, msg_start] and\n           peer_msgs [batch, n_peers, msg_dim]\n        2. Aggregate peer_msgs weighted by rep_weights [batch, n_peers]\n        3. Concatenate own_obs + aggregated_msg → policy MLP → action logits\n\n    The reputation weighting is the ONLY modification from standard MADDPG.\n    When rep_weights is uniform, output is identical to an unmodified actor.\n    """\n\n    def __init__(self, own_obs_dim: int, n_peers: int,\n                 msg_dim: int, act_dim: int, hidden_dim: int):\n        super().__init__()\n        self.n_peers  = n_peers\n        self.msg_dim  = msg_dim\n        self.act_dim  = act_dim\n\n        input_dim = own_obs_dim + msg_dim   # own obs + aggregated msg\n        self.net = nn.Sequential(\n            nn.Linear(input_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, act_dim),\n        )\n\n    def forward(self,\n                own_obs:     torch.Tensor,   # [batch, own_obs_dim]\n                peer_msgs:   torch.Tensor,   # [batch, n_peers, msg_dim]\n                rep_weights: torch.Tensor,   # [batch, n_peers]\n                ) -> torch.Tensor:           # [batch, act_dim]\n\n        # ── Reputation-weighted message aggregation ───────────────────────────\n        w   = rep_weights.unsqueeze(-1)              # [batch, n_peers, 1]\n        agg = (peer_msgs * w).sum(dim=1)             # [batch, msg_dim]\n        # ─────────────────────────────────────────────────────────────────────\n\n        x = torch.cat([own_obs, agg], dim=-1)        # [batch, own_obs_dim + msg_dim]\n        return self.net(x)                           # [batch, act_dim]\n\n    def select_action(self, own_obs: np.ndarray,\n                      peer_msgs: np.ndarray,\n                      rep_weights: np.ndarray,\n                      deterministic: bool = False) -> np.ndarray:\n        with torch.no_grad():\n            o = torch.FloatTensor(own_obs).unsqueeze(0).to(DEVICE)\n            m = torch.FloatTensor(peer_msgs).unsqueeze(0).to(DEVICE)\n            w = torch.FloatTensor(rep_weights).unsqueeze(0).to(DEVICE)\n            logits = self.forward(o, m, w).squeeze(0)\n            if deterministic:\n                action = logits.argmax().item()\n            else:\n                probs  = F.softmax(logits, dim=-1)\n                action = torch.multinomial(probs, 1).item()\n        return action\n\n\nclass Critic(nn.Module):\n    """\n    Centralised MADDPG critic: Q(all_obs, all_actions).\n    Has full visibility during training (CTDE paradigm).\n    """\n\n    def __init__(self, n_agents: int, obs_dim: int,\n                 act_dim: int, hidden_dim: int):\n        super().__init__()\n        input_dim = n_agents * (obs_dim + act_dim)\n        self.net = nn.Sequential(\n            nn.Linear(input_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, 1),\n        )\n\n    def forward(self, all_obs: torch.Tensor,\n                all_acts: torch.Tensor) -> torch.Tensor:\n        """\n        all_obs:  [batch, n_agents, obs_dim]\n        all_acts: [batch, n_agents, act_dim]\n        Returns:  [batch, 1]\n        """\n        batch = all_obs.shape[0]\n        x = torch.cat([\n            all_obs.reshape(batch, -1),\n            all_acts.reshape(batch, -1)\n        ], dim=-1)\n        return self.net(x)\n\n\n# ── Verify parity: uniform weights == standard MADDPG ─────────────────────────\ndef test_network_parity():\n    cfg_t = Config()\n    own_obs_dim = cfg_t.msg_start        # 28 dims (everything before msg slice)\n    actor = Actor(own_obs_dim, cfg_t.n_peers, cfg_t.msg_dim,\n                  cfg_t.act_dim, cfg_t.hidden_dim)\n\n    # Uniform weights → same as concatenating mean messages\n    batch = 4\n    obs   = torch.randn(batch, own_obs_dim)\n    msgs  = torch.randn(batch, cfg_t.n_peers, cfg_t.msg_dim)\n    w_uni = torch.ones(batch, cfg_t.n_peers) / cfg_t.n_peers\n\n    out_uni  = actor(obs, msgs, w_uni)\n\n    # Manually compute what unmodified MADDPG would do: simple mean\n    agg_mean = msgs.mean(dim=1)\n    x_manual = torch.cat([obs, agg_mean], dim=-1)\n    out_man  = actor.net(x_manual)\n\n    diff = (out_uni - out_man).abs().max().item()\n    assert diff < 1e-5, f"Parity test failed, max diff: {diff}"\n    print(f"✅ Network parity test: uniform weights == mean aggregation (max diff = {diff:.2e}) ✓")\n\n    # Shape checks\n    critic = Critic(cfg_t.n_agents, cfg_t.obs_dim, cfg_t.act_dim, cfg_t.hidden_dim)\n    all_obs  = torch.randn(batch, cfg_t.n_agents, cfg_t.obs_dim)\n    all_acts = torch.randn(batch, cfg_t.n_agents, cfg_t.act_dim)\n    q = critic(all_obs, all_acts)\n    assert q.shape == (batch, 1), f"Critic output shape wrong: {q.shape}"\n    print(f"   Critic output shape: {q.shape} ✓")\n    print(f"   Actor  output shape: {out_uni.shape} ✓")\n\ntest_network_parity()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 7 — Replay Buffer (buffer.py)                     ║\n# ╚══════════════════════════════════════════════════════════╝\n\nclass ReplayBuffer:\n    """\n    Circular replay buffer for MADDPG.\n    Stores transitions for all agents simultaneously.\n    Also stores reputation scores for logging/analysis.\n    """\n\n    def __init__(self, cfg: Config):\n        self.capacity   = cfg.buffer_size\n        self.n_agents   = cfg.n_agents\n        self.obs_dim    = cfg.obs_dim\n        self.act_dim    = cfg.act_dim\n        self.n_peers    = cfg.n_peers\n        self.ptr        = 0\n        self.size       = 0\n\n        # Pre-allocate numpy arrays for speed\n        self.obs       = np.zeros((self.capacity, self.n_agents, self.obs_dim),  dtype=np.float32)\n        self.acts      = np.zeros((self.capacity, self.n_agents, self.act_dim),  dtype=np.float32)\n        self.rews      = np.zeros((self.capacity, self.n_agents),                dtype=np.float32)\n        self.next_obs  = np.zeros((self.capacity, self.n_agents, self.obs_dim),  dtype=np.float32)\n        self.dones     = np.zeros((self.capacity, self.n_agents),                dtype=np.float32)\n        self.rep_scores= np.zeros((self.capacity, self.n_agents, self.n_peers),  dtype=np.float32)\n\n    def push(self,\n             obs:        np.ndarray,   # [n_agents, obs_dim]\n             acts:       np.ndarray,   # [n_agents, act_dim]\n             rews:       np.ndarray,   # [n_agents]\n             next_obs:   np.ndarray,   # [n_agents, obs_dim]\n             dones:      np.ndarray,   # [n_agents]\n             rep_scores: np.ndarray,   # [n_agents, n_peers]\n             ):\n        idx = self.ptr % self.capacity\n        self.obs[idx]        = obs\n        self.acts[idx]       = acts\n        self.rews[idx]       = rews\n        self.next_obs[idx]   = next_obs\n        self.dones[idx]      = dones\n        self.rep_scores[idx] = rep_scores\n        self.ptr  += 1\n        self.size  = min(self.size + 1, self.capacity)\n\n    def sample(self, batch_size: int):\n        idx = np.random.randint(0, self.size, size=batch_size)\n        to_t = lambda x: torch.FloatTensor(x).to(DEVICE)\n        return (\n            to_t(self.obs[idx]),        # [B, n_agents, obs_dim]\n            to_t(self.acts[idx]),       # [B, n_agents, act_dim]\n            to_t(self.rews[idx]),       # [B, n_agents]\n            to_t(self.next_obs[idx]),   # [B, n_agents, obs_dim]\n            to_t(self.dones[idx]),      # [B, n_agents]\n        )\n\n    def __len__(self):\n        return self.size\n\n\ndef test_replay_buffer():\n    cfg_t  = Config()\n    buf    = ReplayBuffer(cfg_t)\n    n, na, od, ad, np_ = 1000, cfg_t.n_agents, cfg_t.obs_dim, cfg_t.act_dim, cfg_t.n_peers\n\n    for _ in range(n):\n        buf.push(\n            np.random.randn(na, od).astype(np.float32),\n            np.random.randn(na, ad).astype(np.float32),\n            np.random.randn(na).astype(np.float32),\n            np.random.randn(na, od).astype(np.float32),\n            np.zeros(na, dtype=np.float32),\n            np.random.rand(na, np_).astype(np.float32),\n        )\n\n    assert len(buf) == n\n    obs, acts, rews, nobs, dones = buf.sample(256)\n    assert obs.shape   == (256, na, od), f"obs shape wrong: {obs.shape}"\n    assert acts.shape  == (256, na, ad)\n    assert rews.shape  == (256, na)\n    print(f"✅ ReplayBuffer: pushed {n} transitions, sampled batch of 256")\n    print(f"   obs shape: {obs.shape}, acts shape: {acts.shape} ✓")\n\ntest_replay_buffer()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 8 — MADDPGAgent & Helper Functions                ║\n# ╚══════════════════════════════════════════════════════════╝\n\nclass MADDPGAgent:\n    """\n    Single agent in the MADDPG framework.\n    Holds actor + target_actor, critic + target_critic,\n    their optimizers, and a ReputationTracker.\n    """\n\n    def __init__(self, agent_id: str, peer_ids: List[str], cfg):\n        self.agent_id = agent_id\n        self.peer_ids = peer_ids\n        self.cfg      = cfg\n        own_obs_dim   = cfg.msg_start   # dims before the comm slice\n\n        # ── Networks ──────────────────────────────────────────────────────\n        self.actor   = Actor(own_obs_dim, cfg.n_peers, cfg.msg_dim,\n                             cfg.act_dim, cfg.hidden_dim).to(DEVICE)\n        self.t_actor = copy.deepcopy(self.actor)\n\n        self.critic  = Critic(cfg.n_agents, cfg.obs_dim,\n                              cfg.act_dim, cfg.hidden_dim).to(DEVICE)\n        self.t_critic = copy.deepcopy(self.critic)\n\n        # ── Optimisers ────────────────────────────────────────────────────\n        self.actor_opt  = optim.Adam(self.actor.parameters(),  lr=cfg.actor_lr)\n        self.critic_opt = optim.Adam(self.critic.parameters(), lr=cfg.critic_lr)\n\n        # ── Reputation tracker ────────────────────────────────────────────\n        self.tracker = ReputationTracker(peer_ids, cfg)\n\n    def get_rep_weights(self) -> np.ndarray:\n        c = self.cfg.trust_condition\n        if   c == "none":   return self.tracker.get_weights_uniform()\n        elif c == "binary": return self.tracker.get_weights_binary(self.cfg.binary_threshold)\n        else:               return self.tracker.get_weights()   # soft\n\n    @torch.no_grad()\n    def soft_update(self, tau: float):\n        for p, tp in zip(self.actor.parameters(), self.t_actor.parameters()):\n            tp.data.copy_(tau * p.data + (1 - tau) * tp.data)\n        for p, tp in zip(self.critic.parameters(), self.t_critic.parameters()):\n            tp.data.copy_(tau * p.data + (1 - tau) * tp.data)\n\n\n# ── Helper functions ──────────────────────────────────────────────────────\n\ndef extract_obs_parts(obs: np.ndarray, cfg):\n    """Split raw obs into own_obs and peer_msgs."""\n    own_obs   = obs[:cfg.msg_start]                          # [msg_start]\n    msg_block = obs[cfg.msg_start:cfg.msg_end]               # [n_peers * msg_dim]\n    peer_msgs = msg_block.reshape(cfg.n_peers, cfg.msg_dim)  # [n_peers, msg_dim]\n    return own_obs, peer_msgs\n\n\ndef onehot(action: int, dim: int) -> np.ndarray:\n    v = np.zeros(dim, dtype=np.float32)\n    v[action] = 1.0\n    return v\n\n\ndef stack_agent_data(d: Dict[str, np.ndarray],\n                     agent_list: List[str]) -> np.ndarray:\n    """Stack per-agent arrays into [n_agents, *] numpy array."""\n    return np.stack([d[a] for a in agent_list], axis=0)\n\n\n# ── Quick sanity check ────────────────────────────────────────────────────\ndef test_maddpg_agent():\n    cfg_t    = Config()\n    peer_ids = ["agent_1", "agent_2", "agent_3"]\n    agent    = MADDPGAgent("agent_0", peer_ids, cfg_t)\n\n    # Check networks are on the right device\n    assert next(agent.actor.parameters()).device.type == DEVICE.type\n    assert next(agent.critic.parameters()).device.type == DEVICE.type\n\n    # Check weight shapes\n    w = agent.get_rep_weights()\n    assert len(w) == cfg_t.n_peers, "Wrong number of weights"\n    assert abs(w.sum() - 1.0) < 1e-5, "Weights do not sum to 1"\n\n    # Check extract_obs_parts\n    dummy_obs = np.random.randn(cfg_t.obs_dim).astype(np.float32)\n    own_obs, peer_msgs = extract_obs_parts(dummy_obs, cfg_t)\n    assert own_obs.shape   == (cfg_t.msg_start,),                "own_obs shape wrong"\n    assert peer_msgs.shape == (cfg_t.n_peers, cfg_t.msg_dim),   "peer_msgs shape wrong"\n\n    n_actor  = sum(p.numel() for p in agent.actor.parameters())\n    n_critic = sum(p.numel() for p in agent.critic.parameters())\n    print("MADDPGAgent sanity check passed.")\n    print("  own_obs_dim : " + str(cfg_t.msg_start))\n    print("  actor params: " + str(n_actor))\n    print("  critic params: " + str(n_critic))\n    print("  rep weights : " + str(w) + "  (sum=" + str(round(w.sum(),4)) + ")")\n\ntest_maddpg_agent()\n',
        '# ╔══════════════════════════════════════════════════════════╗\n# ║  CELL 9 — Training & Evaluation Loop (train.py)         ║\n# ╚══════════════════════════════════════════════════════════╝\n\ndef update_agents(agents, buffer, cfg):\n    """One MADDPG gradient update step for all agents."""\n    if len(buffer) < cfg.batch_size:\n        return 0.0, 0.0\n\n    obs_b, acts_b, rews_b, nobs_b, dones_b = buffer.sample(cfg.batch_size)\n    total_closs, total_aloss = 0.0, 0.0\n\n    for i, agent in enumerate(agents):\n        # ── Critic update ─────────────────────────────────────────────────────\n        with torch.no_grad():\n            t_acts_list = []\n            for j, ag in enumerate(agents):\n                o_j = nobs_b[:, j, :cfg.msg_start]\n                m_j = nobs_b[:, j, cfg.msg_start:cfg.msg_end]\n                m_j = m_j.view(-1, cfg.n_peers, cfg.msg_dim)\n                # Target actor always uses uniform weights (stable training target)\n                w_j = torch.ones(cfg.batch_size, cfg.n_peers, device=DEVICE) / cfg.n_peers\n                t_logits = ag.t_actor(o_j, m_j, w_j)\n                t_act    = F.softmax(t_logits, dim=-1)\n                t_acts_list.append(t_act)\n            t_acts = torch.stack(t_acts_list, dim=1)\n            next_q = agent.t_critic(nobs_b, t_acts)\n            r_i    = rews_b[:, i:i+1]\n            d_i    = dones_b[:, i:i+1]\n            target = r_i + cfg.gamma * (1 - d_i) * next_q\n\n        current_q = agent.critic(obs_b, acts_b)\n        closs     = F.mse_loss(current_q, target)\n        agent.critic_opt.zero_grad()\n        closs.backward()\n        torch.nn.utils.clip_grad_norm_(agent.critic.parameters(), 1.0)\n        agent.critic_opt.step()\n\n        # ── Actor update — FIX: use CURRENT reputation weights ────────────────\n        # The actor must be trained WITH reputation weights so it learns to\n        # exploit the trust signal. Using uniform weights here (the original bug)\n        # meant the network was never exposed to reputation-modulated inputs\n        # during training, so it could not benefit from them at inference time.\n        o_i = obs_b[:, i, :cfg.msg_start]\n        m_i = obs_b[:, i, cfg.msg_start:cfg.msg_end].view(-1, cfg.n_peers, cfg.msg_dim)\n\n        # Get current rep weights for this agent (shape: [n_peers])\n        rep_w = torch.FloatTensor(agent.get_rep_weights()).to(DEVICE)\n        # Expand to batch: [1, n_peers] -> [batch, n_peers]\n        rep_w_batch = rep_w.unsqueeze(0).expand(cfg.batch_size, -1)\n\n        logits  = agent.actor(o_i, m_i, rep_w_batch)\n        new_act = F.softmax(logits, dim=-1)\n        acts_q  = acts_b.clone()\n        acts_q[:, i, :] = new_act\n        aloss = -agent.critic(obs_b, acts_q).mean()\n        agent.actor_opt.zero_grad()\n        aloss.backward()\n        torch.nn.utils.clip_grad_norm_(agent.actor.parameters(), 1.0)\n        agent.actor_opt.step()\n\n        agent.soft_update(cfg.tau)\n        total_closs += closs.item()\n        total_aloss += aloss.item()\n\n    return total_closs / cfg.n_agents, total_aloss / cfg.n_agents\n\n\ndef run_eval_episode(agents, cfg, seed=0):\n    """One evaluation episode — deterministic actions, no buffer push.\n    FIX: tracker is updated during eval so rep scores reflect eval behaviour."""\n    env      = DefectorWrapper(cfg, seed=seed)\n    obs_dict = env.reset(seed=seed)\n    ep_reward = 0.0\n\n    for step in range(cfg.max_steps):\n        actions = {}\n        for ag in agents:\n            aid     = ag.agent_id\n            obs_raw = obs_dict.get(aid, np.zeros(cfg.obs_dim, dtype=np.float32))\n            own_obs, peer_msgs = extract_obs_parts(obs_raw, cfg)\n            w      = ag.get_rep_weights()\n            act_oh = np.eye(cfg.act_dim)[\n                ag.actor.select_action(own_obs, peer_msgs, w, deterministic=True)\n            ]\n            actions[aid] = act_oh\n\n        obs_dict, rews, dones = env.step_all(actions)\n        ep_reward += sum(rews.values())\n\n        # FIX: update tracker during eval so scores reflect actual eval dynamics\n        for ag in agents:\n            ag.tracker.update(rews.get(ag.agent_id, 0.0))\n\n    rep_snap = {ag.agent_id: dict(ag.tracker.scores) for ag in agents}\n    env.close()\n    return ep_reward, rep_snap\n\n\ndef train(cfg, verbose=True):\n    """Full training run. Returns results dict with curves and metrics."""\n    set_seed(cfg.seed)\n    agent_ids = ["agent_" + str(i) for i in range(cfg.n_agents)]\n\n    agents = []\n    for aid in agent_ids:\n        peer_ids = [a for a in agent_ids if a != aid]\n        agents.append(MADDPGAgent(aid, peer_ids, cfg))\n\n    buffer  = ReplayBuffer(cfg)\n    results = {\n        "train_rewards":  [],\n        "eval_rewards":   [],\n        "eval_episodes":  [],\n        "rep_scores_log": [],\n        "critic_losses":  [],\n        "actor_losses":   [],\n    }\n\n    total_steps = 0\n    pbar = trange(cfg.n_episodes,\n                  desc="[" + cfg.trust_condition.upper() + "|" + str(cfg.n_defectors) + "def]",\n                  disable=not verbose)\n\n    for episode in pbar:\n        env      = DefectorWrapper(cfg, seed=cfg.seed + episode)\n        obs_dict = env.reset(seed=cfg.seed + episode)\n        ep_reward = 0.0\n\n        for step in range(cfg.max_steps):\n            actions = {}\n\n            for ag in agents:\n                aid     = ag.agent_id\n                obs_raw = obs_dict.get(aid, np.zeros(cfg.obs_dim, dtype=np.float32))\n                own_obs, peer_msgs = extract_obs_parts(obs_raw, cfg)\n                w = ag.get_rep_weights()\n\n                if total_steps < cfg.warmup_steps:\n                    act_int = env.sample_action(aid)\n                else:\n                    act_int = ag.actor.select_action(own_obs, peer_msgs, w)\n\n                actions[aid] = onehot(act_int, cfg.act_dim)\n\n            next_obs_dict, rews, dones = env.step_all(actions)\n            ep_reward += sum(rews.values())\n\n            # Reputation update BEFORE buffer push so scores reflect current step\n            for ag in agents:\n                ag.tracker.update(rews.get(ag.agent_id, 0.0))\n\n            rep_scores_arr = np.array(\n                [list(ag.tracker.scores.values()) for ag in agents],\n                dtype=np.float32\n            )\n            buffer.push(\n                obs        = stack_agent_data(obs_dict,      agent_ids),\n                acts       = stack_agent_data(actions,       agent_ids),\n                rews       = np.array([rews.get(a, 0.) for a in agent_ids], dtype=np.float32),\n                next_obs   = stack_agent_data(next_obs_dict, agent_ids),\n                dones      = np.array([float(dones.get(a, False)) for a in agent_ids], dtype=np.float32),\n                rep_scores = rep_scores_arr,\n            )\n\n            obs_dict    = next_obs_dict\n            total_steps += 1\n\n            if total_steps >= cfg.warmup_steps and total_steps % 2 == 0:\n                cl, al = update_agents(agents, buffer, cfg)\n                results["critic_losses"].append(cl)\n                results["actor_losses"].append(al)\n\n        results["train_rewards"].append(ep_reward)\n        env.close()\n\n        if (episode + 1) % cfg.eval_interval == 0:\n            eval_rews = []\n            rep_snap_last = {}\n            for e in range(cfg.eval_episodes):\n                er, rep_snap = run_eval_episode(agents, cfg, seed=9000 + e)\n                eval_rews.append(er)\n                if e == cfg.eval_episodes - 1:\n                    rep_snap_last = rep_snap\n\n            mean_eval = float(np.mean(eval_rews))\n            results["eval_rewards"].append(mean_eval)\n            results["eval_episodes"].append(episode + 1)\n            results["rep_scores_log"].append({\n                "episode":          episode + 1,\n                "mean_eval_reward": mean_eval,\n                "rep_scores":       rep_snap_last,\n            })\n\n            if verbose:\n                pbar.set_postfix({"eval": str(round(mean_eval, 1)), "buf": len(buffer)})\n\n    return results\n\nprint("Training loop defined — ready to run experiments.")\nprint("FIXES APPLIED:")\nprint("  1. Actor update now uses current reputation weights (not uniform)")\nprint("  2. Reputation baseline initialised from first real reward (not 0.0)")\nprint("  3. Tracker updated during eval episodes for accurate score logging")\n',
    ]
    for _src in _cells:
        exec(_src, globals())
    print("Setup complete.")
# ─────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────

# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 13 — Results Analysis & Publication Figures       ║
# ╚══════════════════════════════════════════════════════════╝
# Run after Cell 11. Generates all 4 report figures.

import pickle, os

SAVE_DIR = "/content/marl_trust"

# ── Load results ─────────────────────────────────────────────────────────
try:
    with open(SAVE_DIR + "/all_results.pkl", "rb") as f:
        all_results = pickle.load(f)
    print("Loaded " + str(len(all_results)) + " runs from disk.")
except FileNotFoundError:
    print("No saved results found. Run Cell 11 first.")
    raise

SEEDS      = [42, 123, 999]
CONDITIONS = ["none", "binary", "soft"]
DEFECTORS  = [0, 1, 2]
COND_LABEL = {"none": "A - No Trust", "binary": "B - Binary Gate", "soft": "C - Soft RWC"}
COND_COLOR = {"none": "#6B7280", "binary": "#F59E0B", "soft": "#2E75B6"}

def get_eval_rewards(condition, n_def, seeds=SEEDS, last_n=3):
    vals = []
    for s in seeds:
        key = (condition, n_def, s)
        if key in all_results:
            rews = all_results[key]["eval_rewards"]
            vals.append(np.mean(rews[-last_n:]) if rews else 0)
    return vals

# ─────────────────────────────────────────────────────────────────────────
# FIGURE 1 — Learning Curves (0 and 1 defector)
# ─────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("Figure 1: Learning Curves", fontsize=13, fontweight="bold")

for n_def, ax in zip([0, 1], axes):
    for cond in CONDITIONS:
        curves = []
        for s in SEEDS:
            key = (cond, n_def, s)
            if key in all_results:
                eps  = all_results[key]["eval_episodes"]
                rews = all_results[key]["eval_rewards"]
                if rews:
                    curves.append((eps, rews))
        if not curves:
            continue
        min_len = min(len(c[1]) for c in curves)
        rew_mat = np.array([c[1][:min_len] for c in curves])
        ep_arr  = curves[0][0][:min_len]
        mean_r  = rew_mat.mean(0)
        std_r   = rew_mat.std(0)
        color   = COND_COLOR[cond]
        ax.plot(ep_arr, mean_r, label=COND_LABEL[cond], color=color, linewidth=2)
        ax.fill_between(ep_arr, mean_r - std_r, mean_r + std_r, color=color, alpha=0.15)
    title = "0 Defectors (Sanity Check)" if n_def == 0 else "1 Defector"
    ax.set_title(title)
    ax.set_xlabel("Training Episode")
    ax.set_ylabel("Mean Team Reward (eval)")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(SAVE_DIR + "/fig1_learning_curves.pdf", bbox_inches="tight")
plt.savefig(SAVE_DIR + "/fig1_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# ─────────────────────────────────────────────────────────────────────────
# FIGURE 2 — Defector Count Ablation
# ─────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4.5))
bar_w = 0.25
x     = np.arange(len(DEFECTORS))

for ci, cond in enumerate(CONDITIONS):
    means, errs = [], []
    for n_def in DEFECTORS:
        vals = get_eval_rewards(cond, n_def)
        means.append(np.mean(vals) if vals else 0)
        errs.append(np.std(vals)   if vals else 0)
    offset = (ci - 1) * bar_w
    ax.bar(x + offset, means, bar_w,
           label=COND_LABEL[cond], color=COND_COLOR[cond],
           yerr=errs, capsize=4, error_kw={"elinewidth": 1.2},
           edgecolor="white", linewidth=0.5)

ax.set_xlabel("Number of Defecting Agents", fontsize=11)
ax.set_ylabel("Mean Team Reward (last 3 evals)", fontsize=11)
ax.set_title("Figure 2: Performance vs. Defector Count", fontsize=12, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(["0 defectors", "1 defector", "2 defectors"])
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(SAVE_DIR + "/fig2_defector_ablation.pdf", bbox_inches="tight")
plt.savefig(SAVE_DIR + "/fig2_defector_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

# ─────────────────────────────────────────────────────────────────────────
# FIGURE 3 — Reputation Score Dynamics
# ─────────────────────────────────────────────────────────────────────────
key = ("soft", 1, 42)
if key in all_results:
    rep_log  = all_results[key]["rep_scores_log"]
    episodes = [r["episode"] for r in rep_log]
    cfg_ref  = Config(n_defectors=1)
    agent_ids    = ["agent_0", "agent_1", "agent_2", "agent_3"]
    defector_ids = cfg_ref.defector_ids

    fig, ax = plt.subplots(figsize=(9, 4.5))

    for agent_id in agent_ids:
        # agent_0 reputation scores for each peer
        scores_over_time = []
        for rec in rep_log:
            ag0_scores = rec["rep_scores"].get("agent_0", {})
            scores_over_time.append(ag0_scores.get(agent_id, np.nan))

        if agent_id == "agent_0":
            continue  # agent does not track itself
        is_def = agent_id in defector_ids
        color  = "#E53E3E" if is_def else "#2E75B6"
        style  = "--" if is_def else "-"
        label  = agent_id + " (DEFECTOR)" if is_def else agent_id + " (honest)"
        lw     = 2.5 if is_def else 1.8
        ax.plot(episodes, scores_over_time, style, color=color,
                linewidth=lw, label=label)

    ax.axhline(0.5, color="gray", linewidth=1, linestyle=":", alpha=0.6,
               label="Neutral (0.5)")
    ax.set_xlabel("Training Episode", fontsize=11)
    ax.set_ylabel("Reputation Score (agent_0 perspective)", fontsize=11)
    ax.set_title("Figure 3: Reputation Score Dynamics — Soft RWC, 1 Defector",
                 fontsize=12, fontweight="bold")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(SAVE_DIR + "/fig3_reputation_dynamics.pdf", bbox_inches="tight")
    plt.savefig(SAVE_DIR + "/fig3_reputation_dynamics.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Key (soft, 1, 42) not found in results — skipping Figure 3.")

# ─────────────────────────────────────────────────────────────────────────
# Results Summary Table
# ─────────────────────────────────────────────────────────────────────────
print("")
print("Results Summary")
print("=" * 70)
print("{:<22} {:<12} {:>12} {:>8} {:>8}".format(
      "Condition", "Defectors", "Mean Reward", "Std", "N seeds"))
print("-" * 70)
for cond in CONDITIONS:
    for n_def in DEFECTORS:
        vals = get_eval_rewards(cond, n_def)
        if vals:
            print("{:<22} {:<12} {:>12.2f} {:>8.2f} {:>8}".format(
                  COND_LABEL[cond], n_def,
                  np.mean(vals), np.std(vals), len(vals)))
print("=" * 70)

# Statistical test: Condition C vs A at 2 defectors
c_vals = get_eval_rewards("soft", 2)
a_vals = get_eval_rewards("none", 2)
if len(c_vals) >= 2 and len(a_vals) >= 2:
    _, pval = stats.mannwhitneyu(c_vals, a_vals, alternative="greater")
    sig = "Significant (p < 0.05)" if pval < 0.05 else "Not significant"
    print("")
    print("Mann-Whitney U: Condition C > A at 2 defectors")
    print("  p-value = " + str(round(pval, 4)) + "  (" + sig + ")")

print("")
print("All figures saved to: " + SAVE_DIR)


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 14 — Download Results from Colab                  ║
# ╚══════════════════════════════════════════════════════════╝

from google.colab import files
import zipfile, glob

save_dir = Config().save_dir
zip_path = "/content/marl_trust_results.zip"

with zipfile.ZipFile(zip_path, "w") as zf:
    for fpath in glob.glob(f"{save_dir}/*"):
        zf.write(fpath, os.path.basename(fpath))

print(f"📦 Zipped results to {zip_path}")
print("   Contents:")
for fpath in glob.glob(f"{save_dir}/*"):
    size = os.path.getsize(fpath) / 1024
    print(f"   {os.path.basename(fpath):40s}  {size:6.1f} KB")

files.download(zip_path)
print("\n✅ Download started")
